# Follow-up: rare-class learning and answer-focused evidence

The previous NLI-initialized pair reader predicted only Supported and Refuted. This notebook tests a concrete training issue before changing evidence: weighted cross-entropy averaged within tiny batches can dilute class weighting. It then asks whether compact, question-conditioned source spans help the same reader.

Five controlled variants share the encoder, train holdout, learning rate, seed and three-item display limit:

| Variant | What changes | What it tests |
| --- | --- | --- |
| `raw_legacy_d01` | Reproduce old loss; distillation 0.1 | Controlled reference, using this notebook's code |
| `raw_sample_d01` | Average weighted losses by sample count | Does effective rare-class weighting help? |
| `raw_sample_d0` | Also remove NLI distillation | Is preserving generic NLI behavior constraining learning? |
| `qa_same3_d0` | Same original three passages, compact questions/answers/source context | Does answer-focused reading help without changing retrieved sources? |
| `qa_selected3_d0` | Select up to three from the union of cached raw and cleaned/NLI top-three passages | Does answerability and question coverage improve selection within this small pool? |

**Upload this notebook to Colab and run in order on a GPU. It includes its source; no repository checkout or push is needed.** It reuses completed preparation from `evidence_aware_reader_v1`. There is no full-corpus embedding, BM25 run or repeat of the 24-candidate NLI scoring pass. New QA work is bounded to six passages and two questions per claim. A timed four-claim smoke test precedes the full pass. Cached extraction resumes per claim; training resumes from complete epoch checkpoints.

The claim-only question generator is [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), used only when previous generated questions are unavailable. The frozen [RoBERTa SQuAD2 extractor](https://huggingface.co/deepset/roberta-base-squad2) proposes exact source spans and can score no-answer cases. Neither is trained here on AVeriTeC gold answers. The four-way verdict reader is trained on official train labels. Answerability scores do not establish truth. Dev has already informed project development, so new dev results remain exploratory; test is untouched.

## 1. Restore the included source

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, subprocess, time, runpy, gc, traceback
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['USE_TORCH'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HOME'] = '/content/huggingface_cache'
SOURCES = json.loads('{"requirements-evidence-reader-colab.txt": "# Keep Colab\'s CUDA-enabled torch installation. No sentence-transformers,\\n# TensorFlow, vision or language-model quantization dependencies are needed.\\ntransformers==4.57.1\\naccelerate>=1,<2\\nsentencepiece>=0.2,<0.3\\nprotobuf>=4,<7\\nnumpy>=1.26,<3\\nscikit-learn>=1.5,<2\\njoblib>=1.4,<2\\ntqdm>=4,<5\\n", "scripts/__init__.py": "", "scripts/run_answer_followup.py": "\\"\\"\\"Prepare resumable claim-only questions and exact-source answer contexts.\\"\\"\\"\\nimport argparse\\nimport sys\\nfrom pathlib import Path\\n\\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    sub = parser.add_subparsers(dest=\\"command\\", required=True)\\n    for command in (\\"questions\\", \\"prepare\\"):\\n        p = sub.add_parser(command)\\n        p.add_argument(\\"--data-root\\", type=Path, required=True)\\n        p.add_argument(\\"--output\\", type=Path, required=True)\\n        p.add_argument(\\"--model\\", default=\\"Qwen/Qwen3-0.6B\\" if command == \\"questions\\" else \\"deepset/roberta-base-squad2\\")\\n        p.add_argument(\\"--revision\\")\\n        p.add_argument(\\"--device\\", default=\\"cuda\\")\\n        if command == \\"questions\\":\\n            p.add_argument(\\"--train-question-cache\\", type=Path)\\n            p.add_argument(\\"--dev-question-cache\\", type=Path)\\n            p.add_argument(\\"--max-new-tokens\\", type=int, default=160)\\n        else:\\n            p.add_argument(\\"--parent-prepared\\", type=Path, required=True)\\n            p.add_argument(\\"--questions\\", type=Path, required=True)\\n            p.add_argument(\\"--batch-size\\", type=int, default=8)\\n            p.add_argument(\\"--max-length\\", type=int, default=384)\\n            p.add_argument(\\"--limit\\", type=int, default=0)\\n    args = parser.parse_args()\\n    if args.device.startswith(\\"cuda\\"):\\n        import torch\\n        if not torch.cuda.is_available():\\n            parser.error(\\"Choose a Colab GPU runtime\\")\\n    for name in (\\"batch_size\\", \\"max_new_tokens\\"):\\n        if hasattr(args, name) and getattr(args, name) < 1:\\n            parser.error(f\\"{name} must be positive\\")\\n    if args.command == \\"prepare\\" and (args.limit < 0 or args.max_length < 256):\\n        parser.error(\\"Use nonnegative limit and max-length >= 256\\")\\n    from src.retrieval.answer_followup import prepare_answers, prepare_questions\\n    {\\"questions\\": prepare_questions, \\"prepare\\": prepare_answers}[args.command](args)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "scripts/run_evidence_reader.py": "\\"\\"\\"CLI for the evidence-quality and NLI pair-reader pipeline.\\"\\"\\"\\nimport argparse\\nimport sys\\nfrom pathlib import Path\\n\\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\\nsys.path.insert(0, str(PROJECT_ROOT))\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    sub = parser.add_subparsers(dest=\\"command\\", required=True)\\n    prep = sub.add_parser(\\"prepare\\")\\n    for name in (\\"train-claims\\", \\"dev-claims\\", \\"train-rankings\\", \\"dev-rankings\\", \\"output\\"):\\n        prep.add_argument(\\"--\\" + name, type=Path, required=True)\\n    prep.add_argument(\\"--model\\", default=\\"cross-encoder/nli-deberta-v3-small\\")\\n    prep.add_argument(\\"--revision\\")\\n    prep.add_argument(\\"--nli-label-order\\", help=\\"Documented model logit order; auto-detected when config names are explicit\\")\\n    prep.add_argument(\\"--max-candidates\\", type=int, default=24)\\n    prep.add_argument(\\"--candidate-depth\\", type=int, default=100)\\n    prep.add_argument(\\"--max-length\\", type=int, default=384)\\n    prep.add_argument(\\"--claim-budget\\", type=int, default=96)\\n    prep.add_argument(\\"--batch-size\\", type=int, default=16)\\n    prep.add_argument(\\"--device\\", default=\\"cuda\\")\\n    prep.add_argument(\\"--limit\\", type=int, default=0, help=\\"Smoke test only; use a separate output directory\\")\\n    for command in (\\"fast\\", \\"train\\", \\"frozen\\", \\"compare\\"):\\n        p = sub.add_parser(command)\\n        for name in (\\"data-root\\", \\"prepared\\", \\"output\\"):\\n            p.add_argument(\\"--\\" + name, type=Path, required=True)\\n        if command == \\"compare\\":\\n            p.add_argument(\\"--qrels\\", type=Path)\\n            p.add_argument(\\"--reference-root\\", type=Path)\\n            continue\\n        modes = (\\"raw\\", \\"clean\\", \\"clean_nli\\", \\"qa_same3\\", \\"qa_selected3\\") if command == \\"train\\" else (\\"raw\\", \\"clean\\", \\"clean_nli\\")\\n        p.add_argument(\\"--mode\\", choices=modes, required=True)\\n        if command == \\"frozen\\":\\n            p.add_argument(\\"--baseline\\", type=Path, required=True)\\n            p.add_argument(\\"--device\\", default=\\"cuda\\")\\n            continue\\n        p.add_argument(\\"--validation-policy\\", choices=(\\"grouped\\", \\"stratified\\"), default=\\"grouped\\")\\n        p.add_argument(\\"--validation-seed\\", type=int, default=718)\\n        p.add_argument(\\"--selection-only\\", action=\\"store_true\\")\\n        if command == \\"train\\":\\n            p.add_argument(\\"--learning-rates\\", type=float, nargs=\\"+\\", default=[1e-5, 3e-5])\\n            p.add_argument(\\"--epochs\\", type=int, default=3)\\n            p.add_argument(\\"--batch-size\\", type=int, default=2)\\n            p.add_argument(\\"--accumulation\\", type=int, default=8)\\n            p.add_argument(\\"--seed\\", type=int, default=718)\\n            p.add_argument(\\"--distillation-weight\\", type=float, default=.1)\\n            p.add_argument(\\"--loss-normalization\\", choices=(\\"sample_mean\\", \\"batch_weight_mean\\"), default=\\"sample_mean\\")\\n            p.add_argument(\\"--unfreeze-embeddings\\", action=\\"store_true\\", help=\\"Also train vocabulary embeddings; increases optimizer memory and checkpoint size\\")\\n            p.add_argument(\\"--device\\", default=\\"cuda\\")\\n    args = parser.parse_args()\\n    for name in (\\"max_candidates\\", \\"candidate_depth\\", \\"batch_size\\", \\"epochs\\", \\"accumulation\\"):\\n        if hasattr(args, name) and getattr(args, name) < 1:\\n            parser.error(name + \\" must be positive\\")\\n    if args.command == \\"prepare\\":\\n        if args.max_length < args.claim_budget + 32 or args.claim_budget < 1 or args.limit < 0:\\n            parser.error(\\"Invalid text length or smoke-test limit\\")\\n        from src.retrieval.evidence_pipeline import prepare_pipeline\\n        prepare_pipeline(args)\\n    elif args.command == \\"train\\":\\n        if any(lr <= 0 for lr in args.learning_rates) or args.distillation_weight < 0:\\n            parser.error(\\"Invalid learning rates or distillation weight\\")\\n        from src.verification.pair_training import train_reader\\n        train_reader(args)\\n    else:\\n        from src.verification.evidence_experiments import fast_aggregate, frozen_distil, compare_results\\n        {\\"fast\\": fast_aggregate, \\"frozen\\": frozen_distil, \\"compare\\": compare_results}[args.command](args)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "scripts/train_tfidf_logreg.py": "\\"\\"\\"Train and evaluate the TF-IDF + logistic-regression verifier baseline.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport sys\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport joblib\\n\\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\\nsys.path.insert(0, str(PROJECT_ROOT))\\n\\nfrom src.evaluation.classification import evaluate_predictions  # noqa: E402\\nfrom src.data.evidence import attach_evidence  # noqa: E402\\nfrom src.evaluation.end_to_end import annotate_retrieval_coverage, evaluate_by_retrieval_coverage  # noqa: E402\\nfrom src.verification.tfidf_logreg import TfidfLogRegVerifier  # noqa: E402\\nfrom src.utils.io import read_jsonl, write_jsonl  # noqa: E402\\nfrom src.utils.runs import resolve_run_directory, write_run_manifest  # noqa: E402\\n\\n\\ndef attach_retrieved_evidence(records: list[dict[str, Any]], rankings_path: Path) -> list[dict[str, Any]]:\\n    return attach_evidence(records, rankings_path)\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\"--evidence-mode\\", choices=(\\"claim_only\\", \\"gold\\", \\"gold_all\\", \\"retrieved\\"), default=\\"gold\\")\\n    parser.add_argument(\\"--data-root\\", type=Path, default=PROJECT_ROOT / \\"data\\")\\n    parser.add_argument(\\"--train-split\\", default=\\"train\\")\\n    parser.add_argument(\\"--eval-split\\", default=\\"dev\\")\\n    parser.add_argument(\\"--output-dir\\", type=Path, default=PROJECT_ROOT / \\"outputs\\" / \\"tfidf_logreg\\")\\n    parser.add_argument(\\"--run-name\\", help=\\"Optional name for an isolated, non-overwriting run subdirectory.\\")\\n    parser.add_argument(\\"--random-state\\", type=int, default=718)\\n    parser.add_argument(\\"--min-df\\", type=int, default=1)\\n    parser.add_argument(\\"--max-features\\", type=int)\\n    parser.add_argument(\\"--max-evidence\\", type=int, default=3)\\n    parser.add_argument(\\"--train-rankings\\", type=Path, help=\\"Required for --evidence-mode retrieved.\\")\\n    parser.add_argument(\\"--eval-rankings\\", type=Path, help=\\"Required for --evidence-mode retrieved.\\")\\n    parser.add_argument(\\"--eval-qrels\\", type=Path, help=\\"Defaults to <data-root>/processed/averitec/<eval-split>_qrels.jsonl in retrieved mode.\\")\\n    parser.add_argument(\\"--unbalanced\\", action=\\"store_true\\", help=\\"Disable class-balanced logistic-regression weights.\\")\\n    args = parser.parse_args()\\n    try:\\n        args.output_dir = resolve_run_directory(args.output_dir, args.run_name)\\n    except ValueError as error:\\n        parser.error(str(error))\\n\\n    processed = args.data_root / \\"processed\\" / \\"averitec\\"\\n    train_records = read_jsonl(processed / f\\"{args.train_split}.jsonl\\")\\n    eval_records = read_jsonl(processed / f\\"{args.eval_split}.jsonl\\")\\n    if args.evidence_mode == \\"retrieved\\":\\n        if not args.train_rankings or not args.eval_rankings:\\n            parser.error(\\"--train-rankings and --eval-rankings are required for retrieved evidence.\\")\\n        train_records = attach_retrieved_evidence(train_records, args.train_rankings)\\n        eval_records = attach_retrieved_evidence(eval_records, args.eval_rankings)\\n    verifier = TfidfLogRegVerifier(\\n        evidence_mode=args.evidence_mode,\\n        max_evidence=args.max_evidence,\\n        min_df=args.min_df,\\n        max_features=args.max_features,\\n        class_weight=None if args.unbalanced else \\"balanced\\",\\n        random_state=args.random_state,\\n    ).fit(train_records)\\n    predictions = verifier.predict_records(eval_records)\\n    metrics = evaluate_predictions(predictions)\\n    run_config = {\\n        \\"model\\": \\"tfidf_logreg\\",\\n        \\"evidence_mode\\": args.evidence_mode,\\n        \\"train_split\\": args.train_split,\\n        \\"eval_split\\": args.eval_split,\\n        \\"random_state\\": args.random_state,\\n        \\"min_df\\": args.min_df,\\n        \\"max_features\\": args.max_features,\\n        \\"max_evidence\\": args.max_evidence,\\n        \\"run_name\\": args.run_name,\\n        \\"class_weight\\": None if args.unbalanced else \\"balanced\\",\\n        \\"warning\\": (\\n            \\"gold mode is an isolated verifier experiment capped at three annotated answers. It must not be presented as an \\"\\n            \\"end-to-end retrieval result.\\"\\n            if args.evidence_mode == \\"gold\\"\\n            else \\"gold_all is an oracle upper bound with no three-item cap.\\" if args.evidence_mode == \\"gold_all\\" else None\\n        ),\\n    }\\n    args.output_dir.mkdir(parents=True, exist_ok=True)\\n    prefix = f\\"{args.evidence_mode}_{args.train_split}_to_{args.eval_split}\\"\\n    write_jsonl(predictions, args.output_dir / f\\"{prefix}_predictions.jsonl\\")\\n    (args.output_dir / f\\"{prefix}_metrics.json\\").write_text(json.dumps(metrics, indent=2) + \\"\\\\n\\", encoding=\\"utf-8\\")\\n    conditioned_input_paths: dict[str, Path] = {}\\n    if args.evidence_mode == \\"retrieved\\":\\n        qrels_path = args.eval_qrels or processed / f\\"{args.eval_split}_qrels.jsonl\\"\\n        if not qrels_path.is_file():\\n            raise FileNotFoundError(f\\"Retrieved-evidence diagnostics require qrels at {qrels_path}\\")\\n        qrels = {\\n            str(row[\\"claim_id\\"]): [str(value) for value in row.get(\\"relevant_passage_ids\\") or []]\\n            for row in read_jsonl(qrels_path)\\n            if row.get(\\"relevant_passage_ids\\")\\n        }\\n        eval_rankings = read_jsonl(args.eval_rankings)\\n        coverage_path = args.output_dir / f\\"{prefix}_evidence_coverage.jsonl\\"\\n        conditioned_metrics_path = args.output_dir / f\\"{prefix}_retrieval_conditioned_metrics.json\\"\\n        write_jsonl(annotate_retrieval_coverage(predictions, eval_rankings, qrels, k=args.max_evidence), coverage_path)\\n        conditioned_metrics_path.write_text(\\n            json.dumps(evaluate_by_retrieval_coverage(predictions, eval_rankings, qrels, k=args.max_evidence), indent=2) + \\"\\\\n\\",\\n            encoding=\\"utf-8\\",\\n        )\\n        run_config[\\"artifacts\\"] = {\\n            \\"evidence_coverage\\": str(coverage_path),\\n            \\"retrieval_conditioned_metrics\\": str(conditioned_metrics_path),\\n        }\\n        conditioned_input_paths[\\"eval_qrels\\"] = qrels_path\\n    (args.output_dir / f\\"{prefix}_config.json\\").write_text(json.dumps(run_config, indent=2) + \\"\\\\n\\", encoding=\\"utf-8\\")\\n    joblib.dump(verifier, args.output_dir / f\\"{prefix}_model.joblib\\")\\n    write_run_manifest(\\n        args.output_dir / f\\"{prefix}_manifest.json\\",\\n        project_root=PROJECT_ROOT,\\n        config=run_config,\\n        input_paths={\\n            \\"train_records\\": processed / f\\"{args.train_split}.jsonl\\",\\n            \\"eval_records\\": processed / f\\"{args.eval_split}.jsonl\\",\\n            **({\\"train_rankings\\": args.train_rankings, \\"eval_rankings\\": args.eval_rankings} if args.evidence_mode == \\"retrieved\\" else {}),\\n            **conditioned_input_paths,\\n        },\\n    )\\n    print(json.dumps({**run_config, **metrics}, indent=2))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "scripts/train_transformer_verifier.py": "\\"\\"\\"Train a compact evidence-aware transformer with resumable tuning trials.\\n\\nModel selection uses a stratified holdout carved only from official train\\nclaims. The selected configuration is then refit on all official train claims\\nand evaluated once on the unchanged official development split.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport re\\nimport sys\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport numpy as np\\nimport torch\\nfrom sklearn.metrics import accuracy_score, f1_score\\n\\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\\nsys.path.insert(0, str(PROJECT_ROOT))\\n\\nfrom scripts.train_tfidf_logreg import attach_retrieved_evidence  # noqa: E402\\nfrom src.data.averitec import CANONICAL_LABELS  # noqa: E402\\nfrom src.evaluation.classification import evaluate_predictions  # noqa: E402\\nfrom src.evaluation.splits import selection_split  # noqa: E402\\nfrom src.utils.io import read_jsonl, write_json, write_jsonl  # noqa: E402\\nfrom src.utils.progress import progress  # noqa: E402\\nfrom src.utils.runs import resolve_run_directory, sha256_file, write_run_manifest  # noqa: E402\\nfrom src.utils.checkpoints import latest_complete_trainer_checkpoint  # noqa: E402\\n\\n\\nLABEL_TO_ID = {label: index for index, label in enumerate(CANONICAL_LABELS)}\\nID_TO_LABEL = {index: label for label, index in LABEL_TO_ID.items()}\\n\\n\\ndef parse_float_grid(value: str, name: str) -> list[float]:\\n    try:\\n        result = [float(item.strip()) for item in value.split(\\",\\") if item.strip()]\\n    except ValueError as error:\\n        raise argparse.ArgumentTypeError(f\\"{name} must be comma-separated numbers\\") from error\\n    if not result or any(item <= 0 for item in result):\\n        raise argparse.ArgumentTypeError(f\\"{name} values must be positive\\")\\n    return list(dict.fromkeys(result))\\n\\n\\ndef parse_int_grid(value: str, name: str) -> list[int]:\\n    try:\\n        result = [int(item.strip()) for item in value.split(\\",\\") if item.strip()]\\n    except ValueError as error:\\n        raise argparse.ArgumentTypeError(f\\"{name} must be comma-separated integers\\") from error\\n    if not result or any(item <= 0 for item in result):\\n        raise argparse.ArgumentTypeError(f\\"{name} values must be positive\\")\\n    return list(dict.fromkeys(result))\\n\\n\\ndef attach_rankings(records: list[dict[str, Any]], rankings_path: Path) -> list[dict[str, Any]]:\\n    return attach_retrieved_evidence(records, rankings_path)\\n\\n\\ndef compose_text(record: dict[str, Any], tokenizer: Any, claim_budget: int, evidence_budget: int) -> str:\\n    claim_tokens = tokenizer.encode(str(record[\\"claim\\"]), add_special_tokens=False)[:claim_budget]\\n    claim_text = tokenizer.decode(claim_tokens, skip_special_tokens=True)\\n    sections = [f\\"Claim: {claim_text}\\"]\\n    evidence = [item for item in (record.get(\\"retrieved_evidence\\") or [])[:3] if str(item.get(\\"text\\") or \\"\\").strip()]\\n    if not evidence:\\n        sections.append(\\"Evidence: No candidate passage was retrieved.\\")\\n    else:\\n        for index, item in enumerate(evidence, start=1):\\n            tokens = tokenizer.encode(str(item[\\"text\\"]), add_special_tokens=False)[:evidence_budget]\\n            text = tokenizer.decode(tokens, skip_special_tokens=True)\\n            sections.append(f\\"Evidence {index}: {text}\\")\\n    return \\"\\\\n\\".join(sections)\\n\\n\\nclass EncodedRecords(torch.utils.data.Dataset):\\n    def __init__(self, records: list[dict[str, Any]], tokenizer: Any, max_length: int, claim_budget: int, evidence_budget: int):\\n        self.records = records\\n        self.features = []\\n        tokenization_batch_size = 128\\n        with progress(total=len(records), description=\\"Tokenizing claims\\", unit=\\"claim\\", leave=False) as bar:\\n            for start in range(0, len(records), tokenization_batch_size):\\n                batch_records = records[start : start + tokenization_batch_size]\\n                texts = [compose_text(row, tokenizer, claim_budget, evidence_budget) for row in batch_records]\\n                encoded = tokenizer(\\n                    texts,\\n                    max_length=max_length,\\n                    truncation=True,\\n                    padding=False,\\n                    return_attention_mask=True,\\n                )\\n                for index, record in enumerate(batch_records):\\n                    feature = {key: values[index] for key, values in encoded.items()}\\n                    feature[\\"labels\\"] = LABEL_TO_ID[str(record[\\"label\\"])]\\n                    self.features.append(feature)\\n                bar.update(len(batch_records))\\n\\n    def __len__(self) -> int:\\n        return len(self.features)\\n\\n    def __getitem__(self, index: int) -> dict[str, Any]:\\n        return self.features[index]\\n\\n\\ndef prediction_metrics(eval_prediction: Any) -> dict[str, float]:\\n    logits = eval_prediction.predictions\\n    if isinstance(logits, tuple):\\n        logits = logits[0]\\n    gold = np.asarray(eval_prediction.label_ids)\\n    predicted = np.asarray(logits).argmax(axis=1)\\n    supported_id = LABEL_TO_ID[\\"Supported\\"]\\n    false_reassurance = np.logical_and(gold != supported_id, predicted == supported_id)\\n    false_alarm = np.logical_and(gold == supported_id, predicted != supported_id)\\n    return {\\n        \\"macro_f1\\": float(f1_score(gold, predicted, labels=list(range(len(CANONICAL_LABELS))), average=\\"macro\\", zero_division=0)),\\n        \\"accuracy\\": float(accuracy_score(gold, predicted)),\\n        \\"false_reassurance_rate\\": float(false_reassurance.mean()),\\n        \\"false_alarm_rate\\": float(false_alarm.mean()),\\n    }\\n\\n\\ndef safe_name(value: float) -> str:\\n    return re.sub(r\\"[^A-Za-z0-9]+\\", \\"p\\", f\\"{value:.0e}\\" if value < 0.001 else f\\"{value:g}\\")\\n\\n\\ndef read_json(path: Path) -> dict[str, Any] | None:\\n    if not path.is_file():\\n        return None\\n    return json.loads(path.read_text(encoding=\\"utf-8\\"))\\n\\n\\ndef best_checkpoint(trial_dir: Path) -> str | None:\\n    \\"\\"\\"Return the newest complete Trainer checkpoint, ignoring torn Drive writes.\\"\\"\\"\\n    return latest_complete_trainer_checkpoint(trial_dir)\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\"--data-root\\", type=Path, required=True)\\n    parser.add_argument(\\"--train-rankings\\", required=True, type=Path)\\n    parser.add_argument(\\"--eval-rankings\\", required=True, type=Path)\\n    parser.add_argument(\\"--output-dir\\", type=Path, required=True)\\n    parser.add_argument(\\"--run-name\\", required=True)\\n    parser.add_argument(\\"--model\\", default=\\"distilbert-base-uncased\\")\\n    parser.add_argument(\\"--revision\\", help=\\"Optional immutable Hugging Face model revision.\\")\\n    parser.add_argument(\\"--max-length\\", type=int, default=512)\\n    parser.add_argument(\\"--claim-token-budget\\", type=int, default=96)\\n    parser.add_argument(\\"--evidence-token-budget\\", type=int, default=110)\\n    parser.add_argument(\\"--validation-fraction\\", type=float, default=0.10)\\n    parser.add_argument(\\"--validation-policy\\", choices=(\\"stratified\\", \\"temporal\\", \\"grouped\\"), default=\\"stratified\\")\\n    parser.add_argument(\\"--learning-rates\\", default=\\"2e-5,5e-5\\")\\n    parser.add_argument(\\"--epoch-grid\\", default=\\"2,3\\")\\n    parser.add_argument(\\"--batch-size\\", type=int, default=8)\\n    parser.add_argument(\\"--gradient-accumulation-steps\\", type=int, default=2)\\n    parser.add_argument(\\"--checkpoint-steps\\", type=int, default=100)\\n    parser.add_argument(\\"--seed\\", type=int, default=718)\\n    parser.add_argument(\\"--validation-seed\\", type=int, help=\\"Fix the holdout while varying model-training seed; defaults to --seed.\\")\\n    parser.add_argument(\\"--resume\\", action=\\"store_true\\")\\n    parser.add_argument(\\"--class-weighting\\", choices=(\\"balanced\\", \\"none\\"), default=\\"balanced\\")\\n    parser.add_argument(\\"--selection-only\\", action=\\"store_true\\", help=\\"Stop after train-only holdout evaluation; do not refit or score dev.\\")\\n    parser.add_argument(\\"--refit-epoch-policy\\", choices=(\\"requested\\", \\"best_checkpoint\\"), default=\\"requested\\",\\n                        help=\\"Refit for the grid epoch count (legacy) or the epoch of the checkpoint actually selected on train holdout.\\")\\n    args = parser.parse_args()\\n\\n    if args.max_length < 32 or args.claim_token_budget < 1 or args.evidence_token_budget < 1:\\n        parser.error(\\"Token lengths must be positive and max-length must be at least 32\\")\\n    if not 0.01 <= args.validation_fraction <= 0.4:\\n        parser.error(\\"validation-fraction must be between 0.01 and 0.4\\")\\n    if min(args.batch_size, args.gradient_accumulation_steps, args.checkpoint_steps) < 1:\\n        parser.error(\\"batch size, gradient accumulation, and checkpoint steps must be positive\\")\\n    try:\\n        learning_rates = parse_float_grid(args.learning_rates, \\"learning-rates\\")\\n        epochs_grid = parse_int_grid(args.epoch_grid, \\"epoch-grid\\")\\n    except argparse.ArgumentTypeError as error:\\n        parser.error(str(error))\\n\\n    try:\\n        output_dir = resolve_run_directory(args.output_dir, args.run_name)\\n    except ValueError as error:\\n        parser.error(str(error))\\n    output_dir.mkdir(parents=True, exist_ok=True)\\n\\n    processed = args.data_root / \\"processed\\" / \\"averitec\\"\\n    train_path = processed / \\"train.jsonl\\"\\n    eval_path = processed / \\"dev.jsonl\\"\\n    for path in (train_path, eval_path, args.train_rankings, args.eval_rankings):\\n        if not path.is_file():\\n            raise FileNotFoundError(path)\\n\\n    train_records = attach_rankings(read_jsonl(train_path), args.train_rankings)\\n    eval_records = attach_rankings(read_jsonl(eval_path), args.eval_rankings)\\n    validation_seed = args.seed if args.validation_seed is None else args.validation_seed\\n    train_indices, validation_indices = selection_split(train_records, args.validation_fraction, validation_seed, args.validation_policy)\\n    selection_train_records = [train_records[int(index)] for index in train_indices]\\n    validation_records = [train_records[int(index)] for index in validation_indices]\\n\\n    config = {\\n        \\"script\\": \\"train_transformer_verifier\\",\\n        \\"model\\": args.model,\\n        \\"requested_revision\\": args.revision,\\n        \\"max_length\\": args.max_length,\\n        \\"claim_token_budget\\": args.claim_token_budget,\\n        \\"evidence_token_budget_per_passage\\": args.evidence_token_budget,\\n        \\"max_evidence\\": 3,\\n        \\"evidence_source\\": str(args.train_rankings),\\n        \\"train_rankings_sha256\\": sha256_file(args.train_rankings),\\n        \\"eval_rankings\\": str(args.eval_rankings),\\n        \\"eval_rankings_sha256\\": sha256_file(args.eval_rankings),\\n        \\"validation_fraction\\": args.validation_fraction,\\n        \\"validation_seed\\": validation_seed,\\n        \\"training_seed\\": args.seed,\\n        \\"validation_policy\\": args.validation_policy,\\n        \\"learning_rate_grid\\": learning_rates,\\n        \\"epoch_grid\\": epochs_grid,\\n        \\"batch_size\\": args.batch_size,\\n        \\"gradient_accumulation_steps\\": args.gradient_accumulation_steps,\\n        \\"checkpoint_steps\\": args.checkpoint_steps,\\n        \\"loss\\": \\"inverse_frequency_weighted_cross_entropy\\" if args.class_weighting == \\"balanced\\" else \\"unweighted_cross_entropy\\",\\n        \\"class_weighting\\": args.class_weighting,\\n        \\"refit_epoch_policy\\": args.refit_epoch_policy,\\n        \\"split_policy\\": f\\"official train split with internal {args.validation_policy} validation; official dev evaluated after refit\\",\\n    }\\n    config_path = output_dir / \\"transformer_run_config.json\\"\\n    existing_config = read_json(config_path)\\n    if existing_config is not None:\\n        existing_config.setdefault(\\"class_weighting\\", \\"balanced\\")\\n        existing_config.setdefault(\\"refit_epoch_policy\\", \\"requested\\")\\n        existing_config.setdefault(\\"validation_policy\\", \\"stratified\\")\\n        existing_config.setdefault(\\"training_seed\\", existing_config.get(\\"validation_seed\\", 718))\\n    if existing_config is not None and any(existing_config.get(key) != value for key, value in config.items()):\\n        raise ValueError(f\\"Run configuration in {config_path} differs; choose a new --run-name.\\")\\n    config = {**(existing_config or {}), **config}\\n    model_revision = args.revision or (existing_config or {}).get(\\"resolved_model_revision\\")\\n    if existing_config is None:\\n        write_json(config, config_path)\\n\\n    final_metrics_path = output_dir / \\"final_dev_metrics.json\\"\\n    final_predictions_path = output_dir / \\"final_dev_predictions.jsonl\\"\\n    if not args.selection_only and final_metrics_path.exists() and final_predictions_path.exists():\\n        manifest_path = output_dir / \\"transformer_manifest.json\\"\\n        if not manifest_path.is_file():\\n            selection_report = read_json(output_dir / \\"model_selection.json\\") or {}\\n            write_run_manifest(\\n                manifest_path,\\n                project_root=PROJECT_ROOT,\\n                config={**config, \\"selection\\": selection_report, \\"final_dev_metrics_path\\": str(final_metrics_path)},\\n                input_paths={\\n                    \\"train_claims\\": train_path,\\n                    \\"dev_claims\\": eval_path,\\n                    \\"train_rankings\\": args.train_rankings,\\n                    \\"dev_rankings\\": args.eval_rankings,\\n                },\\n            )\\n        print(final_metrics_path.read_text(encoding=\\"utf-8\\"))\\n        return\\n\\n    try:\\n        from transformers import (\\n            AutoModelForSequenceClassification,\\n            AutoTokenizer,\\n            DataCollatorWithPadding,\\n            Trainer,\\n            TrainingArguments,\\n            set_seed,\\n        )\\n    except ImportError as error:\\n        raise RuntimeError(\\"Transformer training requires transformers; install requirements-colab.txt first.\\") from error\\n\\n    class WeightedTrainer(Trainer):\\n        def __init__(self, *trainer_args: Any, class_weights: torch.Tensor, **trainer_kwargs: Any):\\n            super().__init__(*trainer_args, **trainer_kwargs)\\n            self.class_weights = class_weights\\n\\n        def compute_loss(\\n            self,\\n            model: Any,\\n            inputs: dict[str, torch.Tensor],\\n            return_outputs: bool = False,\\n            num_items_in_batch: torch.Tensor | None = None,\\n        ):\\n            labels = inputs[\\"labels\\"]\\n            model_inputs = {key: value for key, value in inputs.items() if key != \\"labels\\"}\\n            outputs = model(**model_inputs)\\n            weights = self.class_weights.to(outputs.logits.device)\\n            loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=weights)\\n            return (loss, outputs) if return_outputs else loss\\n\\n    tokenizer = AutoTokenizer.from_pretrained(args.model, revision=model_revision or None, use_fast=True)\\n    resolved_model_revision = getattr(tokenizer, \\"init_kwargs\\", {}).get(\\"_commit_hash\\")\\n    if resolved_model_revision:\\n        config[\\"resolved_model_revision\\"] = resolved_model_revision\\n        write_json(config, config_path)\\n    collator = DataCollatorWithPadding(tokenizer=tokenizer)\\n    selection_train_dataset = EncodedRecords(\\n        selection_train_records, tokenizer, args.max_length, args.claim_token_budget, args.evidence_token_budget\\n    )\\n    validation_dataset = EncodedRecords(\\n        validation_records, tokenizer, args.max_length, args.claim_token_budget, args.evidence_token_budget\\n    )\\n\\n    counts = np.bincount(\\n        [LABEL_TO_ID[str(row[\\"label\\"])] for row in selection_train_records], minlength=len(CANONICAL_LABELS)\\n    )\\n    if np.any(counts == 0):\\n        raise ValueError(f\\"Internal training fold is missing one or more labels: counts={counts.tolist()}\\")\\n    class_weights = torch.tensor(len(selection_train_records) / (len(CANONICAL_LABELS) * counts), dtype=torch.float32)\\n    if args.class_weighting == \\"none\\":\\n        class_weights = torch.ones(len(CANONICAL_LABELS), dtype=torch.float32)\\n    trials_dir = output_dir / \\"trials\\"\\n    trials_dir.mkdir(parents=True, exist_ok=True)\\n    trial_rows: list[dict[str, Any]] = []\\n\\n    def new_model():\\n        nonlocal resolved_model_revision\\n        model = AutoModelForSequenceClassification.from_pretrained(\\n            args.model,\\n            revision=model_revision or None,\\n            num_labels=len(CANONICAL_LABELS),\\n            label2id=LABEL_TO_ID,\\n            id2label=ID_TO_LABEL,\\n        )\\n        resolved_model_revision = getattr(model.config, \\"_commit_hash\\", None) or resolved_model_revision\\n        if resolved_model_revision and config.get(\\"resolved_model_revision\\") != resolved_model_revision:\\n            config[\\"resolved_model_revision\\"] = resolved_model_revision\\n            write_json(config, config_path)\\n        return model\\n\\n    def train_arguments(path: Path, *, lr: float, epochs: float, select_best: bool) -> TrainingArguments:\\n        common = {\\n            \\"output_dir\\": str(path),\\n            \\"learning_rate\\": lr,\\n            \\"num_train_epochs\\": epochs,\\n            \\"per_device_train_batch_size\\": args.batch_size,\\n            \\"per_device_eval_batch_size\\": args.batch_size,\\n            \\"gradient_accumulation_steps\\": args.gradient_accumulation_steps,\\n            \\"weight_decay\\": 0.01,\\n            \\"warmup_ratio\\": 0.06,\\n            \\"logging_steps\\": 25,\\n            \\"logging_first_step\\": True,\\n            \\"disable_tqdm\\": False,\\n            \\"save_strategy\\": \\"steps\\",\\n            \\"save_steps\\": args.checkpoint_steps,\\n            \\"save_total_limit\\": 2,\\n            \\"seed\\": args.seed,\\n            \\"fp16\\": bool(torch.cuda.is_available()),\\n            \\"report_to\\": \\"none\\",\\n            \\"dataloader_num_workers\\": 0,\\n            \\"save_safetensors\\": True,\\n        }\\n        if select_best:\\n            common.update(\\n                {\\n                    \\"eval_strategy\\": \\"steps\\",\\n                    \\"eval_steps\\": args.checkpoint_steps,\\n                    \\"load_best_model_at_end\\": True,\\n                    \\"metric_for_best_model\\": \\"macro_f1\\",\\n                    \\"greater_is_better\\": True,\\n                }\\n            )\\n        else:\\n            common[\\"eval_strategy\\"] = \\"no\\"\\n            common[\\"load_best_model_at_end\\"] = False\\n        return TrainingArguments(**common)\\n\\n    for learning_rate in learning_rates:\\n        for epochs in epochs_grid:\\n            trial_name = f\\"lr_{safe_name(learning_rate)}_epochs_{epochs}\\"\\n            trial_dir = trials_dir / trial_name\\n            summary_path = trial_dir / \\"trial_summary.json\\"\\n            summary = read_json(summary_path)\\n            if summary is None:\\n                if trial_dir.exists() and any(trial_dir.glob(\\"checkpoint-*\\")) and not args.resume:\\n                    raise ValueError(f\\"Found interrupted trial at {trial_dir}; rerun with --resume.\\")\\n                trial_dir.mkdir(parents=True, exist_ok=True)\\n                set_seed(args.seed)\\n                trainer = WeightedTrainer(\\n                    model=new_model(),\\n                    args=train_arguments(trial_dir, lr=learning_rate, epochs=epochs, select_best=True),\\n                    train_dataset=selection_train_dataset,\\n                    eval_dataset=validation_dataset,\\n                    data_collator=collator,\\n                    compute_metrics=prediction_metrics,\\n                    class_weights=class_weights,\\n                )\\n                last = best_checkpoint(trial_dir) if args.resume else None\\n                trainer.train(resume_from_checkpoint=last)\\n                validation_logits = trainer.predict(validation_dataset).predictions\\n                if isinstance(validation_logits, tuple):\\n                    validation_logits = validation_logits[0]\\n                probabilities = torch.softmax(torch.tensor(validation_logits), dim=1).cpu().numpy()\\n                predicted_ids = probabilities.argmax(axis=1)\\n                validation_predictions = []\\n                for row, predicted_id, probability_row in zip(\\n                    validation_records, predicted_ids, probabilities, strict=True\\n                ):\\n                    validation_predictions.append(\\n                        {\\n                            \\"claim_id\\": str(row[\\"claim_id\\"]),\\n                            \\"true_label\\": str(row[\\"label\\"]),\\n                            \\"predicted_label\\": ID_TO_LABEL[int(predicted_id)],\\n                            \\"probabilities\\": {\\n                                ID_TO_LABEL[index]: float(probability_row[index])\\n                                for index in range(len(CANONICAL_LABELS))\\n                            },\\n                        }\\n                    )\\n                metrics = evaluate_predictions(validation_predictions)\\n                selected_checkpoint = trainer.state.best_model_checkpoint\\n                checkpoint_state = read_json(Path(selected_checkpoint) / \\"trainer_state.json\\") if selected_checkpoint else None\\n                summary = {\\n                    \\"trial\\": trial_name,\\n                    \\"learning_rate\\": learning_rate,\\n                    \\"epochs\\": epochs,\\n                    \\"best_checkpoint\\": selected_checkpoint,\\n                    \\"best_checkpoint_epoch\\": checkpoint_state.get(\\"epoch\\") if checkpoint_state else None,\\n                    \\"selection_validation_claim_count\\": len(validation_records),\\n                    \\"macro_f1\\": metrics[\\"macro_f1\\"],\\n                    \\"accuracy\\": metrics[\\"accuracy\\"],\\n                    \\"decision_error_rates\\": metrics[\\"decision_error_rates\\"],\\n                    \\"calibration\\": metrics.get(\\"calibration\\"),\\n                    \\"predictions_path\\": str(trial_dir / \\"validation_predictions.jsonl\\"),\\n                }\\n                write_jsonl(validation_predictions, trial_dir / \\"validation_predictions.jsonl\\")\\n                write_json(summary, summary_path)\\n                del trainer\\n                if torch.cuda.is_available():\\n                    torch.cuda.empty_cache()\\n            trial_rows.append(summary)\\n\\n    selected = sorted(\\n        trial_rows,\\n        key=lambda row: (\\n            -float(row[\\"macro_f1\\"]),\\n            float(row[\\"decision_error_rates\\"][\\"false_reassurance_rate\\"]),\\n            float(row[\\"decision_error_rates\\"][\\"false_alarm_rate\\"]),\\n            float(row[\\"learning_rate\\"]),\\n            int(row[\\"epochs\\"]),\\n        ),\\n    )[0]\\n    selection_report = {\\n        \\"selection_metric\\": \\"validation macro-F1; ties broken by lower false reassurance, then lower false alarm\\",\\n        \\"selected_trial\\": selected,\\n        \\"all_trials\\": trial_rows,\\n        \\"validation_policy\\": args.validation_policy,\\n        \\"validation_seed\\": validation_seed,\\n        \\"training_seed\\": args.seed,\\n        \\"selection_train_claim_count\\": len(selection_train_records),\\n        \\"selection_validation_claim_count\\": len(validation_records),\\n        \\"selection_train_claim_ids\\": [r[\\"claim_id\\"] for r in selection_train_records],\\n        \\"selection_validation_claim_ids\\": [r[\\"claim_id\\"] for r in validation_records],\\n    }\\n    write_json(selection_report, output_dir / \\"model_selection.json\\")\\n    if args.selection_only:\\n        print(json.dumps({k: v for k, v in selection_report.items() if not k.endswith(\\"claim_ids\\")}, indent=2))\\n        return\\n\\n    refit_epochs = float(selected[\\"epochs\\"])\\n    if args.refit_epoch_policy == \\"best_checkpoint\\":\\n        if not selected.get(\\"best_checkpoint_epoch\\"):\\n            raise ValueError(\\"Cached trial lacks best_checkpoint_epoch; use a new run name for best-checkpoint refitting.\\")\\n        refit_epochs = float(selected[\\"best_checkpoint_epoch\\"])\\n\\n    final_dir = output_dir / \\"final_model\\"\\n    final_complete_path = output_dir / \\"final_train_complete.json\\"\\n    if not final_complete_path.exists():\\n        if final_dir.exists() and any(final_dir.glob(\\"checkpoint-*\\")) and not args.resume:\\n            raise ValueError(f\\"Found interrupted final training at {final_dir}; rerun with --resume.\\")\\n        final_dir.mkdir(parents=True, exist_ok=True)\\n        full_counts = np.bincount(\\n            [LABEL_TO_ID[str(row[\\"label\\"])] for row in train_records], minlength=len(CANONICAL_LABELS)\\n        )\\n        full_weights = torch.tensor(len(train_records) / (len(CANONICAL_LABELS) * full_counts), dtype=torch.float32)\\n        if args.class_weighting == \\"none\\":\\n            full_weights = torch.ones(len(CANONICAL_LABELS), dtype=torch.float32)\\n        set_seed(args.seed)\\n        final_trainer = WeightedTrainer(\\n            model=new_model(),\\n            args=train_arguments(final_dir, lr=float(selected[\\"learning_rate\\"]), epochs=refit_epochs, select_best=False),\\n            train_dataset=EncodedRecords(\\n                train_records, tokenizer, args.max_length, args.claim_token_budget, args.evidence_token_budget\\n            ),\\n            data_collator=collator,\\n            class_weights=full_weights,\\n        )\\n        last = best_checkpoint(final_dir) if args.resume else None\\n        final_trainer.train(resume_from_checkpoint=last)\\n        final_trainer.save_model(str(final_dir))\\n        tokenizer.save_pretrained(final_dir)\\n        write_json(\\n            {\\n                \\"selected_trial\\": selected[\\"trial\\"],\\n                \\"training_claim_count\\": len(train_records),\\n                \\"epochs\\": refit_epochs,\\n                \\"refit_epoch_policy\\": args.refit_epoch_policy,\\n                \\"learning_rate\\": selected[\\"learning_rate\\"],\\n            },\\n            final_complete_path,\\n        )\\n        del final_trainer\\n        if torch.cuda.is_available():\\n            torch.cuda.empty_cache()\\n\\n    final_model = AutoModelForSequenceClassification.from_pretrained(final_dir)\\n    final_trainer = WeightedTrainer(\\n        model=final_model,\\n        args=train_arguments(output_dir / \\"prediction_tmp\\", lr=float(selected[\\"learning_rate\\"]), epochs=int(selected[\\"epochs\\"]), select_best=False),\\n        data_collator=collator,\\n        class_weights=torch.ones(len(CANONICAL_LABELS), dtype=torch.float32),\\n    )\\n    eval_dataset = EncodedRecords(eval_records, tokenizer, args.max_length, args.claim_token_budget, args.evidence_token_budget)\\n    prediction_output = final_trainer.predict(eval_dataset)\\n    logits = prediction_output.predictions[0] if isinstance(prediction_output.predictions, tuple) else prediction_output.predictions\\n    probabilities = torch.softmax(torch.tensor(logits), dim=1).cpu().numpy()\\n    predicted_ids = probabilities.argmax(axis=1)\\n    eval_predictions = []\\n    for row, predicted_id, probability_row in zip(eval_records, predicted_ids, probabilities, strict=True):\\n        retrieved = row.get(\\"retrieved_evidence\\") or []\\n        eval_predictions.append(\\n            {\\n                \\"claim_id\\": str(row[\\"claim_id\\"]),\\n                \\"true_label\\": str(row[\\"label\\"]),\\n                \\"predicted_label\\": ID_TO_LABEL[int(predicted_id)],\\n                \\"probabilities\\": {\\n                    ID_TO_LABEL[index]: float(probability_row[index])\\n                    for index in range(len(CANONICAL_LABELS))\\n                },\\n                \\"evidence_item_count\\": min(3, len(retrieved)),\\n                \\"evidence_passage_ids\\": [str(item[\\"passage_id\\"]) for item in retrieved[:3]],\\n            }\\n        )\\n    metrics = evaluate_predictions(eval_predictions)\\n    write_jsonl(eval_predictions, final_predictions_path)\\n    final_report = {\\n        **metrics,\\n        \\"selection\\": selected,\\n        \\"eval_split\\": \\"official dev\\",\\n        \\"eval_claim_count\\": len(eval_records),\\n        \\"evidence_source\\": str(args.eval_rankings),\\n        \\"model\\": args.model,\\n        \\"model_path\\": str(final_dir),\\n        \\"refit_epochs\\": refit_epochs,\\n        \\"refit_epoch_policy\\": args.refit_epoch_policy,\\n    }\\n    write_json(final_report, final_metrics_path)\\n    write_run_manifest(\\n        output_dir / \\"transformer_manifest.json\\",\\n        project_root=PROJECT_ROOT,\\n        config={**config, \\"selection\\": selection_report, \\"final_dev_metrics_path\\": str(final_metrics_path)},\\n        input_paths={\\n            \\"train_claims\\": train_path,\\n            \\"dev_claims\\": eval_path,\\n            \\"train_rankings\\": args.train_rankings,\\n            \\"dev_rankings\\": args.eval_rankings,\\n        },\\n    )\\n    print(json.dumps(final_report, indent=2))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "src/__init__.py": "\\"\\"\\"Misinformation screening assistant source package.\\"\\"\\"\\n", "src/data/__init__.py": "\\"\\"\\"Dataset loading and canonical record schemas.\\"\\"\\"\\n", "src/data/averitec.py": "\\"\\"\\"AVeriTeC claim loading, normalization, and audit helpers.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom collections import Counter\\nfrom pathlib import Path\\nfrom typing import Any, Iterable\\n\\nfrom src.data.schema import ClaimRecord, EvidenceItem\\n\\nCANONICAL_LABELS = (\\n    \\"Supported\\",\\n    \\"Refuted\\",\\n    \\"Not Enough Evidence\\",\\n    \\"Conflicting Evidence\\",\\n)\\n\\n_LABEL_ALIASES = {\\n    \\"supported\\": \\"Supported\\",\\n    \\"refuted\\": \\"Refuted\\",\\n    \\"not enough evidence\\": \\"Not Enough Evidence\\",\\n    \\"conflicting evidence/cherrypicking\\": \\"Conflicting Evidence\\",\\n    \\"conflicting evidence/cherry-picking\\": \\"Conflicting Evidence\\",\\n    \\"conflicting evidence\\": \\"Conflicting Evidence\\",\\n}\\n\\n\\ndef normalize_label(label: str | None) -> str | None:\\n    \\"\\"\\"Return the project label corresponding to an official AVeriTeC label.\\"\\"\\"\\n    if label is None:\\n        return None\\n    cleaned = \\" \\".join(label.strip().split()).casefold()\\n    try:\\n        return _LABEL_ALIASES[cleaned]\\n    except KeyError as exc:\\n        raise ValueError(f\\"Unsupported AVeriTeC label: {label!r}\\") from exc\\n\\n\\ndef _read_json_or_jsonl(path: Path) -> list[dict[str, Any]]:\\n    with path.open(\\"r\\", encoding=\\"utf-8\\") as handle:\\n        text = handle.read().strip()\\n    if not text:\\n        return []\\n    parsed = json.loads(text)\\n    if isinstance(parsed, list):\\n        return parsed\\n    if isinstance(parsed, dict):\\n        return [parsed]\\n    raise ValueError(f\\"Expected a JSON object or list in {path}\\")\\n\\n\\ndef _answers_to_evidence(questions: Iterable[dict[str, Any]], claim_id: str) -> tuple[EvidenceItem, ...]:\\n    \\"\\"\\"Expose annotated answers for gold-evidence verification only.\\n\\n    These records must never be used to construct the retrieval corpus.\\n    \\"\\"\\"\\n    evidence: list[EvidenceItem] = []\\n    for question_index, question in enumerate(questions):\\n        question_text = str(question.get(\\"question\\") or \\"\\").strip()\\n        for answer_index, answer in enumerate(question.get(\\"answers\\") or []):\\n            answer_text = str(answer.get(\\"answer\\") or \\"\\").strip()\\n            if not answer_text:\\n                continue\\n            source_url = str(answer.get(\\"source_url\\") or \\"\\").strip() or None\\n            evidence.append(\\n                EvidenceItem(\\n                    passage_id=f\\"{claim_id}:q{question_index}:a{answer_index}\\",\\n                    text=answer_text,\\n                    url=source_url,\\n                    metadata={\\n                        \\"question\\": question_text,\\n                        \\"answer_type\\": answer.get(\\"answer_type\\"),\\n                        \\"source_medium\\": answer.get(\\"source_medium\\"),\\n                        \\"is_gold_answer\\": True,\\n                    },\\n                )\\n            )\\n    return tuple(evidence)\\n\\n\\ndef raw_to_record(raw: dict[str, Any], split: str, index: int) -> ClaimRecord:\\n    \\"\\"\\"Convert one official AVeriTeC object into the project record shape.\\"\\"\\"\\n    claim = str(raw.get(\\"claim\\") or \\"\\").strip()\\n    if not claim:\\n        raise ValueError(f\\"{split}[{index}] has an empty claim\\")\\n    claim_id = str(raw.get(\\"claim_id\\") or f\\"averitec:{split}:{index:05d}\\")\\n    label = normalize_label(raw.get(\\"label\\"))\\n    questions = raw.get(\\"questions\\") or []\\n    if not isinstance(questions, list):\\n        raise ValueError(f\\"{claim_id} has a non-list questions field\\")\\n    source_urls = tuple(\\n        item.url for item in _answers_to_evidence(questions, claim_id) if item.url is not None\\n    )\\n    return ClaimRecord(\\n        claim_id=claim_id,\\n        claim=claim,\\n        label=label,\\n        source_urls=tuple(dict.fromkeys(source_urls)),\\n        evidence=_answers_to_evidence(questions, claim_id),\\n        metadata={\\n            \\"dataset\\": \\"AVeriTeC\\",\\n            \\"split\\": split,\\n            \\"source_index\\": index,\\n            \\"claim_date\\": raw.get(\\"claim_date\\"),\\n            \\"speaker\\": raw.get(\\"speaker\\"),\\n            \\"reporting_source\\": raw.get(\\"reporting_source\\"),\\n            \\"location_iso_code\\": raw.get(\\"location_ISO_code\\"),\\n            \\"required_reannotation\\": raw.get(\\"required_reannotation\\"),\\n            \\"fact_checking_article\\": raw.get(\\"fact_checking_article\\"),\\n        },\\n    )\\n\\n\\ndef load_split(path: str | Path, split: str) -> list[ClaimRecord]:\\n    \\"\\"\\"Load one official AVeriTeC split from a JSON file.\\"\\"\\"\\n    source_path = Path(path)\\n    records = [raw_to_record(raw, split, index) for index, raw in enumerate(_read_json_or_jsonl(source_path))]\\n    ids = [record.claim_id for record in records]\\n    if len(ids) != len(set(ids)):\\n        raise ValueError(f\\"Duplicate claim IDs found in {source_path}\\")\\n    return records\\n\\n\\ndef load_split_with_rejections(\\n    path: str | Path, split: str\\n) -> tuple[list[ClaimRecord], list[dict[str, Any]]]:\\n    \\"\\"\\"Load a split while preserving an audit trail for unusable source rows.\\n\\n    An empty claim cannot be evaluated by any project model, so it is excluded\\n    from prepared output. Any other schema or label error remains a hard failure\\n    until it is explicitly understood.\\n    \\"\\"\\"\\n    source_path = Path(path)\\n    records: list[ClaimRecord] = []\\n    rejections: list[dict[str, Any]] = []\\n    for index, raw in enumerate(_read_json_or_jsonl(source_path)):\\n        if not str(raw.get(\\"claim\\") or \\"\\").strip():\\n            rejections.append(\\n                {\\n                    \\"source_index\\": index,\\n                    \\"reason\\": \\"empty_claim\\",\\n                    \\"source_label\\": raw.get(\\"label\\"),\\n                    \\"fact_checking_article\\": raw.get(\\"fact_checking_article\\"),\\n                }\\n            )\\n            continue\\n        records.append(raw_to_record(raw, split, index))\\n    ids = [record.claim_id for record in records]\\n    if len(ids) != len(set(ids)):\\n        raise ValueError(f\\"Duplicate claim IDs found in {source_path}\\")\\n    return records, rejections\\n\\n\\ndef split_audit(records: Iterable[ClaimRecord]) -> dict[str, Any]:\\n    \\"\\"\\"Return a small, serializable audit report for a prepared split.\\"\\"\\"\\n    materialized = list(records)\\n    return {\\n        \\"record_count\\": len(materialized),\\n        \\"label_counts\\": dict(sorted(Counter(record.label for record in materialized).items())),\\n        \\"records_with_gold_answers\\": sum(bool(record.evidence) for record in materialized),\\n        \\"gold_answer_count\\": sum(len(record.evidence) for record in materialized),\\n    }\\n\\n\\ndef write_jsonl(records: Iterable[ClaimRecord], path: str | Path) -> None:\\n    destination = Path(path)\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    with destination.open(\\"w\\", encoding=\\"utf-8\\", newline=\\"\\\\n\\") as handle:\\n        for record in records:\\n            handle.write(json.dumps(record.to_dict(), ensure_ascii=False) + \\"\\\\n\\")\\n", "src/data/corpus.py": "\\"\\"\\"Candidate-document validation, passage chunking, and URL-based qrels.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport re\\nfrom collections import Counter, defaultdict\\nfrom collections.abc import Iterable\\nfrom typing import Any\\nfrom urllib.parse import urlsplit, urlunsplit\\n\\n\\ndef canonical_url(value: str | None) -> str | None:\\n    \\"\\"\\"Normalize only URL details that should not change document identity.\\"\\"\\"\\n    if not value:\\n        return None\\n    raw = value.strip()\\n    if not raw:\\n        return None\\n    parts = urlsplit(raw)\\n    if not parts.scheme or not parts.netloc:\\n        return raw.rstrip(\\"/\\")\\n    return urlunsplit((parts.scheme.casefold(), parts.netloc.casefold(), parts.path.rstrip(\\"/\\"), parts.query, \\"\\"))\\n\\n\\ndef chunk_text(text: str, max_words: int = 160, overlap_words: int = 40) -> list[str]:\\n    \\"\\"\\"Split a document into deterministic overlapping word passages.\\"\\"\\"\\n    if max_words < 1 or overlap_words < 0 or overlap_words >= max_words:\\n        raise ValueError(\\"Require max_words > 0 and 0 <= overlap_words < max_words\\")\\n    words = re.findall(r\\"\\\\S+\\", text)\\n    if not words:\\n        return []\\n    passages = []\\n    step = max_words - overlap_words\\n    for start in range(0, len(words), step):\\n        passage = \\" \\".join(words[start : start + max_words]).strip()\\n        if passage:\\n            passages.append(passage)\\n        if start + max_words >= len(words):\\n            break\\n    return passages\\n\\n\\ndef documents_to_passages(\\n    documents: Iterable[dict[str, Any]], max_words: int = 160, overlap_words: int = 40\\n) -> tuple[list[dict[str, Any]], dict[str, Any]]:\\n    \\"\\"\\"Convert normalized candidate documents into provenance-preserving passages.\\n\\n    Input documents must have a non-empty `text` and should have a stable `url`\\n    or `document_id`. This function is intentionally format-agnostic; the\\n    AVeriTeC archive inspector identifies how to create these documents.\\n    \\"\\"\\"\\n    passages: list[dict[str, Any]] = []\\n    rejected = 0\\n    for document_index, document in enumerate(documents):\\n        text = str(document.get(\\"text\\") or \\"\\").strip()\\n        if not text:\\n            rejected += 1\\n            continue\\n        document_id = str(document.get(\\"document_id\\") or document.get(\\"url\\") or f\\"document:{document_index}\\")\\n        url = document.get(\\"url\\")\\n        for passage_index, passage_text in enumerate(chunk_text(text, max_words, overlap_words)):\\n            passages.append(\\n                {\\n                    \\"passage_id\\": f\\"{document_id}:p{passage_index:04d}\\",\\n                    \\"text\\": passage_text,\\n                    \\"url\\": url,\\n                    \\"metadata\\": {\\n                        \\"document_id\\": document_id,\\n                        \\"chunk_index\\": passage_index,\\n                        \\"chunk_max_words\\": max_words,\\n                        \\"chunk_overlap_words\\": overlap_words,\\n                    },\\n                }\\n            )\\n    return passages, {\\"passage_count\\": len(passages), \\"rejected_empty_document_count\\": rejected}\\n\\n\\ndef build_url_qrels(\\n    claims: Iterable[dict[str, Any]], passages: Iterable[dict[str, Any]]\\n) -> tuple[list[dict[str, Any]], dict[str, Any]]:\\n    \\"\\"\\"Build *source-URL* relevance judgments from AVeriTeC annotations.\\n\\n    This is an auditable weak passage-level proxy: every passage from a human\\n    annotated evidence URL is marked relevant. Reports must call it URL-based\\n    evidence coverage, not exact human passage relevance.\\n    \\"\\"\\"\\n    # Claims are compact; materialize them once.  The large corpus itself is\\n    # streamed and only passage IDs for annotated URLs are retained.\\n    claim_rows = list(claims)\\n    annotated_urls = {\\n        canonical_url(url)\\n        for claim in claim_rows\\n        for url in (claim.get(\\"source_urls\\") or [])\\n    }\\n    annotated_urls.discard(None)\\n    passage_ids_by_url: dict[str, list[tuple[str, str | None]]] = defaultdict(list)\\n    for passage in passages:\\n        url = canonical_url(passage.get(\\"url\\"))\\n        if url in annotated_urls:\\n            candidate_claim_id = passage.get(\\"metadata\\", {}).get(\\"candidate_claim_id\\")\\n            passage_ids_by_url[url].append((str(passage[\\"passage_id\\"]), str(candidate_claim_id) if candidate_claim_id else None))\\n\\n    qrels: list[dict[str, Any]] = []\\n    unresolved_url_count = 0\\n    for claim in claim_rows:\\n        gold_urls = {canonical_url(url) for url in claim.get(\\"source_urls\\") or []}\\n        gold_urls.discard(None)\\n        relevant_ids = sorted(\\n            {\\n                passage_id\\n                for url in gold_urls\\n                for passage_id, candidate_claim_id in passage_ids_by_url.get(str(url), [])\\n                if candidate_claim_id is None or candidate_claim_id == str(claim[\\"claim_id\\"])\\n            }\\n        )\\n        unresolved = sorted(\\n            url\\n            for url in gold_urls\\n            if not any(\\n                candidate_claim_id is None or candidate_claim_id == str(claim[\\"claim_id\\"])\\n                for _, candidate_claim_id in passage_ids_by_url.get(str(url), [])\\n            )\\n        )\\n        unresolved_url_count += len(unresolved)\\n        qrels.append(\\n            {\\n                \\"claim_id\\": claim[\\"claim_id\\"],\\n                \\"relevant_passage_ids\\": relevant_ids,\\n                \\"judgment_type\\": \\"annotated_source_url_proxy\\",\\n                \\"annotated_source_url_count\\": len(gold_urls),\\n                \\"unresolved_source_urls\\": unresolved,\\n            }\\n        )\\n    audit = {\\n        \\"judgment_type\\": \\"annotated_source_url_proxy\\",\\n        \\"claim_count\\": len(qrels),\\n        \\"judged_claim_count\\": sum(bool(row[\\"relevant_passage_ids\\"]) for row in qrels),\\n        \\"unresolved_annotated_source_url_count\\": unresolved_url_count,\\n        \\"relevant_passage_count_distribution\\": dict(\\n            sorted(Counter(len(row[\\"relevant_passage_ids\\"]) for row in qrels).items())\\n        ),\\n    }\\n    return qrels, audit\\n", "src/data/evidence.py": "\\"\\"\\"Validated claim/evidence joins shared by baseline and new pipelines.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom typing import Any\\n\\nfrom src.utils.io import read_jsonl\\n\\n\\ndef index_claims(rows: list[dict], expected: set[str] | None = None) -> dict[str, dict]:\\n    result = {}\\n    for row in rows:\\n        cid = str(row.get(\\"claim_id\\") or \\"\\")\\n        if not cid or cid in result:\\n            raise ValueError(f\\"Missing or duplicate claim ID: {cid!r}\\")\\n        result[cid] = row\\n    if expected is not None and set(result) != expected:\\n        raise ValueError(f\\"Claim coverage differs: missing={sorted(expected-set(result))[:3]}, extra={sorted(set(result)-expected)[:3]}\\")\\n    return result\\n\\n\\ndef attach_evidence(records: list[dict[str, Any]], rankings_path, *, exact: bool = False) -> list[dict[str, Any]]:\\n    claims = index_claims(records)\\n    rankings = index_claims(read_jsonl(rankings_path), expected=set(claims) if exact else None)\\n    missing = set(claims) - set(rankings)\\n    if missing:\\n        raise ValueError(f\\"Rankings are missing {len(missing)} records, e.g. {sorted(missing)[:3]}\\")\\n    for cid in claims:\\n        row = rankings[cid]\\n        if row.get(\\"claim\\") and str(row[\\"claim\\"]).strip() != str(claims[cid][\\"claim\\"]).strip():\\n            raise ValueError(f\\"Claim text differs for {cid}\\")\\n        evidence = row.get(\\"retrieved\\") or []\\n        ids = [str(p.get(\\"passage_id\\") or \\"\\") for p in evidence]\\n        if len(ids) != len(set(ids)) or any(not pid for pid in ids):\\n            raise ValueError(f\\"Missing or duplicate passage ID for {cid}\\")\\n    return [{**row, \\"retrieved_evidence\\": rankings[str(row[\\"claim_id\\"])].get(\\"retrieved\\") or []} for row in records]\\n", "src/data/knowledge_store.py": "\\"\\"\\"Safe AVeriTeC knowledge-store archive inspection for Colab.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport csv\\nimport io\\nimport zipfile\\nimport re\\nfrom collections import Counter\\nfrom pathlib import Path, PurePosixPath\\nfrom typing import Any\\n\\nfrom src.utils.progress import progress\\n\\n\\nURL_KEYS = (\\"url\\", \\"source_url\\", \\"document_url\\")\\nTEXT_KEYS = (\\"text\\", \\"scraped_text\\", \\"content\\", \\"lines\\", \\"document\\", \\"body\\")\\n_CLAIM_FILE_PATTERN = re.compile(r\\"(?:^|/)(\\\\d+)\\\\.json(?:#\\\\d+)?$\\")\\n\\n\\ndef inspect_zip_archive(path: str | Path, sample_members: int = 10) -> dict[str, Any]:\\n    \\"\\"\\"Inspect archive structure without extracting large files or assuming schema.\\"\\"\\"\\n    archive_path = Path(path)\\n    with zipfile.ZipFile(archive_path) as archive:\\n        members = [member for member in archive.infolist() if not member.is_dir()]\\n        suffix_counts = Counter(PurePosixPath(member.filename).suffix.casefold() or \\"[no extension]\\" for member in members)\\n        samples = []\\n        inspected_members = members[:sample_members]\\n        with progress(total=len(inspected_members), description=\\"Inspecting archive\\", unit=\\"file\\") as bar:\\n            for member in inspected_members:\\n                sample: dict[str, Any] = {\\"name\\": member.filename, \\"compressed_size\\": member.compress_size, \\"file_size\\": member.file_size}\\n                suffix = PurePosixPath(member.filename).suffix.casefold()\\n                if suffix in {\\".json\\", \\".jsonl\\"} and member.file_size <= 2_000_000:\\n                    with archive.open(member) as handle:\\n                        text = handle.read().decode(\\"utf-8\\", errors=\\"replace\\")\\n                    try:\\n                        parsed = json.loads(text) if suffix == \\".json\\" else json.loads(next(line for line in text.splitlines() if line.strip()))\\n                    except json.JSONDecodeError:\\n                        # Official AVeriTeC stores JSONL rows in files named\\n                        # `N.json`; show the first row shape instead of calling it\\n                        # an opaque parse failure in the inspection report.\\n                        try:\\n                            parsed = json.loads(next(line for line in text.splitlines() if line.strip()))\\n                            sample[\\"jsonl_with_json_suffix\\"] = suffix == \\".json\\"\\n                        except (json.JSONDecodeError, StopIteration):\\n                            sample[\\"json_parse_error\\"] = True\\n                            parsed = None\\n                    except StopIteration:\\n                        sample[\\"json_parse_error\\"] = True\\n                        parsed = None\\n                    if parsed is not None:\\n                        sample[\\"json_type\\"] = type(parsed).__name__\\n                        if isinstance(parsed, dict):\\n                            sample[\\"json_keys\\"] = sorted(parsed)[:30]\\n                        elif isinstance(parsed, list) and parsed and isinstance(parsed[0], dict):\\n                            sample[\\"first_row_keys\\"] = sorted(parsed[0])[:30]\\n                elif suffix in {\\".tsv\\", \\".csv\\"} and member.file_size <= 2_000_000:\\n                    with archive.open(member) as handle:\\n                        sample[\\"header\\"] = handle.readline().decode(\\"utf-8\\", errors=\\"replace\\").strip()\\n                samples.append(sample)\\n                bar.update(1)\\n    return {\\n        \\"archive\\": str(archive_path),\\n        \\"archive_size_bytes\\": archive_path.stat().st_size,\\n        \\"member_count\\": len(members),\\n        \\"suffix_counts\\": dict(sorted(suffix_counts.items())),\\n        \\"sample_members\\": samples,\\n    }\\n\\n\\ndef _text_value(value: Any) -> str | None:\\n    if isinstance(value, str):\\n        return value.strip() or None\\n    if isinstance(value, list) and all(isinstance(item, str) for item in value):\\n        text = \\"\\\\n\\".join(item.strip() for item in value if item.strip())\\n        return text or None\\n    return None\\n\\n\\ndef _candidate_claim_index(source_member: str) -> int | None:\\n    \\"\\"\\"Read the zero-based claim index from official `output_<split>/N.json` names.\\"\\"\\"\\n    match = _CLAIM_FILE_PATTERN.search(source_member)\\n    return int(match.group(1)) if match else None\\n\\n\\ndef _documents_from_value(\\n    value: Any, source_member: str, *, url2text_mode: str = \\"sentence\\"\\n) -> list[dict[str, Any]]:\\n    \\"\\"\\"Interpret common URL/text JSON shapes without fabricating content.\\"\\"\\"\\n    if url2text_mode not in {\\"sentence\\", \\"source_document\\"}:\\n        raise ValueError(\\"url2text_mode must be \'sentence\' or \'source_document\'\\")\\n    if isinstance(value, list):\\n        documents = []\\n        for index, item in enumerate(value):\\n            documents.extend(\\n                _documents_from_value(item, f\\"{source_member}#{index}\\", url2text_mode=url2text_mode)\\n            )\\n        return documents\\n    if not isinstance(value, dict):\\n        return []\\n    url = next((str(value[key]).strip() for key in URL_KEYS if value.get(key)), None)\\n    # The official AVeriTeC knowledge store contains one JSONL file per claim.\\n    # Each row associates a URL with `url2text`, a list of extracted sentences.\\n    # Preserve the candidate-pool boundary so retrieval never crosses claims.\\n    sentence_values = value.get(\\"url2text\\")\\n    if url and isinstance(sentence_values, list) and all(isinstance(item, str) for item in sentence_values):\\n        candidate_claim_index = _candidate_claim_index(source_member)\\n        cleaned_sentences = [sentence.strip() for sentence in sentence_values if sentence.strip()]\\n        if url2text_mode == \\"source_document\\" and cleaned_sentences:\\n            # Each official JSONL row is a URL and its extracted sentences.\\n            # Keep that natural source-document unit, then let the later\\n            # passage builder produce overlapping 160-word retrieval chunks.\\n            # This is substantially smaller than making every sentence an\\n            # independent candidate while preserving the same evidence text.\\n            return [\\n                {\\n                    \\"document_id\\": f\\"{source_member}:source_document\\",\\n                    \\"url\\": url,\\n                    \\"text\\": \\"\\\\n\\".join(cleaned_sentences),\\n                    \\"metadata\\": {\\n                        \\"archive_member\\": source_member.split(\\"#\\", 1)[0],\\n                        \\"candidate_claim_index\\": candidate_claim_index,\\n                        \\"source_format\\": \\"averitec_url2text_source_document\\",\\n                    },\\n                }\\n            ]\\n        return [\\n            {\\n                \\"document_id\\": f\\"{source_member}:sentence:{sentence_index}\\",\\n                \\"url\\": url,\\n                \\"text\\": sentence.strip(),\\n                \\"metadata\\": {\\n                    \\"archive_member\\": source_member.split(\\"#\\", 1)[0],\\n                    \\"candidate_claim_index\\": candidate_claim_index,\\n                    \\"source_format\\": \\"averitec_url2text\\",\\n                },\\n            }\\n            for sentence_index, sentence in enumerate(sentence_values)\\n            if sentence.strip()\\n        ]\\n    text = next((_text_value(value[key]) for key in TEXT_KEYS if _text_value(value.get(key))), None)\\n    if url and text:\\n        return [{\\"document_id\\": f\\"{source_member}:{url}\\", \\"url\\": url, \\"text\\": text, \\"metadata\\": {\\"archive_member\\": source_member}}]\\n    # Some archives use a dictionary mapping source URL to extracted text/lines.\\n    documents = []\\n    for key, nested_value in value.items():\\n        if key.startswith((\\"http://\\", \\"https://\\")):\\n            nested_text = _text_value(nested_value)\\n            if nested_text:\\n                documents.append(\\n                    {\\"document_id\\": f\\"{source_member}:{key}\\", \\"url\\": key, \\"text\\": nested_text, \\"metadata\\": {\\"archive_member\\": source_member}}\\n                )\\n    return documents\\n\\n\\ndef normalize_zip_to_documents(\\n    archive_path: str | Path, *, url2text_mode: str = \\"sentence\\"\\n) -> tuple[list[dict[str, Any]], dict[str, Any]]:\\n    \\"\\"\\"Extract recognized JSON/JSONL/TSV/CSV URL-text records directly from a zip.\\n\\n    Unknown members are counted and omitted rather than guessed. Review the\\n    resulting audit before treating the documents as a valid evidence corpus.\\n    \\"\\"\\"\\n    source = Path(archive_path)\\n    documents: list[dict[str, Any]] = []\\n    audit: dict[str, Any] = {\\"archive\\": str(source), \\"member_counts\\": Counter(), \\"recognized_member_counts\\": Counter(), \\"skipped_member_samples\\": []}\\n    with zipfile.ZipFile(source) as archive:\\n        for member in archive.infolist():\\n            if member.is_dir():\\n                continue\\n            suffix = PurePosixPath(member.filename).suffix.casefold()\\n            audit[\\"member_counts\\"][suffix or \\"[no extension]\\"] += 1\\n            try:\\n                with archive.open(member) as binary_handle:\\n                    if suffix == \\".json\\":\\n                        value = json.load(io.TextIOWrapper(binary_handle, encoding=\\"utf-8\\", errors=\\"replace\\"))\\n                        parsed = _documents_from_value(value, member.filename, url2text_mode=url2text_mode)\\n                    elif suffix == \\".jsonl\\":\\n                        parsed = []\\n                        for index, line in enumerate(io.TextIOWrapper(binary_handle, encoding=\\"utf-8\\", errors=\\"replace\\")):\\n                            if line.strip():\\n                                parsed.extend(\\n                                    _documents_from_value(\\n                                        json.loads(line), f\\"{member.filename}#{index}\\", url2text_mode=url2text_mode\\n                                    )\\n                                )\\n                    elif suffix in {\\".tsv\\", \\".csv\\"}:\\n                        delimiter = \\"\\\\t\\" if suffix == \\".tsv\\" else \\",\\"\\n                        reader = csv.DictReader(io.TextIOWrapper(binary_handle, encoding=\\"utf-8\\", errors=\\"replace\\"), delimiter=delimiter)\\n                        parsed = _documents_from_value(list(reader), member.filename, url2text_mode=url2text_mode)\\n                    else:\\n                        parsed = []\\n                if parsed:\\n                    audit[\\"recognized_member_counts\\"][suffix] += 1\\n                    documents.extend(parsed)\\n                elif len(audit[\\"skipped_member_samples\\"]) < 20:\\n                    audit[\\"skipped_member_samples\\"].append(member.filename)\\n            except (json.JSONDecodeError, UnicodeError, csv.Error) as error:\\n                if len(audit[\\"skipped_member_samples\\"]) < 20:\\n                    audit[\\"skipped_member_samples\\"].append(f\\"{member.filename} ({type(error).__name__})\\")\\n    audit[\\"member_counts\\"] = dict(sorted(audit[\\"member_counts\\"].items()))\\n    audit[\\"recognized_member_counts\\"] = dict(sorted(audit[\\"recognized_member_counts\\"].items()))\\n    audit[\\"normalized_document_count\\"] = len(documents)\\n    audit[\\"documents_with_url\\"] = sum(bool(document.get(\\"url\\")) for document in documents)\\n    audit[\\"url2text_mode\\"] = url2text_mode\\n    return documents, audit\\n\\n\\ndef normalize_zip_to_jsonl(\\n    archive_path: str | Path,\\n    output_path: str | Path,\\n    *,\\n    max_documents: int | None = None,\\n    url2text_mode: str = \\"sentence\\",\\n) -> dict[str, Any]:\\n    \\"\\"\\"Write recognized archive records to JSONL without retaining the corpus in RAM.\\n\\n    This is the Colab-facing path.  It streams JSONL/CSV/TSV records and\\n    writes each normalized document immediately.  JSON members still need to\\n    be valid standalone JSON values, but no list of documents from the whole\\n    archive is accumulated in memory.\\n    \\"\\"\\"\\n    if max_documents is not None and max_documents < 1:\\n        raise ValueError(\\"max_documents must be positive when supplied\\")\\n    if url2text_mode not in {\\"sentence\\", \\"source_document\\"}:\\n        raise ValueError(\\"url2text_mode must be \'sentence\' or \'source_document\'\\")\\n    source = Path(archive_path)\\n    destination = Path(output_path)\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    audit: dict[str, Any] = {\\n        \\"archive\\": str(source),\\n        \\"member_counts\\": Counter(),\\n        \\"recognized_member_counts\\": Counter(),\\n        \\"skipped_member_samples\\": [],\\n        \\"max_documents\\": max_documents,\\n        \\"url2text_mode\\": url2text_mode,\\n    }\\n    normalized_document_count = 0\\n    stopped_early = False\\n\\n    with zipfile.ZipFile(source) as archive, destination.open(\\"w\\", encoding=\\"utf-8\\", newline=\\"\\\\n\\") as output:\\n        members = [member for member in archive.infolist() if not member.is_dir()]\\n        with progress(total=len(members), description=\\"Normalizing archive\\", unit=\\"file\\") as bar:\\n            for member in members:\\n                suffix = PurePosixPath(member.filename).suffix.casefold()\\n                audit[\\"member_counts\\"][suffix or \\"[no extension]\\"] += 1\\n                recognized_member = False\\n                try:\\n                    with archive.open(member) as binary_handle:\\n                        text_handle = io.TextIOWrapper(binary_handle, encoding=\\"utf-8\\", errors=\\"replace\\")\\n                        if suffix == \\".json\\":\\n                            # AVeriTeC calls its per-claim JSONL files `.json`.\\n                            # Try ordinary JSON first, then safely fall back to one\\n                            # JSON object per line without treating an invalid file\\n                            # as a generic format.\\n                            content = text_handle.read()\\n                            try:\\n                                parsed_rows = _documents_from_value(\\n                                    json.loads(content), member.filename, url2text_mode=url2text_mode\\n                                )\\n                                row_iterator = iter(parsed_rows)\\n                            except json.JSONDecodeError:\\n                                def json_rows_with_json_suffix() -> Any:\\n                                    for index, line in enumerate(content.splitlines()):\\n                                        if line.strip():\\n                                            yield from _documents_from_value(\\n                                                json.loads(line),\\n                                                f\\"{member.filename}#{index}\\",\\n                                                url2text_mode=url2text_mode,\\n                                            )\\n                                row_iterator = json_rows_with_json_suffix()\\n                        elif suffix == \\".jsonl\\":\\n                            def jsonl_rows() -> Any:\\n                                for index, line in enumerate(text_handle):\\n                                    if line.strip():\\n                                        yield from _documents_from_value(\\n                                            json.loads(line),\\n                                            f\\"{member.filename}#{index}\\",\\n                                            url2text_mode=url2text_mode,\\n                                        )\\n                            row_iterator = jsonl_rows()\\n                        elif suffix in {\\".tsv\\", \\".csv\\"}:\\n                            delimiter = \\"\\\\t\\" if suffix == \\".tsv\\" else \\",\\"\\n                            reader = csv.DictReader(text_handle, delimiter=delimiter)\\n                            def tabular_rows() -> Any:\\n                                for index, row in enumerate(reader):\\n                                    yield from _documents_from_value(\\n                                        row, f\\"{member.filename}#{index}\\", url2text_mode=url2text_mode\\n                                    )\\n                            row_iterator = tabular_rows()\\n                        else:\\n                            row_iterator = iter(())\\n                        for document in row_iterator:\\n                            output.write(json.dumps(document, ensure_ascii=False) + \\"\\\\n\\")\\n                            normalized_document_count += 1\\n                            recognized_member = True\\n                            if max_documents is not None and normalized_document_count >= max_documents:\\n                                stopped_early = True\\n                                break\\n                    if recognized_member:\\n                        audit[\\"recognized_member_counts\\"][suffix] += 1\\n                    elif len(audit[\\"skipped_member_samples\\"]) < 20:\\n                        audit[\\"skipped_member_samples\\"].append(member.filename)\\n                except (json.JSONDecodeError, UnicodeError, csv.Error) as error:\\n                    if len(audit[\\"skipped_member_samples\\"]) < 20:\\n                        audit[\\"skipped_member_samples\\"].append(f\\"{member.filename} ({type(error).__name__})\\")\\n                bar.update(1)\\n                bar.set_postfix(documents=normalized_document_count)\\n                if stopped_early:\\n                    break\\n\\n    audit[\\"member_counts\\"] = dict(sorted(audit[\\"member_counts\\"].items()))\\n    audit[\\"recognized_member_counts\\"] = dict(sorted(audit[\\"recognized_member_counts\\"].items()))\\n    audit[\\"normalized_document_count\\"] = normalized_document_count\\n    audit[\\"stopped_early_at_document_limit\\"] = stopped_early\\n    audit[\\"warning\\"] = (\\n        \\"This normalized corpus is intentionally incomplete because max_documents was used. \\"\\n        \\"Do not report retrieval metrics from it as full-corpus AVeriTeC results.\\"\\n        if stopped_early\\n        else None\\n    )\\n    return audit\\n", "src/data/quality.py": "\\"\\"\\"Data-quality and split-overlap audits for leakage-aware experiments.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport itertools\\nimport re\\nfrom collections import defaultdict\\nfrom collections.abc import Mapping, Sequence\\nfrom typing import Any\\n\\nfrom src.data.corpus import canonical_url\\nfrom src.data.schema import ClaimRecord\\n\\n\\ndef normalized_claim(text: str) -> str:\\n    return \\" \\".join(re.findall(r\\"\\\\w+\\", text.casefold()))\\n\\n\\ndef _values_to_ids(records: Sequence[ClaimRecord], value_fn) -> dict[str, set[str]]:\\n    output: dict[str, set[str]] = defaultdict(set)\\n    for record in records:\\n        for value in value_fn(record):\\n            if value:\\n                output[str(value)].add(record.claim_id)\\n    return output\\n\\n\\ndef split_overlap_audit(split_records: Mapping[str, Sequence[ClaimRecord]], sample_limit: int = 10) -> dict[str, Any]:\\n    \\"\\"\\"Report exact overlaps for inspection; it never silently drops examples.\\"\\"\\"\\n    claim_values = {\\n        split: _values_to_ids(records, lambda record: [normalized_claim(record.claim)])\\n        for split, records in split_records.items()\\n    }\\n    article_values = {\\n        split: _values_to_ids(records, lambda record: [canonical_url(record.metadata.get(\\"fact_checking_article\\"))])\\n        for split, records in split_records.items()\\n    }\\n    source_values = {\\n        split: _values_to_ids(records, lambda record: [canonical_url(url) for url in record.source_urls])\\n        for split, records in split_records.items()\\n    }\\n    report: dict[str, Any] = {}\\n    for left, right in itertools.combinations(sorted(split_records), 2):\\n        pair = f\\"{left}__{right}\\"\\n        report[pair] = {}\\n        for name, values in ((\\"normalized_claim\\", claim_values), (\\"fact_checking_article_url\\", article_values), (\\"annotated_source_url\\", source_values)):\\n            overlap = sorted(set(values[left]) & set(values[right]))\\n            report[pair][name] = {\\"overlap_count\\": len(overlap), \\"samples\\": overlap[:sample_limit]}\\n    return report\\n", "src/data/schema.py": "\\"\\"\\"Shared, serializable records used throughout the project.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom dataclasses import asdict, dataclass, field\\nfrom typing import Any\\n\\n\\n@dataclass(frozen=True)\\nclass EvidenceItem:\\n    \\"\\"\\"A retrievable evidence passage with its source provenance.\\"\\"\\"\\n\\n    passage_id: str\\n    text: str\\n    url: str | None = None\\n    domain: str | None = None\\n    score: float | None = None\\n    metadata: dict[str, Any] = field(default_factory=dict)\\n\\n    def to_dict(self) -> dict[str, Any]:\\n        return asdict(self)\\n\\n\\n@dataclass(frozen=True)\\nclass ClaimRecord:\\n    \\"\\"\\"Canonical claim record emitted by every dataset loader.\\"\\"\\"\\n\\n    claim_id: str\\n    claim: str\\n    label: str | None\\n    source_urls: tuple[str, ...] = ()\\n    evidence: tuple[EvidenceItem, ...] = ()\\n    metadata: dict[str, Any] = field(default_factory=dict)\\n\\n    def to_dict(self) -> dict[str, Any]:\\n        data = asdict(self)\\n        data[\\"source_urls\\"] = list(self.source_urls)\\n        return data\\n", "src/data/snopes.py": "\\"\\"\\"DisinfoMM Snopes filtering and conservative external-label normalization.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport csv\\nfrom collections import Counter\\nfrom pathlib import Path\\nfrom typing import Any, Iterable\\nfrom urllib.parse import urlparse\\n\\nfrom src.data.schema import ClaimRecord\\n\\nSNOPES_EXTERNAL_LABELS = (\\"Supported\\", \\"Refuted\\", \\"Not Enough Evidence\\")\\n\\n\\ndef normalize_snopes_label(evaluation: str | None) -> str | None:\\n    \\"\\"\\"Map DisinfoMM\'s harmonized verdict to the external three-class task.\\n\\n    The archival CSV retains spelling/capitalization variants. `None` means the\\n    source verdict cannot be mapped safely and should be excluded with an audit\\n    record rather than forced into a project class.\\n    \\"\\"\\"\\n    value = \\" \\".join((evaluation or \\"\\").strip().casefold().split())\\n    value = value.replace(\\"moslty\\", \\"mostly\\").replace(\\"exagerated\\", \\"exaggerated\\")\\n    if value == \\"true\\" or value.startswith(\\"mostly true\\"):\\n        return \\"Supported\\"\\n    if value == \\"false\\" or value.startswith(\\"mostly false\\"):\\n        return \\"Refuted\\"\\n    if value == \\"incomplete\\":\\n        return \\"Not Enough Evidence\\"\\n    return None\\n\\n\\ndef parse_source_urls(value: str | None) -> tuple[str, ...]:\\n    \\"\\"\\"Parse the historical comma-separated cited-source field conservatively.\\"\\"\\"\\n    urls: list[str] = []\\n    for candidate in (value or \\"\\").split(\\",\\"):\\n        url = candidate.strip().rstrip(\\".,;:)\\")\\n        parsed = urlparse(url)\\n        if parsed.scheme not in {\\"http\\", \\"https\\"} or not parsed.netloc:\\n            continue\\n        if url not in urls:\\n            urls.append(url)\\n    return tuple(urls)\\n\\n\\ndef row_to_record(row: dict[str, str], source_index: int) -> ClaimRecord:\\n    claim = (row.get(\\"Claim\\") or \\"\\").strip()\\n    if not claim:\\n        raise ValueError(\\"empty_claim\\")\\n    label = normalize_snopes_label(row.get(\\"Evaluation\\"))\\n    if label is None:\\n        raise ValueError(\\"unmappable_evaluation\\")\\n    source_id = (row.get(\\"Num\\") or str(source_index)).strip()\\n    return ClaimRecord(\\n        claim_id=f\\"snopes:{source_id}\\",\\n        claim=claim,\\n        label=label,\\n        source_urls=parse_source_urls(row.get(\\"Article sources\\")),\\n        metadata={\\n            \\"dataset\\": \\"DisinfoMM\\",\\n            \\"source\\": \\"snopes\\",\\n            \\"source_index\\": source_index,\\n            \\"source_id\\": source_id,\\n            \\"date\\": row.get(\\"Date\\"),\\n            \\"source_evaluation\\": row.get(\\"Evaluation\\"),\\n            \\"source_label\\": row.get(\\"Label\\"),\\n            \\"fact_check_url\\": row.get(\\"Article Link\\"),\\n            \\"declaration_url\\": row.get(\\"Declaration Link\\"),\\n            \\"keywords\\": row.get(\\"Keywords\\"),\\n            \\"tags\\": row.get(\\"Tags\\"),\\n            # Deliberately do not retain the post-hoc Explanation as model input.\\n            \\"has_explanation\\": bool((row.get(\\"Explanation\\") or \\"\\").strip()),\\n        },\\n    )\\n\\n\\ndef load_snopes_records(\\n    path: str | Path,\\n) -> tuple[list[ClaimRecord], dict[str, Any], list[dict[str, Any]]]:\\n    \\"\\"\\"Load only English Snopes rows and return records plus an audit trail.\\"\\"\\"\\n    source_path = Path(path)\\n    records: list[ClaimRecord] = []\\n    rejections: list[dict[str, Any]] = []\\n    raw_evaluations: Counter[str] = Counter()\\n    source_rows = 0\\n    snopes_rows = 0\\n    with source_path.open(\\"r\\", encoding=\\"utf-8-sig\\", newline=\\"\\") as handle:\\n        reader = csv.DictReader(handle)\\n        required = {\\"Num\\", \\"Website\\", \\"Claim\\", \\"Evaluation\\", \\"Article sources\\"}\\n        missing = required - set(reader.fieldnames or [])\\n        if missing:\\n            raise ValueError(f\\"DisinfoMM is missing required columns: {sorted(missing)}\\")\\n        for source_index, row in enumerate(reader):\\n            source_rows += 1\\n            if (row.get(\\"Website\\") or \\"\\").strip().casefold() != \\"snopes\\":\\n                continue\\n            snopes_rows += 1\\n            raw_evaluations[(row.get(\\"Evaluation\\") or \\"\\").strip()] += 1\\n            try:\\n                records.append(row_to_record(row, source_index))\\n            except ValueError as error:\\n                rejections.append(\\n                    {\\n                        \\"source_index\\": source_index,\\n                        \\"source_id\\": row.get(\\"Num\\"),\\n                        \\"reason\\": str(error),\\n                        \\"source_evaluation\\": row.get(\\"Evaluation\\"),\\n                    }\\n                )\\n    claim_ids = [record.claim_id for record in records]\\n    if len(claim_ids) != len(set(claim_ids)):\\n        raise ValueError(\\"Duplicate DisinfoMM Snopes record IDs after filtering\\")\\n    audit = {\\n        \\"dataset\\": \\"DisinfoMM\\",\\n        \\"filter\\": \\"Website == snopes\\",\\n        \\"source_row_count\\": source_rows,\\n        \\"snopes_row_count\\": snopes_rows,\\n        \\"accepted_record_count\\": len(records),\\n        \\"label_counts\\": dict(sorted(Counter(record.label for record in records).items())),\\n        \\"raw_evaluation_counts\\": dict(sorted(raw_evaluations.items())),\\n        \\"records_with_source_urls\\": sum(bool(record.source_urls) for record in records),\\n        \\"rejection_count\\": len(rejections),\\n    }\\n    return records, audit, rejections\\n", "src/evaluation/__init__.py": "\\"\\"\\"Shared evaluation functions for project baselines and models.\\"\\"\\"\\n", "src/evaluation/classification.py": "\\"\\"\\"Standardized four-way classification reporting.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom collections.abc import Iterable, Sequence\\nfrom typing import Any\\n\\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score\\n\\nfrom src.data.averitec import CANONICAL_LABELS\\n\\n\\ndef evaluate_predictions(\\n    predictions: Iterable[dict[str, Any]], labels: Sequence[str] = CANONICAL_LABELS\\n) -> dict[str, Any]:\\n    \\"\\"\\"Compute standardized verdict metrics from prediction artifacts.\\"\\"\\"\\n    materialized = list(predictions)\\n    if not materialized:\\n        raise ValueError(\\"Cannot evaluate an empty prediction list\\")\\n    true_labels = [str(row[\\"true_label\\"]) for row in materialized]\\n    predicted_labels = [str(row[\\"predicted_label\\"]) for row in materialized]\\n    invalid = (set(true_labels) | set(predicted_labels)) - set(labels)\\n    if invalid:\\n        raise ValueError(f\\"Predictions contain invalid labels: {sorted(invalid)}\\")\\n    report: dict[str, Any] = {\\n        \\"claim_count\\": len(materialized),\\n        \\"macro_f1\\": float(f1_score(true_labels, predicted_labels, labels=labels, average=\\"macro\\", zero_division=0)),\\n        \\"accuracy\\": float(accuracy_score(true_labels, predicted_labels)),\\n        \\"per_class\\": classification_report(\\n            true_labels,\\n            predicted_labels,\\n            labels=labels,\\n            output_dict=True,\\n            zero_division=0,\\n        ),\\n        \\"confusion_matrix\\": {\\n            \\"labels\\": list(labels),\\n            \\"matrix\\": confusion_matrix(true_labels, predicted_labels, labels=labels).tolist(),\\n        },\\n    }\\n    supported = \\"Supported\\"\\n    report[\\"decision_error_rates\\"] = {\\n        \\"false_reassurance_count\\": sum(true != supported and predicted == supported for true, predicted in zip(true_labels, predicted_labels, strict=True)),\\n        \\"false_reassurance_rate\\": sum(true != supported and predicted == supported for true, predicted in zip(true_labels, predicted_labels, strict=True)) / len(materialized),\\n        \\"false_alarm_count\\": sum(true == supported and predicted != supported for true, predicted in zip(true_labels, predicted_labels, strict=True)),\\n        \\"false_alarm_rate\\": sum(true == supported and predicted != supported for true, predicted in zip(true_labels, predicted_labels, strict=True)) / len(materialized),\\n    }\\n    if all(isinstance(row.get(\\"probabilities\\"), dict) for row in materialized):\\n        report[\\"calibration\\"] = calibration_metrics(materialized, labels)\\n    return report\\n\\n\\ndef calibration_metrics(predictions: Sequence[dict[str, Any]], labels: Sequence[str], bin_count: int = 10) -> dict[str, float | int]:\\n    \\"\\"\\"Compute multiclass Brier score and ECE from model probabilities.\\n\\n    LLM self-reported confidence is intentionally not accepted as a probability\\n    distribution and therefore does not receive a calibration score.\\n    \\"\\"\\"\\n    brier_total = 0.0\\n    confidence_bins: list[list[tuple[float, float]]] = [[] for _ in range(bin_count)]\\n    for row in predictions:\\n        probabilities = row[\\"probabilities\\"]\\n        values = {label: float(probabilities.get(label, 0.0)) for label in labels}\\n        brier_total += sum((values[label] - float(row[\\"true_label\\"] == label)) ** 2 for label in labels)\\n        predicted_label, confidence = max(values.items(), key=lambda item: item[1])\\n        bin_index = min(int(confidence * bin_count), bin_count - 1)\\n        confidence_bins[bin_index].append((confidence, float(predicted_label == row[\\"true_label\\"])))\\n    ece = 0.0\\n    for values in confidence_bins:\\n        if not values:\\n            continue\\n        avg_confidence = sum(value[0] for value in values) / len(values)\\n        avg_accuracy = sum(value[1] for value in values) / len(values)\\n        ece += len(values) / len(predictions) * abs(avg_confidence - avg_accuracy)\\n    return {\\"multiclass_brier_score\\": brier_total / len(predictions), \\"expected_calibration_error\\": ece, \\"ece_bin_count\\": bin_count}\\n", "src/evaluation/end_to_end.py": "\\"\\"\\"Diagnostic reports joining retrieval coverage and final verdict predictions.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom collections.abc import Iterable\\nfrom typing import Any\\n\\nfrom src.data.averitec import CANONICAL_LABELS\\nfrom src.evaluation.classification import evaluate_predictions\\nfrom src.retrieval.metrics import evidence_hit_at_k\\n\\n\\ndef annotate_retrieval_coverage(\\n    predictions: Iterable[dict[str, Any]], rankings: Iterable[dict[str, Any]], qrels: dict[str, list[str]], k: int = 3\\n) -> list[dict[str, Any]]:\\n    \\"\\"\\"Attach an auditable evidence-coverage status to each verdict prediction.\\n\\n    A prediction may explicitly identify the passages the verifier used.  This\\n    matters for the LLM reranking baseline: its selected three are the evidence\\n    set being diagnosed, rather than the original BM25 top three.\\n    \\"\\"\\"\\n    ranking_by_claim = {str(row[\\"claim_id\\"]): row for row in rankings}\\n    annotations = []\\n    for prediction in predictions:\\n        claim_id = str(prediction[\\"claim_id\\"])\\n        ranking = ranking_by_claim.get(claim_id)\\n        evidence_ids = prediction.get(\\"evidence_passage_ids\\")\\n        if evidence_ids is None and ranking is not None:\\n            evidence_ids = [item[\\"passage_id\\"] for item in ranking.get(\\"retrieved\\") or []]\\n        if evidence_ids is None or claim_id not in qrels:\\n            status = \\"unjudged\\"\\n        elif evidence_hit_at_k([str(item) for item in evidence_ids], qrels[claim_id], k):\\n            status = \\"evidence_covered\\"\\n        else:\\n            status = \\"evidence_missed\\"\\n        annotations.append(\\n            {\\n                \\"claim_id\\": claim_id,\\n                \\"retrieval_coverage\\": status,\\n                \\"evidence_passage_ids_used\\": [str(item) for item in evidence_ids or []],\\n            }\\n        )\\n    return annotations\\n\\n\\ndef evaluate_by_retrieval_coverage(\\n    predictions: Iterable[dict[str, Any]], rankings: Iterable[dict[str, Any]], qrels: dict[str, list[str]], k: int = 3\\n) -> dict[str, Any]:\\n    \\"\\"\\"Report verdict quality overall and conditioned on URL/passage coverage.\\n\\n    This is diagnostic conditioning, not a causal claim that retrieval alone\\n    caused every miss or hit.\\n    \\"\\"\\"\\n    materialized_predictions = list(predictions)\\n    coverage_by_claim = {\\n        row[\\"claim_id\\"]: row[\\"retrieval_coverage\\"]\\n        for row in annotate_retrieval_coverage(materialized_predictions, rankings, qrels, k)\\n    }\\n    covered = [row for row in materialized_predictions if coverage_by_claim[row[\\"claim_id\\"]] == \\"evidence_covered\\"]\\n    missed = [row for row in materialized_predictions if coverage_by_claim[row[\\"claim_id\\"]] == \\"evidence_missed\\"]\\n    unjudged = [row for row in materialized_predictions if coverage_by_claim[row[\\"claim_id\\"]] == \\"unjudged\\"]\\n    report: dict[str, Any] = {\\n        \\"k\\": k,\\n        \\"judged_prediction_count\\": len(covered) + len(missed),\\n        \\"unjudged_prediction_count\\": len(unjudged),\\n        \\"retrieval_covered_count\\": len(covered),\\n        \\"retrieval_missed_count\\": len(missed),\\n        \\"warning\\": \\"Conditioning on retrieval coverage is diagnostic, not causal attribution.\\",\\n    }\\n    if covered:\\n        report[\\"verdict_metrics_when_evidence_covered\\"] = evaluate_predictions(covered, CANONICAL_LABELS)\\n    if missed:\\n        report[\\"verdict_metrics_when_evidence_missed\\"] = evaluate_predictions(missed, CANONICAL_LABELS)\\n    return report\\n", "src/evaluation/paired.py": "\\"\\"\\"Paired uncertainty conditional on the observed class counts.\\"\\"\\"\\nimport numpy as np\\n\\nfrom src.data.averitec import CANONICAL_LABELS\\nfrom src.data.evidence import index_claims\\n\\n\\ndef paired_delta(reference, candidate, trials=1000, seed=718):\\n    a, b = index_claims(reference), index_claims(candidate)\\n    if set(a) != set(b) or any(a[c][\\"true_label\\"] != b[c][\\"true_label\\"] for c in a):\\n        raise ValueError(\\"Paired evaluations must cover identical claims and gold labels\\")\\n    keys = sorted(a)\\n    label = {k: i for i, k in enumerate(CANONICAL_LABELS)}\\n    y = np.array([label[a[c][\\"true_label\\"]] for c in keys])\\n    old = np.array([label[a[c][\\"predicted_label\\"]] for c in keys])\\n    new = np.array([label[b[c][\\"predicted_label\\"]] for c in keys])\\n\\n    def f1(gold, predicted):\\n        counts = np.bincount(4 * gold + predicted, minlength=16).reshape(4, 4)\\n        denom = counts.sum(0) + counts.sum(1)\\n        return np.divide(2 * counts.diagonal(), denom, out=np.zeros(4), where=denom > 0).mean()\\n\\n    rng = np.random.default_rng(seed)\\n    groups = [np.flatnonzero(y == i) for i in range(4)]\\n    differences = []\\n    for _ in range(trials):\\n        sample = np.concatenate([rng.choice(g, len(g)) for g in groups if len(g)])\\n        differences.append(f1(y[sample], new[sample]) - f1(y[sample], old[sample]))\\n    return {\\"delta_macro_f1\\": float(f1(y, new) - f1(y, old)), \\"interval_95pct\\": np.quantile(differences, [.025, .975]).tolist(),\\n            \\"changed_predictions\\": int((old != new).sum()), \\"fixed\\": int(((old != y) & (new == y)).sum()),\\n            \\"broken\\": int(((old == y) & (new != y)).sum()),\\n            \\"caveat\\": \\"Class-stratified paired bootstrap; excludes training randomness, related claims and repeated dev inspection.\\"}\\n", "src/evaluation/splits.py": "\\"\\"\\"Train-only holdouts for diagnostics; never uses official dev claims.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom datetime import datetime\\nfrom math import ceil\\nimport re\\nimport unicodedata\\nfrom typing import Any\\n\\nimport numpy as np\\nfrom sklearn.model_selection import StratifiedGroupKFold, train_test_split\\n\\n\\ndef parse_claim_date(value: Any) -> str | None:\\n    if not value:\\n        return None\\n    text = str(value).strip()\\n    for fmt in (\\"%Y-%m-%d\\", \\"%d-%m-%Y\\", \\"%d/%m/%Y\\", \\"%Y/%m/%d\\", \\"%m/%d/%Y\\", \\"%B %d, %Y\\"):\\n        try:\\n            return datetime.strptime(text[:10] if fmt == \\"%Y-%m-%d\\" else text, fmt).date().isoformat()\\n        except ValueError:\\n            pass\\n    return None\\n\\n\\ndef selection_split(records: list[dict], fraction: float, seed: int, policy: str = \\"stratified\\") -> tuple[np.ndarray, np.ndarray]:\\n    indices = np.arange(len(records))\\n    if policy == \\"stratified\\":\\n        return train_test_split(indices, test_size=fraction, random_state=seed, stratify=[r[\\"label\\"] for r in records])\\n    if policy == \\"grouped\\":\\n        groups = [\\" \\".join(re.findall(r\\"\\\\w+\\", unicodedata.normalize(\\"NFKC\\", r[\\"claim\\"]).casefold())) for r in records]\\n        labels = [r[\\"label\\"] for r in records]\\n        folds = min(max(2, round(1 / fraction)), min(labels.count(label) for label in set(labels)))\\n        if folds < 2:\\n            raise ValueError(\\"Too few examples for a grouped train holdout\\")\\n        splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)\\n        return next(splitter.split(indices, labels, groups))\\n    if policy != \\"temporal\\":\\n        raise ValueError(f\\"Unknown validation policy: {policy}\\")\\n    dates = [parse_claim_date(r.get(\\"metadata\\", {}).get(\\"claim_date\\")) for r in records]\\n    if any(date is None for date in dates):\\n        raise ValueError(\\"Temporal holdout requires a parseable claim_date for every train claim; audit missing dates first\\")\\n    ordered = sorted(indices, key=lambda i: (dates[i], str(records[i][\\"claim_id\\"])))\\n    boundary = dates[ordered[max(1, len(records) - ceil(fraction * len(records)))]]\\n    train = np.asarray([i for i in ordered if dates[i] < boundary], dtype=int)\\n    holdout = np.asarray([i for i in ordered if dates[i] >= boundary], dtype=int)\\n    if not len(train) or not len(holdout):\\n        raise ValueError(\\"Temporal dates do not admit a nonempty earlier/later split\\")\\n    return train, holdout\\n", "src/external/__init__.py": "\\"\\"\\"External-corpus preparation with provenance and leakage safeguards.\\"\\"\\"\\n", "src/external/sampling.py": "\\"\\"\\"Deterministic, label-stratified sampling for manageable external corpora.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport random\\nfrom collections import Counter, defaultdict\\nfrom collections.abc import Iterable, Sequence\\nfrom typing import Any\\n\\n\\ndef stratified_sample(\\n    records: Iterable[dict[str, Any]], labels: Sequence[str], total_size: int, seed: int\\n) -> list[dict[str, Any]]:\\n    \\"\\"\\"Sample as evenly as possible across labels without oversampling records.\\n\\n    Selection uses labels only to create an evaluation set with meaningful\\n    coverage of rare classes. Labels remain unavailable to every retriever and\\n    verifier at inference time.\\n    \\"\\"\\"\\n    if total_size < 1:\\n        raise ValueError(\\"total_size must be positive\\")\\n    buckets: dict[str, list[dict[str, Any]]] = defaultdict(list)\\n    for record in records:\\n        label = record.get(\\"label\\")\\n        if label in labels:\\n            buckets[str(label)].append(record)\\n    unknown = set(labels) - set(buckets)\\n    if unknown:\\n        raise ValueError(f\\"Cannot sample absent labels: {sorted(unknown)}\\")\\n    rng = random.Random(seed)\\n    for bucket in buckets.values():\\n        rng.shuffle(bucket)\\n\\n    selected: list[dict[str, Any]] = []\\n    remaining = total_size\\n    active_labels = list(labels)\\n    while active_labels and remaining:\\n        quota = max(1, remaining // len(active_labels))\\n        next_active: list[str] = []\\n        for label in active_labels:\\n            take = min(quota, len(buckets[label]))\\n            selected.extend(buckets[label][:take])\\n            buckets[label] = buckets[label][take:]\\n            remaining -= take\\n            if buckets[label]:\\n                next_active.append(label)\\n            if not remaining:\\n                break\\n        active_labels = next_active\\n    if remaining:\\n        raise ValueError(\\"Requested more records than are available\\")\\n    return sorted(selected, key=lambda record: str(record[\\"claim_id\\"]))\\n\\n\\ndef sample_audit(records: Iterable[dict[str, Any]]) -> dict[str, Any]:\\n    materialized = list(records)\\n    return {\\n        \\"record_count\\": len(materialized),\\n        \\"label_counts\\": dict(sorted(Counter(str(record[\\"label\\"]) for record in materialized).items())),\\n        \\"source_url_count\\": sum(len(record.get(\\"source_urls\\") or []) for record in materialized),\\n    }\\n", "src/external/snopes_manifest.py": "\\"\\"\\"Build an auditable cited-source manifest before any web fetching occurs.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom collections import Counter\\nfrom collections.abc import Iterable\\nfrom typing import Any\\nfrom urllib.parse import urlparse\\n\\n\\n# Snopes itself is always prohibited as evaluation evidence. Other fact-checking\\n# domains are flagged for review rather than silently discarded: they may be\\n# useful provenance but often contain verdict leakage.\\nREJECTED_EVIDENCE_DOMAINS = {\\"snopes.com\\", \\"www.snopes.com\\"}\\nFACT_CHECK_REVIEW_DOMAINS = {\\n    \\"politifact.com\\",\\n    \\"www.politifact.com\\",\\n    \\"factcheck.org\\",\\n    \\"www.factcheck.org\\",\\n    \\"fullfact.org\\",\\n    \\"fullfact.org\\",\\n    \\"leadstories.com\\",\\n    \\"www.leadstories.com\\",\\n    \\"checkyourfact.com\\",\\n    \\"www.checkyourfact.com\\",\\n    \\"africacheck.org\\",\\n    \\"www.africacheck.org\\",\\n}\\n\\n\\ndef domain_for_url(url: str) -> str:\\n    return urlparse(url).netloc.casefold().removeprefix(\\"www.\\")\\n\\n\\ndef source_screening_status(domain: str) -> str:\\n    normalized = domain.casefold().removeprefix(\\"www.\\")\\n    if normalized in {item.removeprefix(\\"www.\\") for item in REJECTED_EVIDENCE_DOMAINS}:\\n        return \\"excluded_fact_check_source\\"\\n    if normalized in {item.removeprefix(\\"www.\\") for item in FACT_CHECK_REVIEW_DOMAINS}:\\n        return \\"requires_fact_check_review\\"\\n    return \\"unreviewed\\"\\n\\n\\ndef build_source_manifest(records: Iterable[dict[str, Any]]) -> tuple[list[dict[str, Any]], dict[str, Any]]:\\n    \\"\\"\\"Deduplicate cited URLs while retaining every claim that cited each URL.\\"\\"\\"\\n    by_url: dict[str, dict[str, Any]] = {}\\n    for record in records:\\n        for url in record.get(\\"source_urls\\") or []:\\n            url = str(url)\\n            domain = domain_for_url(url)\\n            entry = by_url.setdefault(\\n                url,\\n                {\\n                    \\"url\\": url,\\n                    \\"domain\\": domain,\\n                    \\"screening_status\\": source_screening_status(domain),\\n                    \\"claim_ids\\": [],\\n                },\\n            )\\n            claim_id = str(record[\\"claim_id\\"])\\n            if claim_id not in entry[\\"claim_ids\\"]:\\n                entry[\\"claim_ids\\"].append(claim_id)\\n    manifest = sorted(by_url.values(), key=lambda row: row[\\"url\\"])\\n    status_counts = Counter(row[\\"screening_status\\"] for row in manifest)\\n    domain_counts = Counter(row[\\"domain\\"] for row in manifest)\\n    audit = {\\n        \\"unique_source_url_count\\": len(manifest),\\n        \\"screening_status_counts\\": dict(sorted(status_counts.items())),\\n        \\"top_domains\\": [\\n            {\\"domain\\": domain, \\"url_count\\": count}\\n            for domain, count in domain_counts.most_common(25)\\n        ],\\n    }\\n    return manifest, audit\\n", "src/retrieval/__init__.py": "\\"\\"\\"Evidence retrieval implementations and metrics.\\"\\"\\"\\n", "src/retrieval/answer_aware.py": "\\"\\"\\"Pure helpers for extractive, question-conditioned evidence selection.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\nfrom typing import Any\\n\\nfrom src.data.corpus import canonical_url\\n\\n\\nFEATURE_KEYS = (\\n    \\"max_answer_margin\\",\\n    \\"mean_answer_margin\\",\\n    \\"bm25_reciprocal_rank\\",\\n    \\"question_guided_top3\\",\\n    \\"answer_word_count\\",\\n)\\n\\n\\ndef choose_candidates(\\n    bm25_row: dict[str, Any], question_row: dict[str, Any], *, limit: int, bm25_keep: int\\n) -> list[dict[str, Any]]:\\n    \\"\\"\\"Use only prediction-time rankings; retain a lexical safety net.\\"\\"\\"\\n    if limit < 1 or not 0 <= bm25_keep <= limit:\\n        raise ValueError(\\"Require limit > 0 and 0 <= bm25_keep <= limit\\")\\n    bm25 = bm25_row.get(\\"retrieved\\") or []\\n    by_id = {str(item[\\"passage_id\\"]): item for item in bm25}\\n    if len(by_id) != len(bm25):\\n        raise ValueError(\\"Duplicate passage ID in BM25 ranking\\")\\n    selected = {str(item[\\"passage_id\\"]) for item in (question_row.get(\\"retrieved\\") or [])[:3]}\\n    ordered_ids = [str(item[\\"passage_id\\"]) for item in bm25[:bm25_keep]]\\n    ordered_ids += [str(item[\\"passage_id\\"]) for item in (question_row.get(\\"retrieved\\") or [])[:3]]\\n    ordered_ids += [str(value) for value in question_row.get(\\"shortlist_passage_ids\\") or []]\\n    ordered_ids += [str(item[\\"passage_id\\"]) for item in bm25]\\n    seen: set[str] = set()\\n    chosen: list[dict[str, Any]] = []\\n    bm25_positions = {str(item[\\"passage_id\\"]): index + 1 for index, item in enumerate(bm25)}\\n    for passage_id in ordered_ids:\\n        if passage_id in seen or passage_id not in by_id:\\n            continue\\n        seen.add(passage_id)\\n        item = by_id[passage_id]\\n        if not str(item.get(\\"text\\") or \\"\\").strip():\\n            continue\\n        chosen.append({\\n            **item,\\n            \\"bm25_rank\\": bm25_positions[passage_id],\\n            \\"question_guided_top3\\": int(passage_id in selected),\\n        })\\n        if len(chosen) >= limit:\\n            break\\n    return chosen\\n\\n\\ndef best_extractable_span(\\n    passage: str,\\n    offsets: list[tuple[int, int]],\\n    sequence_ids: list[int | None],\\n    start_logits: list[float],\\n    end_logits: list[float],\\n    *,\\n    cls_index: int,\\n    max_answer_tokens: int,\\n    max_logit_candidates: int = 20,\\n) -> tuple[str, float]:\\n    \\"\\"\\"Return a verbatim span and SQuAD2-style answer-vs-null margin.\\"\\"\\"\\n    if not (len(offsets) == len(sequence_ids) == len(start_logits) == len(end_logits)):\\n        raise ValueError(\\"Token arrays must have identical lengths\\")\\n    null_score = start_logits[cls_index] + end_logits[cls_index]\\n    if max_answer_tokens < 1 or max_logit_candidates < 1:\\n        raise ValueError(\\"Span and logit candidate limits must be positive\\")\\n    context_tokens = [index for index, segment in enumerate(sequence_ids)\\n                      if segment == 1 and offsets[index][1] > offsets[index][0]]\\n    top_starts = sorted(context_tokens, key=lambda index: -start_logits[index])[:max_logit_candidates]\\n    top_ends = sorted(context_tokens, key=lambda index: -end_logits[index])[:max_logit_candidates]\\n    best_score = -math.inf\\n    best_text = \\"\\"\\n    for start in top_starts:\\n        for end in top_ends:\\n            if end < start or end >= start + max_answer_tokens:\\n                continue\\n            begin_char, end_char = offsets[start][0], offsets[end][1]\\n            if not 0 <= begin_char < end_char <= len(passage):\\n                continue\\n            score = start_logits[start] + end_logits[end]\\n            if score > best_score:\\n                best_score = score\\n                best_text = passage[begin_char:end_char]\\n    if not best_text:\\n        return \\"\\", -100.0\\n    return best_text, float(best_score - null_score)\\n\\n\\ndef source_excerpt(passage: str, answer: str, *, max_chars: int = 330) -> str:\\n    \\"\\"\\"Keep the exact source substring around a candidate answer.\\"\\"\\"\\n    if max_chars < 1:\\n        raise ValueError(\\"max_chars must be positive\\")\\n    position = passage.find(answer) if answer else -1\\n    if position < 0:\\n        return passage[:max_chars].strip()\\n    start = max(0, position - max(0, (max_chars - len(answer)) // 2))\\n    end = min(len(passage), start + max_chars)\\n    start = max(0, end - max_chars)\\n    return passage[start:end].strip()\\n\\n\\ndef qa_text(question: str, answer: str, passage: str, margin: float) -> str:\\n    excerpt = source_excerpt(passage, answer if margin > 0 else \\"\\")\\n    if margin > 0 and answer:\\n        return f\\"Question: {question}\\\\nCandidate answer (verbatim): {answer}\\\\nSource excerpt: {excerpt}\\"\\n    return f\\"Question: {question}\\\\nNo confident extractive answer. Source excerpt: {excerpt}\\"\\n\\n\\ndef feature_vector(item: dict[str, Any]) -> list[float]:\\n    return [float(item[key]) for key in FEATURE_KEYS]\\n\\n\\ndef rank_with_source_diversity(items: list[dict[str, Any]], scores: list[float], *, k: int = 3) -> list[dict[str, Any]]:\\n    if len(items) != len(scores):\\n        raise ValueError(\\"One score is required per passage\\")\\n    ordered = sorted(range(len(items)), key=lambda index: (-scores[index], str(items[index][\\"passage_id\\"])))\\n    first: list[int] = []\\n    seen_sources: set[str] = set()\\n    for index in ordered:\\n        item = items[index]\\n        source = canonical_url(item.get(\\"url\\")) or str(item[\\"passage_id\\"])\\n        if source not in seen_sources:\\n            seen_sources.add(source)\\n            first.append(index)\\n            if len(first) == k:\\n                break\\n    if len(first) < min(k, len(items)):\\n        first.extend(index for index in ordered if index not in first)\\n    indices = first[:k] + [index for index in ordered if index not in first[:k]]\\n    return [{**items[index], \\"selector_score\\": float(scores[index])} for index in indices]\\n", "src/retrieval/answer_followup.py": "\\"\\"\\"Bounded answer-focused follow-up, using cached evidence rather than a corpus scan.\\n\\nQuestion generation and span extraction never receive verdicts or annotated\\nanswers. The extractive model is externally pretrained and stays frozen.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport csv\\nimport json\\nimport math\\nfrom collections import Counter\\nfrom pathlib import Path\\n\\nfrom src.data.corpus import canonical_url\\nfrom src.data.evidence import index_claims\\nfrom src.evaluation.splits import parse_claim_date, selection_split\\nfrom src.retrieval.answer_aware import best_extractable_span, source_excerpt\\nfrom src.retrieval.evidence_quality import EvidencePolicy, clean_candidates\\nfrom src.retrieval.question_guided import parse_questions\\nfrom src.utils.checkpoints import append_journal, atomic_jsonl, bind_run, read_journal\\nfrom src.utils.io import read_jsonl, write_json\\nfrom src.utils.progress import progress\\nfrom src.utils.runs import sha256_file\\n\\n\\nQUESTION_PROMPT = (\\n    \\"You break down a factual claim into two short, independently answerable \\"\\n    \\"verification questions. Do not decide if the claim is true. Do not use \\"\\n    \'outside knowledge. Return JSON only: {\\"questions\\": [\\"...\\", \\"...\\"]}.\'\\n)\\n\\n\\ndef resolved_revision(model, requested, saved):\\n    if requested or saved:\\n        return requested or saved\\n    from huggingface_hub import model_info\\n    return model_info(model).sha\\n\\n\\ndef valid_questions(value):\\n    return (isinstance(value, list) and len(value) == 2\\n            and all(isinstance(q, str) and 3 <= len(q.strip()) <= 400 for q in value)\\n            and value[0].strip().casefold() != value[1].strip().casefold())\\n\\n\\ndef cached_questions(path, claims_path, records):\\n    \\"\\"\\"Accept only the known claim-only generator\'s artifacts with matching inputs.\\"\\"\\"\\n    if path is None or not path.is_file():\\n        return {}\\n    config_path = path.parent / \\"question_guided_config.json\\"\\n    if not config_path.is_file():\\n        raise ValueError(f\\"Question cache has no provenance: {config_path}\\")\\n    config = json.loads(config_path.read_text(encoding=\\"utf-8\\"))\\n    if (config.get(\\"script\\") != \\"run_question_guided_retrieval\\"\\n            or config.get(\\"claims_sha256\\") != sha256_file(claims_path)\\n            or config.get(\\"question_prompt\\") != QUESTION_PROMPT):\\n        raise ValueError(f\\"Question cache provenance/claim hash differs: {path}\\")\\n    expected = index_claims(records)\\n    result = {}\\n    for row in read_jsonl(path):\\n        cid = row.get(\\"claim_id\\")\\n        if cid not in expected or cid in result or row.get(\\"claim\\") != expected[cid][\\"claim\\"]:\\n            raise ValueError(f\\"Question cache claim ID/text mismatch: {cid}\\")\\n        if valid_questions(row.get(\\"questions\\")):\\n            result[cid] = {\\"claim_id\\": cid, \\"claim\\": expected[cid][\\"claim\\"], \\"questions\\": row[\\"questions\\"],\\n                           \\"question_origin\\": \\"previous_claim_only_generator\\", \\"fallback\\": row[\\"questions\\"][0].startswith(\\"What evidence supports\\")}\\n    return result\\n\\n\\nclass QuestionGenerator:\\n    def __init__(self, name, revision, device, max_new_tokens):\\n        import torch\\n        from transformers import AutoModelForCausalLM, AutoTokenizer\\n        torch.manual_seed(718)\\n        self.torch, self.device, self.max_new_tokens = torch, device, max_new_tokens\\n        self.tokenizer = AutoTokenizer.from_pretrained(name, revision=revision)\\n        self.model = AutoModelForCausalLM.from_pretrained(name, revision=revision,\\n            torch_dtype=torch.float16 if device.startswith(\\"cuda\\") else torch.float32).to(device).eval()\\n\\n    def __call__(self, claim):\\n        messages = [{\\"role\\": \\"system\\", \\"content\\": QUESTION_PROMPT}, {\\"role\\": \\"user\\", \\"content\\": claim}]\\n        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)\\n        batch = self.tokenizer(text, return_tensors=\\"pt\\").to(self.device)\\n        with self.torch.inference_mode():\\n            output = self.model.generate(**batch, do_sample=False, max_new_tokens=self.max_new_tokens,\\n                                         pad_token_id=self.tokenizer.eos_token_id)\\n        response = self.tokenizer.decode(output[0, batch.input_ids.shape[1]:], skip_special_tokens=True)\\n        questions = parse_questions(response, claim)\\n        fallback = not valid_questions(questions) or questions[0].startswith(\\"What evidence supports\\")\\n        if not valid_questions(questions):\\n            questions = [f\\"What evidence supports or contradicts this claim: {claim[:200]}?\\",\\n                         f\\"Which facts establish the names, dates or numbers in this claim: {claim[:200]}?\\"]\\n        return questions, fallback\\n\\n\\ndef prepare_questions(args, *, generator_factory=QuestionGenerator):\\n    config_path = args.output / \\"questions.json\\"\\n    previous = json.loads(config_path.read_text()) if config_path.is_file() else {}\\n    revision = resolved_revision(args.model, args.revision, previous.get(\\"revision\\"))\\n    paths = {s: args.data_root / \\"processed/averitec\\" / f\\"{s}.jsonl\\" for s in (\\"train\\", \\"dev\\")}\\n    caches = {s: getattr(args, s + \\"_question_cache\\", None) for s in paths}\\n    config = {\\"version\\": 1, \\"model\\": args.model, \\"revision\\": revision, \\"prompt\\": QUESTION_PROMPT,\\n              \\"claim_hashes\\": {s: sha256_file(p) for s, p in paths.items()}, \\"seed\\": 718,\\n              \\"cache_hashes\\": {s: {\\"rows\\": sha256_file(p), \\"config\\": sha256_file(p.parent / \\"question_guided_config.json\\")}\\n                               for s, p in caches.items() if p is not None and p.is_file()},\\n              \\"max_new_tokens\\": args.max_new_tokens, \\"implementation_sha256\\": sha256_file(Path(__file__))}\\n    bind_run(config_path, config)\\n    generator = None\\n    totals = {}\\n    for split, path in paths.items():\\n        records = read_jsonl(path)\\n        expected = index_claims(records)\\n        journal = args.output / f\\"{split}_questions.partial.jsonl\\"\\n        done = read_journal(journal, set(expected))\\n        reusable = cached_questions(caches[split], path, records)\\n        with progress(total=len(records), initial=len(done), description=f\\"Claim-only questions / {split}\\", unit=\\"claim\\") as bar:\\n            for record in records:\\n                cid = record[\\"claim_id\\"]\\n                if cid in done:\\n                    continue\\n                if cid in reusable:\\n                    row = reusable[cid]\\n                else:\\n                    if generator is None:\\n                        print(f\\"Loading question generator {args.model}@{revision}\\", flush=True)\\n                        generator = generator_factory(args.model, revision, args.device, args.max_new_tokens)\\n                    questions, fallback = generator(record[\\"claim\\"])\\n                    row = {\\"claim_id\\": cid, \\"claim\\": record[\\"claim\\"], \\"questions\\": questions,\\n                           \\"question_origin\\": \\"new_claim_only_generator\\", \\"fallback\\": fallback}\\n                append_journal(journal, [row])\\n                done[cid] = row\\n                bar.update(1)\\n        ordered = [done[r[\\"claim_id\\"]] for r in records]\\n        atomic_jsonl(ordered, args.output / f\\"{split}_questions.jsonl\\")\\n        totals[split] = {\\"claim_count\\": len(records), \\"fallback_count\\": sum(r[\\"fallback\\"] for r in ordered),\\n                         \\"origins\\": dict(Counter(r[\\"question_origin\\"] for r in ordered))}\\n    write_json({\\"config\\": config, \\"totals\\": totals, \\"finalized\\": True}, args.output / \\"questions_report.json\\")\\n    print(json.dumps(totals, indent=2))\\n\\n\\ndef candidate_union(raw, selected):\\n    \\"\\"\\"At most six passages, with original top-three passages first.\\"\\"\\"\\n    result = {}\\n    for item in [*raw[:3], *selected[:3]]:\\n        pid = str(item.get(\\"passage_id\\") or \\"\\")\\n        if not pid:\\n            raise ValueError(\\"Missing passage ID\\")\\n        if pid in result and result[pid].get(\\"text\\") != item.get(\\"text\\"):\\n            raise ValueError(f\\"Same passage ID has different source text: {pid}\\")\\n        result.setdefault(pid, dict(item))\\n    return list(result.values())\\n\\n\\nclass ExtractiveReader:\\n    def __init__(self, name, revision, device, batch_size, max_length):\\n        import torch\\n        from transformers import AutoModelForQuestionAnswering, AutoTokenizer\\n        self.torch, self.device, self.batch_size, self.max_length = torch, device, batch_size, max_length\\n        self.tokenizer = AutoTokenizer.from_pretrained(name, revision=revision, use_fast=True)\\n        if not self.tokenizer.is_fast:\\n            raise ValueError(\\"Extractive offsets require a fast tokenizer\\")\\n        self.model = AutoModelForQuestionAnswering.from_pretrained(name, revision=revision).to(device).eval()\\n\\n    def __call__(self, questions, candidates):\\n        jobs = [(pi, qi, question[:400], str(item.get(\\"text\\") or \\"\\"))\\n                for pi, item in enumerate(candidates) for qi, question in enumerate(questions)]\\n        answers = [[] for _ in candidates]\\n        for start in range(0, len(jobs), self.batch_size):\\n            batch = jobs[start:start + self.batch_size]\\n            encoded = self.tokenizer([j[2] for j in batch], [j[3] for j in batch], padding=True,\\n                truncation=\\"only_second\\", max_length=self.max_length, return_offsets_mapping=True, return_tensors=\\"pt\\")\\n            offsets = encoded.pop(\\"offset_mapping\\").tolist()\\n            segments = [encoded.sequence_ids(i) for i in range(len(batch))]\\n            with self.torch.inference_mode(), self.torch.autocast(device_type=\\"cuda\\", dtype=self.torch.float16, enabled=self.device.startswith(\\"cuda\\")):\\n                output = self.model(**{k: v.to(self.device) for k, v in encoded.items()})\\n            starts, ends = output.start_logits.float().cpu().tolist(), output.end_logits.float().cpu().tolist()\\n            for i, (pi, qi, question, text) in enumerate(batch):\\n                if not all(math.isfinite(v) for v in starts[i] + ends[i]):\\n                    raise ValueError(\\"Nonfinite QA logits; cached rows are preserved\\")\\n                ids = encoded[\\"input_ids\\"][i].tolist()\\n                cls_index = ids.index(self.tokenizer.cls_token_id)\\n                answer, margin = best_extractable_span(text, offsets[i], segments[i], starts[i], ends[i],\\n                    cls_index=cls_index, max_answer_tokens=24)\\n                answers[pi].append({\\"question_index\\": qi, \\"question\\": question, \\"answer\\": answer, \\"margin\\": margin})\\n        return answers\\n\\n\\ndef pack_answers(item, answers):\\n    \\"\\"\\"Keep two questions and their exact source spans; never generate factual text.\\"\\"\\"\\n    text = str(item.get(\\"text\\") or \\"\\")\\n    lines = []\\n    for answer in answers:\\n        span, margin = answer[\\"answer\\"], answer[\\"margin\\"]\\n        if span and span not in text:\\n            raise ValueError(\\"Extracted answer is not a substring of its source\\")\\n        if not math.isfinite(margin):\\n            raise ValueError(\\"Nonfinite answer-vs-null margin\\")\\n        confident = bool(span) and margin > 0\\n        lines += [f\\"Verification question: {answer[\'question\']}\\",\\n                  f\\"Candidate source span: {span}\\" if confident else \\"No confident extractive answer for this question.\\",\\n                  \\"Verbatim source context: \\" + source_excerpt(text, span if confident else \\"\\", max_chars=220)]\\n    return {**item, \\"source_text\\": text, \\"text\\": \\"\\\\n\\".join(lines), \\"extracted_answers\\": answers,\\n            \\"teacher_text_matches\\": False}\\n\\n\\ndef select_answer_evidence(claim, date, candidates, answers):\\n    \\"\\"\\"Fixed coverage heuristic; no qrels, labels, learned weights or dev tuning.\\"\\"\\"\\n    packed = [pack_answers(p, a) for p, a in zip(candidates, answers, strict=True)]\\n    eligible, _ = clean_candidates(claim, date, candidates, EvidencePolicy(max_candidates=6, candidate_depth=6))\\n    eligible_ids = {p[\\"passage_id\\"] for p in eligible}\\n    remaining = [p for p in packed if p[\\"passage_id\\"] in eligible_ids]\\n    chosen, covered, sources = [], set(), set()\\n    for _ in range(min(3, len(remaining))):\\n        def score(p):\\n            answer_scores = [max(0.0, min(1.0, a[\\"margin\\"] / 12)) if a[\\"answer\\"] else 0.0 for a in p[\\"extracted_answers\\"]]\\n            fresh = max((s for a, s in zip(p[\\"extracted_answers\\"], answer_scores) if a[\\"question_index\\"] not in covered), default=0.0)\\n            nli = p.get(\\"nli_probabilities\\", {})\\n            stance = max(nli.get(\\"entailment\\", 0), nli.get(\\"contradiction\\", 0))\\n            source = canonical_url(p.get(\\"url\\")) or p[\\"passage_id\\"]\\n            prior = 1 / math.log2(max(1, int(p.get(\\"original_rank\\", 100))) + 2)\\n            return max(answer_scores, default=0) + .35 * fresh + .15 * stance + .1 * prior + .1 * (source not in sources)\\n        best = sorted(remaining, key=lambda p: (-score(p), str(p[\\"passage_id\\"])))[0]\\n        chosen.append({**best, \\"answer_selection_score\\": score(best)})\\n        sources.add(canonical_url(best.get(\\"url\\")) or best[\\"passage_id\\"])\\n        covered.update(a[\\"question_index\\"] for a in best[\\"extracted_answers\\"] if a[\\"answer\\"] and a[\\"margin\\"] > 0)\\n        remaining.remove(best)\\n    return chosen\\n\\n\\ndef prepare_answers(args, *, reader_factory=ExtractiveReader):\\n    parent_path = args.parent_prepared / \\"preparation.json\\"\\n    parent = json.loads(parent_path.read_text(encoding=\\"utf-8\\"))\\n    parent_report = json.loads((args.parent_prepared / \\"preparation_report.json\\").read_text(encoding=\\"utf-8\\"))\\n    if parent.get(\\"limit\\") or not parent_report.get(\\"finalized\\"):\\n        raise ValueError(\\"Reuse a fully finalized prior evidence preparation\\")\\n    config_path = args.output / \\"preparation.json\\"\\n    previous = json.loads(config_path.read_text()) if config_path.is_file() else {}\\n    revision = resolved_revision(args.model, args.revision, previous.get(\\"qa_revision\\"))\\n    paths = {f\\"{s}_{kind}\\": path for s in (\\"train\\", \\"dev\\") for kind, path in (\\n        (\\"claims\\", args.data_root / \\"processed/averitec\\" / f\\"{s}.jsonl\\"),\\n        (\\"raw\\", args.parent_prepared / \\"raw\\" / f\\"{s}_rankings.jsonl\\"),\\n        (\\"clean_nli\\", args.parent_prepared / \\"clean_nli\\" / f\\"{s}_rankings.jsonl\\"),\\n        (\\"questions\\", args.questions / f\\"{s}_questions.jsonl\\"))}\\n    for s in (\\"train\\", \\"dev\\"):\\n        if parent[\\"input_hashes\\"][s + \\"_claims\\"] != sha256_file(paths[s + \\"_claims\\"]):\\n            raise ValueError(\\"Official claim input differs from cached NLI preparation\\")\\n    config = {\\"version\\": 2, \\"nli_model\\": parent[\\"nli_model\\"], \\"revision\\": parent[\\"revision\\"],\\n              \\"max_length\\": parent[\\"max_length\\"], \\"claim_budget\\": parent[\\"claim_budget\\"],\\n              \\"nli_label_order\\": parent.get(\\"nli_label_order\\"), \\"limit\\": args.limit,\\n              \\"qa_model\\": args.model, \\"qa_revision\\": revision, \\"qa_max_length\\": args.max_length,\\n              \\"batch_size\\": args.batch_size, \\"parent_sha256\\": sha256_file(parent_path),\\n              \\"input_hashes\\": {k: sha256_file(p) for k, p in paths.items()},\\n              \\"question_config_sha256\\": sha256_file(args.questions / \\"questions.json\\"),\\n              \\"distillation_incompatible_modes\\": [\\"qa_same3\\", \\"qa_selected3\\"],\\n              \\"implementation_sha256\\": sha256_file(Path(__file__))}\\n    bind_run(config_path, config)\\n    reader, totals = None, {}\\n    for split in (\\"train\\", \\"dev\\"):\\n        all_records = read_jsonl(paths[split + \\"_claims\\"])\\n        all_ids = set(index_claims(all_records))\\n        records = all_records[:args.limit] if args.limit else all_records\\n        raw = index_claims(read_jsonl(paths[split + \\"_raw\\"]), expected=all_ids)\\n        selected = index_claims(read_jsonl(paths[split + \\"_clean_nli\\"]), expected=all_ids)\\n        questions = index_claims(read_jsonl(paths[split + \\"_questions\\"]), expected=all_ids)\\n        journal = args.output / f\\"{split}_answers.partial.jsonl\\"\\n        done = read_journal(journal, {r[\\"claim_id\\"] for r in records})\\n        with progress(total=len(records), initial=len(done), description=f\\"Extract exact-source answers / {split}\\", unit=\\"claim\\") as bar:\\n            for record in records:\\n                cid = record[\\"claim_id\\"]\\n                if cid in done:\\n                    continue\\n                for source in (raw[cid], selected[cid], questions[cid]):\\n                    if source.get(\\"claim\\") != record[\\"claim\\"]:\\n                        raise ValueError(f\\"Claim text mismatch: {cid}\\")\\n                qs = questions[cid][\\"questions\\"]\\n                if not valid_questions(qs):\\n                    raise ValueError(f\\"Invalid question cache: {cid}\\")\\n                candidates = candidate_union(raw[cid][\\"retrieved\\"], selected[cid][\\"retrieved\\"])\\n                if candidates:\\n                    if reader is None:\\n                        print(f\\"Loading frozen extractive model {args.model}@{revision}\\", flush=True)\\n                        reader = reader_factory(args.model, revision, args.device, args.batch_size, args.max_length)\\n                    answers = reader(qs, candidates)\\n                    if len(answers) != len(candidates) or any(len(a) != 2 for a in answers):\\n                        raise ValueError(\\"Need an answer result for both questions for every candidate\\")\\n                else:\\n                    answers = []\\n                packed = {p[\\"passage_id\\"]: pack_answers(p, a) for p, a in zip(candidates, answers, strict=True)}\\n                date = parse_claim_date((record.get(\\"metadata\\") or {}).get(\\"claim_date\\"))\\n                row = {\\"claim_id\\": cid, \\"claim\\": record[\\"claim\\"], \\"questions\\": qs, \\"raw\\": raw[cid][\\"retrieved\\"],\\n                       \\"qa_same3\\": [packed[p[\\"passage_id\\"]] for p in raw[cid][\\"retrieved\\"][:3]],\\n                       \\"qa_selected3\\": select_answer_evidence(record[\\"claim\\"], date, candidates, answers),\\n                       \\"candidate_count\\": len(candidates), \\"answered_pairs\\": sum(a[\\"margin\\"] > 0 and bool(a[\\"answer\\"]) for aa in answers for a in aa)}\\n                append_journal(journal, [row])\\n                done[cid] = row\\n                bar.update(1)\\n                bar.set_postfix(candidates=len(candidates), answered=row[\\"answered_pairs\\"])\\n        ordered = [done[r[\\"claim_id\\"]] for r in records]\\n        for mode in (\\"raw\\", \\"qa_same3\\", \\"qa_selected3\\"):\\n            atomic_jsonl([{\\"claim_id\\": r[\\"claim_id\\"], \\"claim\\": r[\\"claim\\"], \\"retrieved\\": r[mode]} for r in ordered],\\n                         args.output / mode / f\\"{split}_rankings.jsonl\\")\\n        totals[split] = {\\"claim_count\\": len(records), \\"candidate_count\\": sum(r[\\"candidate_count\\"] for r in ordered),\\n                         \\"answered_pairs\\": sum(r[\\"answered_pairs\\"] for r in ordered),\\n                         \\"empty_qa_selected3\\": sum(not r[\\"qa_selected3\\"] for r in ordered)}\\n        if split == \\"train\\" and not args.limit:\\n            fit, _ = selection_split(records, .1, 718, \\"grouped\\")\\n            fit_ids = {records[int(i)][\\"claim_id\\"] for i in fit}\\n            audit = [{\\"claim_id\\": r[\\"claim_id\\"], \\"claim\\": r[\\"claim\\"], \\"passage_id\\": p[\\"passage_id\\"], \\"url\\": p.get(\\"url\\"),\\n                      \\"question\\": a[\\"question\\"], \\"extracted_answer\\": a[\\"answer\\"], \\"answer_margin\\": a[\\"margin\\"],\\n                      \\"source_text\\": p[\\"source_text\\"], \\"human_decisive\\": \\"\\", \\"human_note\\": \\"\\"}\\n                     for r in ordered if r[\\"claim_id\\"] in fit_ids for p in r[\\"qa_same3\\"] for a in p[\\"extracted_answers\\"]]\\n            import random\\n            random.Random(718).shuffle(audit)\\n            with (args.output / \\"train_fit_answer_audit.csv\\").open(\\"w\\", encoding=\\"utf-8-sig\\", newline=\\"\\") as handle:\\n                writer = csv.DictWriter(handle, fieldnames=list(audit[0]) if audit else [\\"claim_id\\"])\\n                writer.writeheader()\\n                writer.writerows(audit[:80])\\n    write_json({\\"config\\": config, \\"totals\\": totals, \\"finalized\\": True,\\n                \\"supervision\\": \\"External frozen QA; claim-only questions. Only the later verdict reader is trained on AVeriTeC labels.\\",\\n                \\"caveat\\": \\"Answer-vs-null margin is uncalibrated answerability, not truth or decisive relevance.\\"}, args.output / \\"preparation_report.json\\")\\n    print(json.dumps(totals, indent=2))\\n", "src/retrieval/bm25.py": "\\"\\"\\"Deterministic BM25 retrieval over a static passage corpus.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport heapq\\nimport re\\nfrom collections.abc import Sequence\\n\\nfrom rank_bm25 import BM25Okapi\\n\\nfrom src.data.schema import EvidenceItem\\n\\n_TOKEN_PATTERN = re.compile(r\\"\\\\b\\\\w+\\\\b\\", flags=re.UNICODE)\\n\\n\\ndef tokenize(text: str) -> list[str]:\\n    \\"\\"\\"A transparent, deterministic tokenizer suitable for a lexical baseline.\\"\\"\\"\\n    return _TOKEN_PATTERN.findall(text.casefold())\\n\\n\\nclass BM25Retriever:\\n    \\"\\"\\"BM25 retriever with stable tie-breaking by passage ID.\\"\\"\\"\\n\\n    def __init__(self, corpus: Sequence[EvidenceItem]) -> None:\\n        if not corpus:\\n            raise ValueError(\\"BM25 corpus cannot be empty\\")\\n        self.corpus = tuple(corpus)\\n        self._bm25 = BM25Okapi([tokenize(item.text) for item in self.corpus])\\n\\n    def retrieve(self, claim: str, k: int = 3) -> list[EvidenceItem]:\\n        if k < 1:\\n            raise ValueError(\\"k must be at least 1\\")\\n        scores = self._bm25.get_scores(tokenize(claim))\\n        # Ranking a large candidate pool only needs the first k results.  A\\n        # full sort is O(n log n), whereas nsmallest is O(n log k).  This is\\n        # exactly the same ordering rule as the previous full sort, including\\n        # deterministic passage-ID tie breaking, but avoids sorting millions\\n        # of unreturned candidates in AVeriTeC pools.\\n        ranked_indices = heapq.nsmallest(\\n            min(k, len(self.corpus)),\\n            range(len(self.corpus)),\\n            key=lambda index: (-float(scores[index]), self.corpus[index].passage_id),\\n        )\\n        return [\\n            EvidenceItem(\\n                passage_id=self.corpus[index].passage_id,\\n                text=self.corpus[index].text,\\n                url=self.corpus[index].url,\\n                domain=self.corpus[index].domain,\\n                score=float(scores[index]),\\n                metadata=self.corpus[index].metadata,\\n            )\\n            for index in ranked_indices\\n        ]\\n", "src/retrieval/evidence_pipeline.py": "\\"\\"\\"Resumable preparation of raw, cleaned and NLI-ranked top-three evidence.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport math\\nfrom collections import Counter\\nfrom pathlib import Path\\n\\nfrom src.data.evidence import index_claims\\nfrom src.evaluation.splits import parse_claim_date\\nfrom src.retrieval.evidence_quality import (EvidencePolicy, clean_candidates, nli_label_order, pair_premise,\\n                                          publication_date, quality_features, select_evidence)\\nfrom src.utils.checkpoints import append_journal, atomic_jsonl, bind_run, read_journal\\nfrom src.utils.io import iter_jsonl, read_jsonl, write_json\\nfrom src.utils.progress import progress\\nfrom src.utils.runs import sha256_file\\n\\n\\ndef prepare_pipeline(args):\\n    import torch\\n    from huggingface_hub import model_info\\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer\\n\\n    if args.device.startswith(\\"cuda\\") and not torch.cuda.is_available():\\n        raise RuntimeError(\\"Choose a GPU Colab runtime before preparing NLI scores\\")\\n    output = args.output\\n    output.mkdir(parents=True, exist_ok=True)\\n    existing_path = output / \\"preparation.json\\"\\n    previous = json.loads(existing_path.read_text(encoding=\\"utf-8\\")) if existing_path.exists() else {}\\n    policy = EvidencePolicy(max_candidates=args.max_candidates, candidate_depth=args.candidate_depth)\\n    revision = args.revision or previous.get(\\"revision\\") or model_info(args.model).sha\\n    inputs = {\\"train_claims\\": args.train_claims, \\"dev_claims\\": args.dev_claims,\\n              \\"train_rankings\\": args.train_rankings, \\"dev_rankings\\": args.dev_rankings}\\n    config = {\\"version\\": 1, \\"nli_model\\": args.model, \\"revision\\": revision, \\"max_length\\": args.max_length,\\n              \\"claim_budget\\": args.claim_budget, \\"policy\\": policy.to_dict(), \\"limit\\": args.limit,\\n              \\"batch_size\\": args.batch_size, \\"device\\": args.device, \\"nli_label_order\\": args.nli_label_order,\\n              \\"input_hashes\\": {name: sha256_file(path) for name, path in inputs.items()},\\n              \\"implementation_sha256\\": sha256_file(Path(__file__)),\\n              \\"quality_policy_sha256\\": sha256_file(Path(__file__).with_name(\\"evidence_quality.py\\"))}\\n    bind_run(existing_path, config)\\n    records = {s: read_jsonl(inputs[f\\"{s}_claims\\"]) for s in (\\"train\\", \\"dev\\")}\\n    by_id = {s: index_claims(rows) for s, rows in records.items()}\\n    if set(by_id[\\"train\\"]) & set(by_id[\\"dev\\"]):\\n        raise ValueError(\\"Official train/dev claim IDs overlap\\")\\n    if args.limit:\\n        records = {s: rows[:args.limit] for s, rows in records.items()}\\n        by_id = {s: index_claims(rows) for s, rows in records.items()}\\n    model, tokenizer, order = None, None, None\\n    totals = {}\\n\\n    def score(claim, items):\\n        nonlocal model, tokenizer, order\\n        if not items:\\n            return\\n        if model is None:\\n            print(f\\"Loading NLI scorer {args.model}@{revision}\\", flush=True)\\n            tokenizer = AutoTokenizer.from_pretrained(args.model, revision=revision, use_fast=True)\\n            model = AutoModelForSequenceClassification.from_pretrained(args.model, revision=revision).to(args.device).eval()\\n            if int(model.config.num_labels) != 3:\\n                raise ValueError(\\"The passage scorer must have exactly three NLI labels\\")\\n            order = nli_label_order(model.config.id2label, args.nli_label_order)\\n        hypothesis = tokenizer.decode(tokenizer.encode(claim, add_special_tokens=False)[:args.claim_budget], skip_special_tokens=True)\\n        for start in range(0, len(items), args.batch_size):\\n            batch = items[start:start + args.batch_size]\\n            encoded = tokenizer([pair_premise(p) for p in batch], [hypothesis] * len(batch), padding=True,\\n                                truncation=\\"only_first\\", max_length=args.max_length, return_tensors=\\"pt\\")\\n            with torch.inference_mode(), torch.autocast(device_type=\\"cuda\\", dtype=torch.float16, enabled=args.device.startswith(\\"cuda\\")):\\n                values = model(**{k: v.to(args.device) for k, v in encoded.items()}).logits.float().softmax(-1).cpu().tolist()\\n            for item, vector in zip(batch, values, strict=True):\\n                if any(not math.isfinite(v) or not 0 <= v <= 1 for v in vector) or abs(sum(vector) - 1) > 1e-4:\\n                    raise ValueError(f\\"Non-finite NLI output for {item[\'passage_id\']}; checkpoint preserved\\")\\n                item[\\"nli_probabilities\\"] = dict(zip(order, vector, strict=True))\\n\\n    for split in (\\"train\\", \\"dev\\"):\\n        expected = set(by_id[split])\\n        journal = output / f\\"{split}.partial.jsonl\\"\\n        done = read_journal(journal, expected)\\n        seen = set()\\n        with progress(total=len(expected), initial=len(done), description=f\\"NLI evidence / {split}\\", unit=\\"claim\\") as bar:\\n            for ranking in iter_jsonl(inputs[f\\"{split}_rankings\\"]):\\n                cid = str(ranking.get(\\"claim_id\\") or \\"\\")\\n                if args.limit and cid not in expected:\\n                    continue\\n                if cid not in expected or cid in seen:\\n                    raise ValueError(f\\"Unexpected/duplicate ranking claim ID: {cid}\\")\\n                seen.add(cid)\\n                if cid in done:\\n                    continue\\n                record = by_id[split][cid]\\n                if ranking.get(\\"claim\\") and ranking[\\"claim\\"].strip() != record[\\"claim\\"].strip():\\n                    raise ValueError(f\\"Claim text differs for {cid}\\")\\n                candidates = ranking.get(\\"retrieved\\")\\n                if not isinstance(candidates, list):\\n                    raise ValueError(f\\"Malformed retrieved list: {cid}\\")\\n                date = parse_claim_date(record.get(\\"metadata\\", {}).get(\\"claim_date\\"))\\n                cleaned, rejected = clean_candidates(record[\\"claim\\"], date, candidates, policy)\\n                # Raw reference is the exact original top three, even if hygiene rejects it.\\n                raw = [{\\"passage_id\\": p[\\"passage_id\\"], \\"text\\": str(p.get(\\"text\\") or \\"\\"), \\"url\\": p.get(\\"url\\"),\\n                        \\"publication_date\\": publication_date(p), \\"original_rank\\": i + 1,\\n                        \\"quality\\": quality_features(record[\\"claim\\"], p, i + 1, date)} for i, p in enumerate(candidates[:3])]\\n                union = {p[\\"passage_id\\"]: p for p in [*cleaned, *raw]}\\n                score(record[\\"claim\\"], list(union.values()))\\n                for p in [*cleaned, *raw]:\\n                    p[\\"nli_probabilities\\"] = union[p[\\"passage_id\\"]][\\"nli_probabilities\\"]\\n                row = {\\"claim_id\\": cid, \\"claim\\": record[\\"claim\\"], \\"raw\\": raw,\\n                       \\"clean\\": select_evidence(cleaned, rerank=False), \\"clean_nli\\": select_evidence(cleaned, rerank=True),\\n                       \\"scored_candidates\\": [{**p, \\"eligible_after_cleaning\\": p[\\"passage_id\\"] in {q[\\"passage_id\\"] for q in cleaned}} for p in union.values()],\\n                       \\"rejections\\": rejected, \\"scored_pair_count\\": len(union), \\"missing_date\\": date is None}\\n                append_journal(journal, [row])\\n                done[cid] = row\\n                bar.update(1)\\n                bar.set_postfix(pairs=len(union), rejected=len(rejected))\\n        if seen != expected or set(done) != expected:\\n            raise ValueError(f\\"Incomplete {split} rankings: {len(done)} / {len(expected)}\\")\\n        ordered = [done[r[\\"claim_id\\"]] for r in records[split]]\\n        for mode in (\\"raw\\", \\"clean\\", \\"clean_nli\\"):\\n            atomic_jsonl([{\\"claim_id\\": r[\\"claim_id\\"], \\"claim\\": r[\\"claim\\"], \\"retrieved\\": r[mode]} for r in ordered],\\n                         output / mode / f\\"{split}_rankings.jsonl\\")\\n        counts = Counter(r[\\"reason\\"].split(\\":\\")[0] for row in ordered for r in row[\\"rejections\\"])\\n        totals[split] = {\\"claim_count\\": len(ordered), \\"scored_pairs\\": sum(r[\\"scored_pair_count\\"] for r in ordered),\\n                         \\"rejections_among_scanned_candidates\\": dict(counts), \\"missing_claim_dates\\": sum(r[\\"missing_date\\"] for r in ordered),\\n                         \\"empty_clean_pools\\": sum(not r[\\"clean\\"] for r in ordered)}\\n    report = {\\"config\\": config, \\"totals\\": totals, \\"finalized\\": True,\\n              \\"temporal_caveat\\": \\"Only explicit publication metadata is filtered. Unknown/updated pages are not certified historical snapshots.\\",\\n              \\"supervision\\": \\"Frozen external NLI model; no AVeriTeC verdicts, gold answers or qrels used in selection.\\"}\\n    write_json(report, output / \\"preparation_report.json\\")\\n    print(json.dumps(report, indent=2))\\n", "src/retrieval/evidence_quality.py": "\\"\\"\\"Prediction-time evidence hygiene and stance-aware selection.\\n\\nNever consumes verdicts, annotated answers, source_urls, qrels, or the original\\nfact-check URL. Unknown publication dates stay unknown. Future years in prose\\nare flagged, not treated as proof of publication after the claim.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport math\\nimport re\\nimport unicodedata\\nfrom dataclasses import asdict, dataclass\\nfrom urllib.parse import urlparse\\n\\nimport numpy as np\\n\\nfrom src.data.corpus import canonical_url\\nfrom src.evaluation.splits import parse_claim_date\\nfrom src.retrieval.question_guided import words\\n\\nQUALITY_NAMES = (\\"query_coverage\\", \\"number_overlap\\", \\"rank_prior\\", \\"satire_source\\", \\"factcheck_source\\", \\"publication_known\\", \\"future_year_mentioned\\", \\"search_results_source\\")\\nSATIRE_HOSTS = {\\"babylonbee.com\\", \\"theonion.com\\"}\\n\\n\\n@dataclass(frozen=True)\\nclass EvidencePolicy:\\n    max_candidates: int = 24\\n    candidate_depth: int = 100\\n    near_duplicate_threshold: float = 0.90\\n    remove_search_pages: bool = True\\n    remove_known_future_publications: bool = True\\n\\n    def to_dict(self):\\n        return asdict(self)\\n\\n\\ndef original_url(url: str | None) -> str:\\n    value = str(url or \\"\\")\\n    # Capture time is deliberately not used as publication time.\\n    match = re.match(r\\"https?://web\\\\.archive\\\\.org/web/\\\\d+(?:[a-z_]+)?/(https?://.*)\\", value, re.I)\\n    return match.group(1) if match else value\\n\\n\\ndef source_host(url: str | None) -> str:\\n    return (urlparse(original_url(url)).hostname or \\"\\").lower().removeprefix(\\"www.\\")\\n\\n\\ndef is_search_page(url: str | None) -> bool:\\n    parsed = urlparse(original_url(url))\\n    host = (parsed.hostname or \\"\\").lower().removeprefix(\\"www.\\")\\n    return bool((re.fullmatch(r\\"google\\\\.[a-z.]+\\", host) and parsed.path.startswith((\\"/search\\", \\"/url\\")))\\n                or (host in {\\"bing.com\\", \\"search.yahoo.com\\", \\"duckduckgo.com\\"} and (parsed.query or \\"/search\\" in parsed.path)))\\n\\n\\ndef publication_date(item: dict) -> str | None:\\n    metadata = item.get(\\"metadata\\") or {}\\n    for key in (\\"publication_date\\", \\"published_date\\", \\"published_at\\", \\"datePublished\\"):\\n        date = parse_claim_date(metadata.get(key) or item.get(key))\\n        if date:\\n            return date\\n    # URL paths and Wayback captures do not establish publication dates.\\n    return None\\n\\n\\ndef normalized_text(text: str) -> str:\\n    return \\" \\".join(re.findall(r\\"\\\\w+\\", unicodedata.normalize(\\"NFKC\\", text).casefold()))\\n\\n\\ndef shingles(text: str) -> set[tuple[str, ...]]:\\n    tokens = normalized_text(text).split()\\n    return {tuple(tokens[i:i + 5]) for i in range(max(1, len(tokens) - 4))}\\n\\n\\ndef quality_features(claim: str, item: dict, rank: int, claim_date: str | None) -> list[float]:\\n    text = str(item.get(\\"text\\") or \\"\\")\\n    query, terms = words(claim), words(text)\\n    numbers = set(re.findall(r\\"\\\\b\\\\d+(?:[.,]\\\\d+)*\\\\b\\", claim))\\n    passage_numbers = set(re.findall(r\\"\\\\b\\\\d+(?:[.,]\\\\d+)*\\\\b\\", text))\\n    host, url = source_host(item.get(\\"url\\")), original_url(item.get(\\"url\\")).lower()\\n    year = int(claim_date[:4]) if claim_date else None\\n    later = bool(year and any(int(y) > year for y in re.findall(r\\"\\\\b(20\\\\d{2})\\\\b\\", text)))\\n    return [len(query & terms) / max(1, len(query)), len(numbers & passage_numbers) / max(1, len(numbers)),\\n            1.0 / math.log2(rank + 2), float(host in SATIRE_HOSTS),\\n            float(bool(re.search(r\\"fact[-_]?check\\", url))), float(publication_date(item) is not None), float(later), float(is_search_page(item.get(\\"url\\")))]\\n\\n\\ndef pair_premise(item: dict) -> str:\\n    host = source_host(item.get(\\"url\\")) or \\"unknown\\"\\n    kind = \\"satire source\\" if host in SATIRE_HOSTS else \\"search results\\" if is_search_page(item.get(\\"url\\")) else \\"unspecified\\"\\n    return f\\"Source domain: {host}. Source type: {kind}.\\\\nPassage: {item.get(\'text\') or \'\'}\\"\\n\\n\\ndef clean_candidates(claim: str, claim_date: str | None, candidates: list[dict], policy: EvidencePolicy) -> tuple[list[dict], list[dict]]:\\n    date = parse_claim_date(claim_date)\\n    selected, rejected, signatures, seen_ids = [], [], [], set()\\n    if policy.max_candidates < 1 or policy.candidate_depth < 1 or not 0 < policy.near_duplicate_threshold <= 1:\\n        raise ValueError(\\"Invalid evidence policy\\")\\n    for rank, source in enumerate(candidates[:policy.candidate_depth]):\\n        pid = str(source.get(\\"passage_id\\") or \\"\\")\\n        if not pid or pid in seen_ids:\\n            raise ValueError(f\\"Missing/duplicate candidate passage ID: {pid!r}\\")\\n        seen_ids.add(pid)\\n        text = str(source.get(\\"text\\") or \\"\\").strip()\\n        reason = \\"blank\\" if not text else None\\n        if policy.remove_search_pages and is_search_page(source.get(\\"url\\")):\\n            reason = \\"search_results_page\\"\\n        published = publication_date(source)\\n        if policy.remove_known_future_publications and date and published and published > date:\\n            reason = \\"published_after_claim\\"\\n        signature = shingles(text)\\n        if reason is None:\\n            for old_pid, old_signature in signatures:\\n                ratio = len(signature & old_signature) / max(1, len(signature | old_signature))\\n                if ratio >= policy.near_duplicate_threshold:\\n                    reason = f\\"duplicate_of:{old_pid}\\"\\n                    break\\n        if reason:\\n            rejected.append({\\"passage_id\\": pid, \\"reason\\": reason, \\"original_rank\\": rank + 1})\\n            continue\\n        signatures.append((pid, signature))\\n        # Whitelist only fields that exist before a verdict is annotated.\\n        item = {\\"passage_id\\": pid, \\"text\\": text, \\"url\\": source.get(\\"url\\"), \\"original_rank\\": rank + 1,\\n                \\"publication_date\\": published, \\"quality\\": quality_features(claim, source, rank + 1, date)}\\n        selected.append(item)\\n        if len(selected) == policy.max_candidates:\\n            break\\n    return selected, rejected\\n\\n\\ndef nli_label_order(id2label: dict, explicit: str | None = None) -> list[str]:\\n    names = explicit.split(\\",\\") if explicit else [str(id2label.get(i, id2label.get(str(i), \\"\\"))).lower() for i in range(3)]\\n    names = [n.strip().lower() for n in names]\\n    if len(names) != 3 or set(names) != {\\"entailment\\", \\"neutral\\", \\"contradiction\\"}:\\n        raise ValueError(\\"Cannot identify NLI labels. Supply --nli-label-order in the model\'s documented logit order.\\")\\n    return names\\n\\n\\ndef stance_rank(item: dict) -> float:\\n    p = item[\\"nli_probabilities\\"]\\n    q = item[\\"quality\\"]\\n    # Both support and refutation are valuable; neither is a claim verdict.\\n    return 0.60 * (1 - p[\\"neutral\\"]) + 0.25 * q[0] + 0.15 * q[2] - 0.20 * q[3]\\n\\n\\ndef select_evidence(candidates: list[dict], *, rerank: bool) -> list[dict]:\\n    order = sorted(candidates, key=lambda p: (-stance_rank(p), p[\\"original_rank\\"], p[\\"passage_id\\"])) if rerank else list(candidates)\\n    chosen, urls = [], set()\\n    for p in order:\\n        source = canonical_url(original_url(p.get(\\"url\\"))) or p[\\"passage_id\\"]\\n        if rerank and source in urls:\\n            continue\\n        chosen.append(p)\\n        urls.add(source)\\n        if len(chosen) == 3:\\n            return chosen\\n    # Low source count: use additional distinct passages, never deleted garbage.\\n    used = {p[\\"passage_id\\"] for p in chosen}\\n    return (chosen + [p for p in order if p[\\"passage_id\\"] not in used])[:3]\\n\\n\\ndef aggregate_features(row: dict) -> np.ndarray:\\n    \\"\\"\\"Cheap learned-aggregation baseline using the same selected three passages.\\"\\"\\"\\n    evidence = row.get(\\"retrieved\\") or []\\n    p = np.asarray([[item[\\"nli_probabilities\\"][k] for k in (\\"entailment\\", \\"contradiction\\", \\"neutral\\")] for item in evidence[:3]], dtype=float)\\n    q = np.asarray([item[\\"quality\\"] for item in evidence[:3]], dtype=float)\\n    if not len(p):\\n        return np.zeros(3 * 2 + len(QUALITY_NAMES) * 2 + 2, dtype=float)\\n    return np.concatenate([p.mean(0), p.max(0), q.mean(0), q.max(0), [p[:, 0].max() * p[:, 1].max(), len(p) / 3]])\\n", "src/retrieval/metrics.py": "\\"\\"\\"Retrieval metrics for passage-level relevance judgments.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom collections.abc import Iterable, Mapping, Sequence\\n\\n\\ndef reciprocal_rank(retrieved_ids: Sequence[str], relevant_ids: Iterable[str]) -> float:\\n    relevant = set(relevant_ids)\\n    for rank, passage_id in enumerate(retrieved_ids, start=1):\\n        if passage_id in relevant:\\n            return 1.0 / rank\\n    return 0.0\\n\\n\\ndef evidence_hit_at_k(retrieved_ids: Sequence[str], relevant_ids: Iterable[str], k: int) -> float:\\n    \\"\\"\\"Whether at least one judged evidence unit appears in the first *k* results.\\"\\"\\"\\n    relevant = set(relevant_ids)\\n    if not relevant:\\n        raise ValueError(\\"Evidence hit rate is undefined without relevance judgments\\")\\n    return float(bool(set(retrieved_ids[:k]) & relevant))\\n\\n\\ndef passage_recall_at_k(retrieved_ids: Sequence[str], relevant_ids: Iterable[str], k: int) -> float:\\n    \\"\\"\\"Fraction of all judged relevant passages returned in the first *k* results.\\"\\"\\"\\n    relevant = set(relevant_ids)\\n    if not relevant:\\n        raise ValueError(\\"Passage recall is undefined without relevance judgments\\")\\n    return len(set(retrieved_ids[:k]) & relevant) / len(relevant)\\n\\n\\ndef precision_at_k(retrieved_ids: Sequence[str], relevant_ids: Iterable[str], k: int) -> float:\\n    \\"\\"\\"Relevant retrieved items divided by k; missing ranks count as non-relevant.\\"\\"\\"\\n    if k < 1:\\n        raise ValueError(\\"k must be at least one\\")\\n    return len(set(retrieved_ids[:k]) & set(relevant_ids)) / k\\n\\n\\ndef recall_at_k(retrieved_ids: Sequence[str], relevant_ids: Iterable[str], k: int) -> float:\\n    \\"\\"\\"Backward-compatible alias for the project-defined evidence hit metric.\\"\\"\\"\\n    return evidence_hit_at_k(retrieved_ids, relevant_ids, k)\\n\\n\\ndef evaluate_rankings(\\n    rankings: Mapping[str, Sequence[str]], qrels: Mapping[str, Iterable[str]], k: int = 3\\n) -> dict[str, float | int]:\\n    \\"\\"\\"Evaluate only judged claims and report their number explicitly.\\"\\"\\"\\n    judged_ids = sorted(set(rankings) & set(qrels))\\n    if not judged_ids:\\n        raise ValueError(\\"No overlap between rankings and relevance judgments\\")\\n    hit_rates = [evidence_hit_at_k(rankings[claim_id], qrels[claim_id], k) for claim_id in judged_ids]\\n    passage_recalls = [passage_recall_at_k(rankings[claim_id], qrels[claim_id], k) for claim_id in judged_ids]\\n    precisions = [precision_at_k(rankings[claim_id], qrels[claim_id], k) for claim_id in judged_ids]\\n    reciprocal_ranks = [reciprocal_rank(rankings[claim_id], qrels[claim_id]) for claim_id in judged_ids]\\n    return {\\n        f\\"evidence_hit_at_{k}\\": sum(hit_rates) / len(hit_rates),\\n        f\\"passage_recall_at_{k}\\": sum(passage_recalls) / len(passage_recalls),\\n        f\\"precision_at_{k}\\": sum(precisions) / len(precisions),\\n        \\"mrr\\": sum(reciprocal_ranks) / len(reciprocal_ranks),\\n        \\"judged_claim_count\\": len(judged_ids),\\n    }\\n", "src/retrieval/question_guided.py": "\\"\\"\\"Pure, testable parts of question-guided evidence selection.\\n\\nNo labels, gold answers, or qrels are inputs to these functions.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport math\\nimport re\\nfrom typing import Any\\n\\nfrom src.data.corpus import canonical_url\\n\\n\\nTOKEN = re.compile(r\\"[a-z0-9]+\\", re.IGNORECASE)\\nSENTENCE_BOUNDARY = re.compile(r\\"(?<=[.!?])\\\\s+(?=[A-Z0-9\\\\\\"\'([])\\")\\nSTOPWORDS = {\\n    \\"a\\", \\"an\\", \\"and\\", \\"are\\", \\"as\\", \\"at\\", \\"be\\", \\"by\\", \\"did\\", \\"do\\", \\"does\\", \\"for\\", \\"from\\",\\n    \\"has\\", \\"have\\", \\"how\\", \\"in\\", \\"is\\", \\"it\\", \\"of\\", \\"on\\", \\"or\\", \\"that\\", \\"the\\", \\"their\\",\\n    \\"there\\", \\"this\\", \\"to\\", \\"was\\", \\"were\\", \\"what\\", \\"when\\", \\"where\\", \\"which\\", \\"who\\", \\"why\\",\\n    \\"with\\", \\"would\\", \\"could\\", \\"should\\", \\"claim\\", \\"evidence\\", \\"supports\\", \\"contradicts\\",\\n}\\n\\n\\ndef words(text: str) -> set[str]:\\n    return set(TOKEN.findall(text.lower())) - STOPWORDS\\n\\n\\ndef parse_questions(response: str, claim: str, *, limit: int = 2) -> list[str]:\\n    \\"\\"\\"Accept JSON questions only; fall back to a claim-derived question.\\"\\"\\"\\n    found: list[str] = []\\n    decoder = json.JSONDecoder()\\n    for index, char in enumerate(response):\\n        if char != \\"{\\":\\n            continue\\n        try:\\n            value, _ = decoder.raw_decode(response[index:])\\n        except json.JSONDecodeError:\\n            continue\\n        if isinstance(value, dict) and isinstance(value.get(\\"questions\\"), list):\\n            for item in value[\\"questions\\"]:\\n                if isinstance(item, str):\\n                    question = \\" \\".join(item.split())[:240]\\n                    if 8 <= len(question) and question.lower() not in {q.lower() for q in found}:\\n                        found.append(question)\\n                if len(found) == limit:\\n                    break\\n            break\\n    if len(found) < limit:\\n        return [\\n            f\\"What evidence supports or contradicts this claim: {claim[:180]}?\\",\\n            f\\"Which source verifies the date, number, or entity in this claim: {claim[:180]}?\\",\\n        ][:limit]\\n    return found\\n\\n\\ndef shortlist_candidates(row: dict[str, Any], questions: list[str], *, limit: int, bm25_keep: int) -> list[dict[str, Any]]:\\n    \\"\\"\\"Retain BM25 leaders and candidates matching the question terms.\\"\\"\\"\\n    candidates = list(row.get(\\"retrieved\\") or [])\\n    ids = [str(item[\\"passage_id\\"]) for item in candidates]\\n    if len(ids) != len(set(ids)):\\n        raise ValueError(f\\"Duplicate candidate passage IDs for {row.get(\'claim_id\')}\\")\\n    if limit < 1 or bm25_keep < 0:\\n        raise ValueError(\\"limit must be positive and bm25_keep nonnegative\\")\\n    query_terms = words(\\" \\".join([str(row.get(\\"claim\\") or \\"\\"), *questions]))\\n    leader_indices = list(range(min(bm25_keep, limit, len(candidates))))\\n    scored = []\\n    for index, item in enumerate(candidates):\\n        overlap = len(query_terms & words(str(item.get(\\"text\\") or \\"\\")[:1600]))\\n        scored.append((-overlap, index))\\n    chosen = set(leader_indices)\\n    for _, index in sorted(scored):\\n        if len(chosen) >= min(limit, len(candidates)):\\n            break\\n        chosen.add(index)\\n    return [candidates[index] for index in sorted(chosen)]\\n\\n\\ndef reciprocal_rank_scores(score_matrix: list[list[float]], *, claim_weight: float = 1.0) -> list[float]:\\n    \\"\\"\\"Fuse per-query cross-encoder ranks without comparing raw logits across queries.\\"\\"\\"\\n    if not score_matrix:\\n        return []\\n    size = len(score_matrix[0])\\n    if any(len(row) != size or any(not math.isfinite(float(value)) for value in row) for row in score_matrix):\\n        raise ValueError(\\"Cross-encoder score matrix is ragged or non-finite\\")\\n    result = [0.0] * size\\n    for query_index, scores in enumerate(score_matrix):\\n        weight = claim_weight if query_index == 0 else 1.0\\n        for rank, index in enumerate(sorted(range(size), key=lambda i: (-scores[i], i)), start=1):\\n            result[index] += weight / (20 + rank)\\n    return result\\n\\n\\ndef select_diverse(candidates: list[dict[str, Any]], scores: list[float], *, k: int = 3) -> list[dict[str, Any]]:\\n    \\"\\"\\"Select at most k unique-source passages, then fill if source count is low.\\"\\"\\"\\n    if len(candidates) != len(scores):\\n        raise ValueError(\\"Candidate and score counts differ\\")\\n    order = sorted(range(len(candidates)), key=lambda i: (-scores[i], str(candidates[i][\\"passage_id\\"])))\\n    selected: list[int] = []\\n    seen_sources: set[str] = set()\\n    for index in order:\\n        item = candidates[index]\\n        url = canonical_url(item.get(\\"url\\"))\\n        source = f\\"url:{url}\\" if url else f\\"passage:{item[\'passage_id\']}\\"\\n        if source not in seen_sources:\\n            selected.append(index)\\n            seen_sources.add(source)\\n        if len(selected) == k:\\n            break\\n    if len(selected) < k:\\n        selected.extend(index for index in order if index not in selected)\\n    return [{**candidates[index], \\"score\\": float(scores[index])} for index in selected[:k]]\\n\\n\\ndef extract_sentences(text: str, claim: str, questions: list[str], *, max_chars: int = 900, max_sentences: int = 3) -> str:\\n    \\"\\"\\"Select exact source sentences by term coverage; never generate evidence text.\\"\\"\\"\\n    original = str(text).strip()\\n    if len(original) <= max_chars:\\n        return original\\n    sentences = [item.strip() for item in SENTENCE_BOUNDARY.split(original) if item.strip()]\\n    if len(sentences) < 2:\\n        return original[:max_chars].rsplit(\\" \\", 1)[0] or original[:max_chars]\\n    query_sets = [words(claim), *(words(q) for q in questions)]\\n    scored = []\\n    for index, sentence in enumerate(sentences):\\n        tokens = words(sentence)\\n        coverage = sum(len(tokens & query) / max(1, len(query)) for query in query_sets)\\n        scored.append((-coverage, index))\\n    chosen: list[int] = []\\n    used = 0\\n    for _, index in sorted(scored):\\n        sentence = sentences[index]\\n        if used + len(sentence) + (1 if chosen else 0) > max_chars:\\n            continue\\n        chosen.append(index)\\n        used += len(sentence) + (1 if len(chosen) > 1 else 0)\\n        if len(chosen) == max_sentences:\\n            break\\n    if not chosen:\\n        return original[:max_chars].rsplit(\\" \\", 1)[0] or original[:max_chars]\\n    return \\" \\".join(sentences[index] for index in sorted(chosen))\\n", "src/utils/__init__.py": "\\"\\"\\"Small shared utilities for reproducible experiment scripts.\\"\\"\\"\\n", "src/utils/checkpoints.py": "\\"\\"\\"Small atomic artifacts and interruption-safe append-only JSONL journals.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\n\\nfrom src.utils.io import write_json, write_jsonl\\n\\n\\ndef atomic_jsonl(rows, path: Path) -> None:\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    temporary = path.with_name(path.name + \\".tmp\\")\\n    write_jsonl(rows, temporary)\\n    temporary.replace(path)\\n\\n\\ndef bind_run(path: Path, config: dict) -> None:\\n    if path.exists() and json.loads(path.read_text(encoding=\\"utf-8\\")) != config:\\n        raise ValueError(f\\"Inputs or settings changed; use a new run directory: {path.parent}\\")\\n    write_json(config, path)\\n\\n\\ndef read_journal(path: Path, expected_ids: set[str]) -> dict[str, dict]:\\n    if not path.exists():\\n        return {}\\n    found = {}\\n    data = path.read_bytes()\\n    lines = data.splitlines(keepends=True)\\n    last = max((i for i, line in enumerate(lines) if line.strip()), default=-1)\\n    offset = 0\\n    repaired = False\\n    for i, raw in enumerate(lines):\\n        start = offset\\n        offset += len(raw)\\n        if not raw.strip():\\n            continue\\n        try:\\n            row = json.loads(raw.decode(\\"utf-8\\"))\\n        except (UnicodeDecodeError, json.JSONDecodeError) as error:\\n            if i != last:\\n                raise ValueError(f\\"Corrupt checkpoint before final line: {path}\\") from error\\n            with path.open(\\"r+b\\") as handle:\\n                handle.truncate(start)\\n            repaired = True\\n            break\\n        cid = str(row.get(\\"claim_id\\") or \\"\\")\\n        if cid not in expected_ids or cid in found:\\n            raise ValueError(f\\"Unexpected or duplicate journal claim ID: {cid!r}\\")\\n        found[cid] = row\\n    if not repaired and data and not data.endswith(b\\"\\\\n\\"):\\n        with path.open(\\"ab\\") as handle:\\n            handle.write(b\\"\\\\n\\")\\n    return found\\n\\n\\ndef append_journal(path: Path, rows: list[dict]) -> None:\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    with path.open(\\"a\\", encoding=\\"utf-8\\", newline=\\"\\\\n\\") as handle:\\n        for row in rows:\\n            handle.write(json.dumps(row, ensure_ascii=False) + \\"\\\\n\\")\\n        handle.flush()\\n\\n\\ndef latest_complete_trainer_checkpoint(directory: Path) -> str | None:\\n    candidates = []\\n    required = (\\"trainer_state.json\\", \\"optimizer.pt\\", \\"scheduler.pt\\", \\"rng_state.pth\\")\\n    for path in directory.glob(\\"checkpoint-*\\"):\\n        try:\\n            step = int(path.name.removeprefix(\\"checkpoint-\\"))\\n        except ValueError:\\n            continue\\n        if (any(path.glob(\\"model*.safetensors\\")) or any(path.glob(\\"pytorch_model*.bin\\"))) and all((path / k).is_file() for k in required):\\n            candidates.append((step, path))\\n    return str(max(candidates, key=lambda p: p[0])[1]) if candidates else None\\n", "src/utils/io.py": "\\"\\"\\"Minimal JSONL helpers used by scripts and tests.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\nfrom typing import Any, Iterable\\n\\n\\ndef read_jsonl(path: str | Path) -> list[dict[str, Any]]:\\n    return list(iter_jsonl(path))\\n\\n\\ndef iter_jsonl(path: str | Path) -> Iterable[dict[str, Any]]:\\n    \\"\\"\\"Read JSONL lazily for large Colab corpus files.\\"\\"\\"\\n    source = Path(path)\\n    with source.open(\\"r\\", encoding=\\"utf-8\\") as handle:\\n        for line in handle:\\n            if line.strip():\\n                yield json.loads(line)\\n\\n\\ndef write_jsonl(rows: Iterable[dict[str, Any]], path: str | Path) -> None:\\n    destination = Path(path)\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    with destination.open(\\"w\\", encoding=\\"utf-8\\", newline=\\"\\\\n\\") as handle:\\n        for row in rows:\\n            handle.write(json.dumps(row, ensure_ascii=False) + \\"\\\\n\\")\\n\\n\\ndef write_json(data: dict[str, Any], path: str | Path) -> None:\\n    \\"\\"\\"Atomically replace a small JSON artifact after a complete write.\\"\\"\\"\\n    destination = Path(path)\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    temporary = destination.with_name(f\\"{destination.name}.tmp\\")\\n    temporary.write_text(json.dumps(data, indent=2, ensure_ascii=False) + \\"\\\\n\\", encoding=\\"utf-8\\")\\n    temporary.replace(destination)\\n", "src/utils/notebooks.py": "\\"\\"\\"Minimal reproducible Colab artifact helpers (standard library only).\\"\\"\\"\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\n\\n\\nclass NotebookBuilder:\\n    def __init__(self, name):\\n        self.name, self.cells = name, []\\n\\n    def markdown(self, text):\\n        self.cells.append({\\"cell_type\\": \\"markdown\\", \\"metadata\\": {}, \\"source\\": text.strip().splitlines(keepends=True)})\\n\\n    def code(self, text, *, hidden=False):\\n        self.cells.append({\\"cell_type\\": \\"code\\", \\"execution_count\\": None, \\"outputs\\": [],\\n                           \\"metadata\\": {\\"jupyter\\": {\\"source_hidden\\": True}} if hidden else {},\\n                           \\"source\\": (text.strip() + \\"\\\\n\\").splitlines(keepends=True)})\\n\\n    def document(self, **metadata):\\n        for i, cell in enumerate(self.cells):\\n            cell[\\"id\\"] = f\\"evidence-reader-{i:03d}\\"\\n        return {\\"cells\\": self.cells, \\"metadata\\": {\\"colab\\": {\\"name\\": self.name}, \\"accelerator\\": \\"GPU\\",\\n                \\"kernelspec\\": {\\"name\\": \\"python3\\", \\"display_name\\": \\"Python 3\\", \\"language\\": \\"python\\"},\\n                \\"language_info\\": {\\"name\\": \\"python\\"}, **metadata}, \\"nbformat\\": 4, \\"nbformat_minor\\": 5}\\n\\n\\ndef source_snapshot(root, extra_files):\\n    root = Path(root)\\n    paths = sorted((root / \\"src\\").rglob(\\"*.py\\")) + [root / p for p in extra_files]\\n    sources = {p.relative_to(root).as_posix(): p.read_text(encoding=\\"utf-8\\") for p in paths}\\n    sources[\\"scripts/__init__.py\\"] = \\"\\"\\n    payload = json.dumps(sources, sort_keys=True, ensure_ascii=True)\\n    return payload, hashlib.sha256(payload.encode()).hexdigest()\\n\\n\\ndef emit_notebook(document, target, *, patch=False):\\n    text = json.dumps(document, ensure_ascii=True, indent=1) + \\"\\\\n\\"\\n    target = Path(target)\\n    if not patch:\\n        print(text, end=\\"\\")\\n        return\\n    print(\\"*** Begin Patch\\")\\n    if target.exists():\\n        print(\\"*** Update File: \\" + target.as_posix())\\n        print(\\"@@\\")\\n        print(\\"\\\\n\\".join(\\"-\\" + line for line in target.read_text(encoding=\\"utf-8\\").splitlines()))\\n    else:\\n        print(\\"*** Add File: \\" + target.as_posix())\\n    print(\\"\\\\n\\".join(\\"+\\" + line for line in text.splitlines()))\\n    print(\\"*** End Patch\\")\\n", "src/utils/progress.py": "\\"\\"\\"Optional notebook-friendly progress bars with a dependency-free fallback.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any\\n\\n\\nclass _NoOpProgress:\\n    def update(self, _: int | float = 1) -> None:\\n        return None\\n\\n    def set_postfix(self, **_: Any) -> None:\\n        return None\\n\\n    def __enter__(self) -> \\"_NoOpProgress\\":\\n        return self\\n\\n    def __exit__(self, *_: Any) -> None:\\n        return None\\n\\n\\ndef progress(*, total: int | None, description: str, unit: str = \\"item\\", **kwargs: Any) -> Any:\\n    \\"\\"\\"Return `tqdm.auto` when installed; scripts remain usable without it.\\"\\"\\"\\n    try:\\n        from tqdm.auto import tqdm\\n    except ImportError:\\n        return _NoOpProgress()\\n    return tqdm(total=total, desc=description, unit=unit, **kwargs)\\n", "src/utils/runs.py": "\\"\\"\\"Run manifests that make every reported experiment traceable.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport platform\\nimport re\\nimport subprocess\\nimport sys\\nfrom datetime import UTC, datetime\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nfrom src.utils.io import write_json\\n\\n\\n_RUN_NAME_PATTERN = re.compile(r\\"[A-Za-z0-9][A-Za-z0-9_.-]*$\\")\\n\\n\\ndef sha256_file(path: str | Path) -> str:\\n    digest = hashlib.sha256()\\n    with Path(path).open(\\"rb\\") as handle:\\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\\"\\"):\\n            digest.update(chunk)\\n    return digest.hexdigest()\\n\\n\\ndef git_commit(project_root: Path) -> str | None:\\n    try:\\n        return subprocess.check_output(\\n            [\\"git\\", \\"rev-parse\\", \\"HEAD\\"], cwd=project_root, text=True, stderr=subprocess.DEVNULL\\n        ).strip()\\n    except (OSError, subprocess.CalledProcessError):\\n        return None\\n\\n\\ndef resolve_run_directory(base_directory: str | Path, run_name: str | None) -> Path:\\n    \\"\\"\\"Return a stable output directory and reject path-like run identifiers.\\"\\"\\"\\n    base = Path(base_directory)\\n    if not run_name:\\n        return base\\n    if not _RUN_NAME_PATTERN.fullmatch(run_name):\\n        raise ValueError(\\"run_name must use only letters, numbers, dots, underscores, or hyphens\\")\\n    return base / run_name\\n\\n\\ndef write_run_manifest(\\n    output_path: str | Path,\\n    *,\\n    project_root: Path,\\n    config: dict[str, Any],\\n    input_paths: dict[str, str | Path],\\n) -> dict[str, Any]:\\n    \\"\\"\\"Write metadata without recording credentials or full dataset contents.\\"\\"\\"\\n    inputs: dict[str, dict[str, Any]] = {}\\n    for name, value in input_paths.items():\\n        path = Path(value)\\n        inputs[name] = {\\n            \\"path\\": str(path),\\n            \\"exists\\": path.exists(),\\n            \\"sha256\\": sha256_file(path) if path.is_file() else None,\\n        }\\n    manifest = {\\n        \\"created_at_utc\\": datetime.now(UTC).isoformat(),\\n        \\"git_commit\\": git_commit(project_root),\\n        \\"python_version\\": sys.version,\\n        \\"platform\\": platform.platform(),\\n        \\"config\\": config,\\n        \\"inputs\\": inputs,\\n    }\\n    target = Path(output_path)\\n    target.parent.mkdir(parents=True, exist_ok=True)\\n    write_json(manifest, target)\\n    return manifest\\n", "src/verification/__init__.py": "\\"\\"\\"Claim-verification baselines and later learned verifiers.\\"\\"\\"\\n", "src/verification/evidence_experiments.py": "\\"\\"\\"Fast aggregation, frozen-baseline controls and common result reporting.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\n\\nimport joblib\\nimport numpy as np\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.pipeline import make_pipeline\\nfrom sklearn.preprocessing import StandardScaler\\n\\nfrom src.data.averitec import CANONICAL_LABELS\\nfrom src.data.evidence import attach_evidence\\nfrom src.evaluation.classification import evaluate_predictions\\nfrom src.evaluation.paired import paired_delta\\nfrom src.evaluation.splits import selection_split\\nfrom src.retrieval.evidence_quality import aggregate_features, normalized_text\\nfrom src.retrieval.metrics import evaluate_rankings\\nfrom src.utils.checkpoints import atomic_jsonl, bind_run\\nfrom src.utils.io import read_jsonl, write_json\\nfrom src.utils.runs import sha256_file\\nfrom src.verification.inference import predict_cached, validate_predictions\\nfrom src.verification.pair_training import selection_key\\n\\n\\ndef fast_aggregate(args):\\n    claims = {s: read_jsonl(args.data_root / \\"processed/averitec\\" / f\\"{s}.jsonl\\") for s in (\\"train\\", \\"dev\\")}\\n    rankings = {s: args.prepared / args.mode / f\\"{s}_rankings.jsonl\\" for s in claims}\\n    rows = {s: attach_evidence(claims[s], rankings[s], exact=True) for s in claims}\\n    if json.loads((args.prepared / \\"preparation.json\\").read_text())[\\"limit\\"]:\\n        raise ValueError(\\"Fast training requires full preparation\\")\\n    x = {s: np.stack([aggregate_features({\\"retrieved\\": r[\\"retrieved_evidence\\"]}) for r in rows[s]]) for s in rows}\\n    y = np.array([r[\\"label\\"] for r in rows[\\"train\\"]])\\n    a, b = selection_split(rows[\\"train\\"], .1, args.validation_seed, args.validation_policy)\\n    output = args.output\\n    output.mkdir(parents=True, exist_ok=True)\\n    bind_run(output / \\"fast_run.json\\", {\\"mode\\": args.mode, \\"rankings_sha256\\": {s: sha256_file(p) for s, p in rankings.items()},\\n        \\"claims_sha256\\": {s: sha256_file(args.data_root / \\"processed/averitec\\" / f\\"{s}.jsonl\\") for s in claims},\\n        \\"validation_seed\\": args.validation_seed, \\"validation_policy\\": args.validation_policy,\\n        \\"regularization\\": [.1, 1, 10], \\"implementation_sha256\\": sha256_file(Path(__file__))})\\n\\n    def model(c):\\n        return make_pipeline(StandardScaler(), LogisticRegression(C=c, class_weight=\\"balanced\\", max_iter=2000, random_state=args.validation_seed))\\n\\n    def predict(classifier, features, records):\\n        probabilities = classifier.predict_proba(features)\\n        indices = [list(classifier.classes_).index(k) for k in CANONICAL_LABELS]\\n        return [{\\"claim_id\\": r[\\"claim_id\\"], \\"true_label\\": r[\\"label\\"], \\"predicted_label\\": CANONICAL_LABELS[int(p[indices].argmax())],\\n                 \\"probabilities\\": dict(zip(CANONICAL_LABELS, map(float, p[indices]), strict=True))}\\n                for r, p in zip(records, probabilities, strict=True)]\\n\\n    trials = []\\n    for c in (.1, 1, 10):\\n        classifier = model(c).fit(x[\\"train\\"][a], y[a])\\n        p = predict(classifier, x[\\"train\\"][b], [rows[\\"train\\"][int(i)] for i in b])\\n        trials.append({**evaluate_predictions(p), \\"C\\": c})\\n    chosen = sorted(trials, key=selection_key)[0]\\n    write_json({\\"selected\\": chosen, \\"trials\\": trials, \\"validation_claim_ids\\": [rows[\\"train\\"][int(i)][\\"claim_id\\"] for i in b]}, output / \\"model_selection.json\\")\\n    if args.selection_only:\\n        print(\\"Fast train-only macro-F1:\\", chosen[\\"macro_f1\\"])\\n        return\\n    classifier = model(chosen[\\"C\\"]).fit(x[\\"train\\"], y)\\n    joblib.dump(classifier, output / \\"aggregator.joblib\\")\\n    predictions = predict(classifier, x[\\"dev\\"], rows[\\"dev\\"])\\n    atomic_jsonl(predictions, output / \\"dev/predictions.jsonl\\")\\n    write_json(evaluate_predictions(predictions), output / \\"dev/metrics.json\\")\\n    print(\\"Fast dev macro-F1:\\", evaluate_predictions(predictions)[\\"macro_f1\\"])\\n\\n\\ndef frozen_distil(args):\\n    import torch\\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer\\n    from scripts.train_transformer_verifier import compose_text\\n\\n    if args.device.startswith(\\"cuda\\") and not torch.cuda.is_available():\\n        raise RuntimeError(\\"Select a GPU runtime\\")\\n    path = args.prepared / args.mode / \\"dev_rankings.jsonl\\"\\n    records = attach_evidence(read_jsonl(args.data_root / \\"processed/averitec/dev.jsonl\\"), path, exact=True)\\n    config_path, model_dir = args.baseline / \\"transformer_run_config.json\\", args.baseline / \\"final_model\\"\\n    settings = json.loads(config_path.read_text(encoding=\\"utf-8\\"))\\n    bind_run(args.output / \\"frozen_run.json\\", {\\"rankings\\": sha256_file(path), \\"claims\\": sha256_file(args.data_root / \\"processed/averitec/dev.jsonl\\"),\\n        \\"settings\\": settings, \\"weights\\": {p.name: sha256_file(p) for p in [*model_dir.glob(\\"*.safetensors\\"), *model_dir.glob(\\"pytorch_model*.bin\\")]}})\\n    tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True, use_fast=True)\\n    model = AutoModelForSequenceClassification.from_pretrained(model_dir, local_files_only=True).to(args.device).eval()\\n    if [model.config.id2label.get(i) for i in range(4)] != list(CANONICAL_LABELS):\\n        raise ValueError(\\"Baseline label order differs\\")\\n\\n    def batch(rows):\\n        text = [compose_text(r, tokenizer, settings[\\"claim_token_budget\\"], settings[\\"evidence_token_budget_per_passage\\"]) for r in rows]\\n        encoded = tokenizer(text, truncation=True, padding=True, max_length=settings[\\"max_length\\"], return_tensors=\\"pt\\")\\n        with torch.inference_mode():\\n            return model(**{k: v.to(args.device) for k, v in encoded.items()}).logits.float().softmax(-1).cpu().tolist()\\n\\n    print(json.dumps(predict_cached(records, batch, args.output, batch_size=16, description=f\\"Frozen DistilBERT / {args.mode}\\"), indent=2))\\n\\n\\ndef compare_results(args):\\n    claims = read_jsonl(args.data_root / \\"processed/averitec/dev.jsonl\\")\\n    reports, predictions = {}, {}\\n    for path in sorted(args.output.glob(\\"**/metrics.json\\")):\\n        pred_path = path.with_name(\\"predictions.jsonl\\")\\n        if not pred_path.is_file():\\n            continue\\n        name = str(path.parent.relative_to(args.output)).replace(\\"\\\\\\\\\\", \\"/\\").removesuffix(\\"/dev\\")\\n        predictions[name] = validate_predictions(read_jsonl(pred_path), claims)\\n        reports[name] = evaluate_predictions(predictions[name])\\n    for path in sorted(args.output.glob(\\"distil_*/final_dev_metrics.json\\")):\\n        name = path.parent.name\\n        predictions[name] = validate_predictions(read_jsonl(path.with_name(\\"final_dev_predictions.jsonl\\")), claims)\\n        reports[name] = evaluate_predictions(predictions[name])\\n    reference_root = getattr(args, \\"reference_root\\", None)\\n    if reference_root is not None:\\n        # Read completed old predictions without refitting or rewriting old runs.\\n        for path in sorted(reference_root.glob(\\"**/metrics.json\\")):\\n            pred_path = path.with_name(\\"predictions.jsonl\\")\\n            if not pred_path.is_file():\\n                continue\\n            name = \\"previous/\\" + path.parent.relative_to(reference_root).as_posix().removesuffix(\\"/dev\\")\\n            predictions[name] = validate_predictions(read_jsonl(pred_path), claims)\\n            reports[name] = evaluate_predictions(predictions[name])\\n        for path in sorted(reference_root.glob(\\"distil_*/final_dev_predictions.jsonl\\")):\\n            name = \\"previous/\\" + path.parent.name\\n            predictions[name] = validate_predictions(read_jsonl(path), claims)\\n            reports[name] = evaluate_predictions(predictions[name])\\n    reference_name = \\"frozen/raw\\" if \\"frozen/raw\\" in predictions else \\"previous/frozen/raw\\"\\n    reference = predictions.get(reference_name)\\n    paired = {name: paired_delta(reference, rows) for name, rows in predictions.items() if reference and name != reference_name}\\n    contrasts = {}\\n    for old, new in ((\\"pair/raw_legacy_d01\\", \\"pair/raw_sample_d01\\"),\\n                     (\\"pair/raw_sample_d01\\", \\"pair/raw_sample_d0\\"),\\n                     (\\"pair/raw_sample_d0\\", \\"pair/qa_same3_d0\\"),\\n                     (\\"pair/qa_same3_d0\\", \\"pair/qa_selected3_d0\\")):\\n        if old in predictions and new in predictions:\\n            contrasts[new + \\" minus \\" + old] = paired_delta(predictions[old], predictions[new])\\n    normalized_train = {normalized_text(r[\\"claim\\"]) for r in read_jsonl(args.data_root / \\"processed/averitec/train.jsonl\\")}\\n    keep = {r[\\"claim_id\\"] for r in claims if normalized_text(r[\\"claim\\"]) not in normalized_train}\\n    nonoverlap = {name: evaluate_predictions([r for r in rows if r[\\"claim_id\\"] in keep]) for name, rows in predictions.items()} if keep else {}\\n    retrieval = {}\\n    if args.qrels:\\n        qrels = {r[\\"claim_id\\"]: r[\\"relevant_passage_ids\\"] for r in read_jsonl(args.qrels) if r.get(\\"relevant_passage_ids\\")}\\n        for path in sorted(args.prepared.glob(\\"*/dev_rankings.jsonl\\")):\\n            mode = path.parent.name\\n            rows = read_jsonl(path)\\n            retrieval[mode] = evaluate_rankings({r[\\"claim_id\\"]: [p[\\"passage_id\\"] for p in r[\\"retrieved\\"]] for r in rows}, qrels, k=3)\\n    counts = {name: {label: sum(r[\\"predicted_label\\"] == label for r in rows) for label in CANONICAL_LABELS}\\n              for name, rows in predictions.items()}\\n    report = {\\"verdicts\\": reports, \\"predicted_label_counts\\": counts, \\"paired_vs_frozen_raw\\": paired,\\n              \\"reference_name\\": reference_name, \\"planned_paired_contrasts\\": contrasts, \\"nonoverlapping_text_subset\\": nonoverlap,\\n              \\"subset_claim_count\\": len(keep), \\"retrieval_fixed_original_url_proxy\\": retrieval,\\n              \\"caveats\\": [\\"Official dev is exploratory after repeated inspection; test untouched.\\",\\n                          \\"NLI probabilities are not calibrated four-way verdict probabilities.\\",\\n                          \\"Original qrels and all claims are retained even when a source is filtered.\\",\\n                          \\"Unknown publication dates do not establish temporal validity.\\"]}\\n    write_json(report, args.output / \\"evidence_reader_results.json\\")\\n    lines = [\\"# Evidence reader experiment\\", \\"\\", f\\"Same {len(claims)} dev claims; FR/FA are counts.\\", \\"\\",\\n             \\"| Method | Macro-F1 | Accuracy | FR | FA |\\", \\"| --- | ---: | ---: | ---: | ---: |\\"]\\n    for name, m in reports.items():\\n        errors = m[\\"decision_error_rates\\"]\\n        lines.append(f\\"| {name} | {m[\'macro_f1\']:.4f} | {m[\'accuracy\']:.4f} | {errors[\'false_reassurance_count\']} | {errors[\'false_alarm_count\']} |\\")\\n    lines += [\\"\\", \\"| Method | Supported F1 | Refuted F1 | NEI F1 | Conflict F1 | Predicted NEI / conflict |\\",\\n              \\"| --- | ---: | ---: | ---: | ---: | ---: |\\"]\\n    for name, m in reports.items():\\n        f1 = [m[\\"per_class\\"][label][\\"f1-score\\"] for label in CANONICAL_LABELS]\\n        rare = f\\"{counts[name][\'Not Enough Evidence\']} / {counts[name][\'Conflicting Evidence\']}\\"\\n        lines.append(f\\"| {name} | \\" + \\" | \\".join(f\\"{v:.4f}\\" for v in f1) + f\\" | {rare} |\\")\\n    if contrasts:\\n        lines += [\\"\\", \\"Planned paired comparisons (macro-F1 delta and 95% interval):\\"]\\n        for name, contrast in contrasts.items():\\n            low, high = contrast[\\"interval_95pct\\"]\\n            lines.append(f\\"- {name}: {contrast[\'delta_macro_f1\']:+.4f} [{low:+.4f}, {high:+.4f}]\\")\\n    lines += [\\"\\", \\"Full per-class, calibration, overlap-subset and paired interval results: evidence_reader_results.json.\\",\\n              \\"\\", \\"Frozen rows use the original BM25-trained DistilBERT. Pair rows train the new NLI-initialized reader. Fast rows train a classifier on frozen NLI features.\\"]\\n    (args.output / \\"evidence_reader_results.md\\").write_text(\\"\\\\n\\".join(lines) + \\"\\\\n\\", encoding=\\"utf-8\\")\\n    print(\\"\\\\n\\".join(lines))\\n", "src/verification/inference.py": "\\"\\"\\"Shared resumable verdict inference; labels are used only for reporting.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport math\\nfrom pathlib import Path\\n\\nfrom src.data.averitec import CANONICAL_LABELS\\nfrom src.data.evidence import index_claims\\nfrom src.evaluation.classification import evaluate_predictions\\nfrom src.utils.checkpoints import append_journal, atomic_jsonl, read_journal\\nfrom src.utils.io import read_jsonl, write_json\\nfrom src.utils.progress import progress\\n\\n\\ndef validate_predictions(rows, records):\\n    found = index_claims(rows, expected={r[\\"claim_id\\"] for r in records})\\n    for r in records:\\n        p = found[r[\\"claim_id\\"]]\\n        values = [float(p.get(\\"probabilities\\", {}).get(k, float(\\"nan\\"))) for k in CANONICAL_LABELS]\\n        if p[\\"true_label\\"] != r[\\"label\\"] or p[\\"predicted_label\\"] not in CANONICAL_LABELS:\\n            raise ValueError(\\"Prediction labels differ from the evaluated claims\\")\\n        if any(not math.isfinite(v) or not 0 <= v <= 1 for v in values) or abs(sum(values) - 1) > 1e-4:\\n            raise ValueError(\\"Invalid class probability vector\\")\\n    return [found[r[\\"claim_id\\"]] for r in records]\\n\\n\\ndef predict_cached(records, predict_batch, output: Path, *, batch_size=8, description=\\"Predict verdicts\\"):\\n    output.mkdir(parents=True, exist_ok=True)\\n    final, partial = output / \\"predictions.jsonl\\", output / \\"predictions.partial.jsonl\\"\\n    if final.exists():\\n        predictions = validate_predictions(read_jsonl(final), records)\\n    else:\\n        done = read_journal(partial, {r[\\"claim_id\\"] for r in records})\\n        if done:\\n            subset = [r for r in records if r[\\"claim_id\\"] in done]\\n            validate_predictions(list(done.values()), subset)\\n        pending = [r for r in records if r[\\"claim_id\\"] not in done]\\n        with progress(total=len(records), initial=len(done), description=description, unit=\\"claim\\") as bar:\\n            for start in range(0, len(pending), batch_size):\\n                batch = pending[start:start + batch_size]\\n                probabilities = predict_batch(batch)\\n                new = []\\n                for r, values in zip(batch, probabilities, strict=True):\\n                    row = {\\"claim_id\\": r[\\"claim_id\\"], \\"true_label\\": r[\\"label\\"],\\n                           \\"predicted_label\\": CANONICAL_LABELS[max(range(4), key=lambda i: values[i])],\\n                           \\"probabilities\\": dict(zip(CANONICAL_LABELS, map(float, values), strict=True)),\\n                           \\"evidence_passage_ids\\": [p[\\"passage_id\\"] for p in (r.get(\\"retrieved_evidence\\") or [])[:3]]}\\n                    new.append(row)\\n                validate_predictions(new, batch)\\n                append_journal(partial, new)\\n                done.update({r[\\"claim_id\\"]: r for r in new})\\n                bar.update(len(new))\\n        predictions = [done[r[\\"claim_id\\"]] for r in records]\\n        atomic_jsonl(predictions, final)\\n    metrics = evaluate_predictions(predictions)\\n    write_json(metrics, output / \\"metrics.json\\")\\n    return metrics\\n", "src/verification/llm_zero_shot.py": "\\"\\"\\"Evidence-only zero-shot LLM baseline with strict structured outputs.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nfrom typing import Any, Literal\\n\\nfrom pydantic import BaseModel, Field\\n\\nfrom src.data.averitec import CANONICAL_LABELS\\n\\nLLMVariant = Literal[\\"classify_top3\\", \\"rerank_top20_and_classify\\"]\\n\\n\\nclass LLMVerdict(BaseModel):\\n    \\"\\"\\"Required model response; confidence is self-reported, not calibrated.\\"\\"\\"\\n\\n    label: Literal[\\"Supported\\", \\"Refuted\\", \\"Not Enough Evidence\\", \\"Conflicting Evidence\\"]\\n    selected_passage_ids: list[str] = Field(min_length=1, max_length=3)\\n    reported_confidence: float = Field(ge=0.0, le=1.0)\\n    brief_rationale: str = Field(min_length=1, max_length=600)\\n\\n\\nclass ClaimOnlyLLMVerdict(BaseModel):\\n    \\"\\"\\"Required response when the model receives no evidence passages.\\"\\"\\"\\n\\n    label: Literal[\\"Supported\\", \\"Refuted\\", \\"Not Enough Evidence\\", \\"Conflicting Evidence\\"]\\n    reported_confidence: float = Field(ge=0.0, le=1.0)\\n    brief_rationale: str = Field(min_length=1, max_length=600)\\n\\n\\nSYSTEM_PROMPT = \\"\\"\\"You are an evidence-grounded claim-verification baseline.\\nUse only the supplied passages. Do not use outside knowledge, the claim\'s likely\\ntruth, or information that is absent from the passages. Return one label:\\n- Supported: evidence establishes the claim.\\n- Refuted: evidence establishes that the claim is false or materially wrong.\\n- Not Enough Evidence: supplied evidence does not establish either conclusion.\\n- Conflicting Evidence: supplied evidence contains material support and refutation.\\n\\nSelect the passage IDs that most directly support your decision. Your reported\\nconfidence is a self-assessment, not a probability. Return the required JSON\\nobject only, with exactly these keys:\\n{\\"label\\": \\"one of the four labels\\", \\"selected_passage_ids\\": [\\"passage ID\\"],\\n \\"reported_confidence\\": 0.0, \\"brief_rationale\\": \\"one brief explanation\\"}.\\"\\"\\"\\n\\n\\nCLAIM_ONLY_SYSTEM_PROMPT = \\"\\"\\"You are a claim-only claim-verification baseline.\\nYou receive a claim but no evidence passages. Do not imply that you retrieved,\\nread, or verified external evidence. Return one label:\\n- Supported\\n- Refuted\\n- Not Enough Evidence\\n- Conflicting Evidence\\n\\nReturn the required JSON object only, with exactly these keys:\\n{\\"label\\": \\"one of the four labels\\", \\"reported_confidence\\": 0.0,\\n \\"brief_rationale\\": \\"one brief explanation\\"}.\\"\\"\\"\\n\\n\\ndef build_user_prompt(\\n    claim: str,\\n    passages: list[dict[str, Any]],\\n    variant: LLMVariant,\\n    max_chars_per_passage: int = 1200,\\n    evidence_description: str = \\"retrieved passages\\",\\n) -> str:\\n    \\"\\"\\"Build the frozen evidence-only request for one claim.\\"\\"\\"\\n    expected = min(3, len(passages))\\n    if expected < 1:\\n        raise ValueError(\\"LLM baseline requires at least one candidate passage\\")\\n    instructions = (\\n        f\\"Rerank the {len(passages)} candidate passages and select exactly {expected} passage IDs before classifying.\\"\\n        if variant == \\"rerank_top20_and_classify\\"\\n        else f\\"Classify using these {len(passages)} {evidence_description} and select 1 to {expected} relevant IDs.\\"\\n    )\\n    rendered_passages = []\\n    for index, passage in enumerate(passages, start=1):\\n        passage_id = str(passage[\\"passage_id\\"])\\n        text = str(passage.get(\\"text\\") or \\"\\").strip()[:max_chars_per_passage]\\n        rendered_passages.append(f\\"[PASSAGE {index} | ID={passage_id}]\\\\n{text}\\")\\n    return f\\"{instructions}\\\\n\\\\n[CLAIM]\\\\n{claim.strip()}\\\\n\\\\n\\" + \\"\\\\n\\\\n\\".join(rendered_passages)\\n\\n\\ndef build_claim_only_prompt(claim: str) -> str:\\n    \\"\\"\\"Build the frozen no-evidence request used by the claim-only diagnostic.\\"\\"\\"\\n    return f\\"Classify this claim without evidence passages.\\\\n\\\\n[CLAIM]\\\\n{claim.strip()}\\"\\n\\n\\ndef prompt_sha256(system_prompt: str, user_prompt: str) -> str:\\n    return hashlib.sha256(f\\"{system_prompt}\\\\n\\\\n{user_prompt}\\".encode(\\"utf-8\\")).hexdigest()\\n\\n\\ndef validate_llm_verdict(result: LLMVerdict, passages: list[dict[str, Any]], variant: LLMVariant) -> None:\\n    \\"\\"\\"Verify selections against the exact candidate set after schema parsing.\\"\\"\\"\\n    candidate_ids = {str(passage[\\"passage_id\\"]) for passage in passages}\\n    selected = result.selected_passage_ids\\n    if len(selected) != len(set(selected)):\\n        raise ValueError(\\"LLM selected duplicate passage IDs\\")\\n    if set(selected) - candidate_ids:\\n        raise ValueError(\\"LLM selected a passage ID outside the supplied candidates\\")\\n    if variant == \\"rerank_top20_and_classify\\" and len(selected) != min(3, len(passages)):\\n        raise ValueError(\\"Reranking variant must select exactly three candidates when available\\")\\n\\n\\ndef call_openai_structured(\\n    *, model: str, system_prompt: str, user_prompt: str, temperature: float\\n) -> tuple[LLMVerdict, dict[str, Any]]:\\n    \\"\\"\\"Call OpenAI\'s supported Python structured-output interface.\\n\\n    The OpenAI package is imported only at call time, letting tests run without a\\n    key and preventing accidental API calls during import.\\n    \\"\\"\\"\\n    try:\\n        from openai import OpenAI\\n    except ImportError as error:\\n        raise RuntimeError(\\"Install requirements-colab.txt before using the LLM baseline.\\") from error\\n    completion = OpenAI().chat.completions.parse(\\n        model=model,\\n        messages=[{\\"role\\": \\"system\\", \\"content\\": system_prompt}, {\\"role\\": \\"user\\", \\"content\\": user_prompt}],\\n        response_format=LLMVerdict,\\n        temperature=temperature,\\n    )\\n    message = completion.choices[0].message\\n    if message.refusal:\\n        raise RuntimeError(f\\"Model refusal: {message.refusal}\\")\\n    if message.parsed is None:\\n        raise RuntimeError(\\"Model returned no parsed structured response\\")\\n    return message.parsed, completion.model_dump(mode=\\"json\\")\\n\\n\\ndef build_prediction(claim_id: str, true_label: str | None, result: LLMVerdict | ClaimOnlyLLMVerdict) -> dict[str, Any]:\\n    if result.label not in CANONICAL_LABELS:\\n        raise ValueError(f\\"Invalid LLM label: {result.label}\\")\\n    return {\\n        \\"claim_id\\": claim_id,\\n        \\"true_label\\": true_label,\\n        \\"predicted_label\\": result.label,\\n        \\"probabilities\\": None,\\n        \\"selected_passage_ids\\": getattr(result, \\"selected_passage_ids\\", []),\\n        \\"reported_confidence\\": result.reported_confidence,\\n        \\"brief_rationale\\": result.brief_rationale,\\n    }\\n", "src/verification/pair_reader.py": "\\"\\"\\"NLI-initialized, permutation-invariant three-passage verdict reader.\\n\\nFour-way verdict supervision applies to the aggregate, never to every passage.\\nA small distillation term retains the pretrained three-way NLI behavior using\\ncached teacher probabilities computed from prediction-time text alone.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport torch\\nfrom torch import nn\\nfrom transformers import AutoConfig, AutoModelForSequenceClassification, PretrainedConfig, PreTrainedModel\\nfrom transformers.modeling_outputs import SequenceClassifierOutput\\n\\nfrom src.data.averitec import CANONICAL_LABELS\\nfrom src.retrieval.evidence_quality import QUALITY_NAMES, nli_label_order, pair_premise\\n\\n\\ndef weighted_verdict_loss(logits, labels, weights, normalization=\\"sample_mean\\"):\\n    \\"\\"\\"Keep class weights effective even for singleton/single-class microbatches.\\n\\n    Weights have dataset mean one. Averaging weighted per-example losses gives\\n    the same gradient under equal-size microbatch accumulation as a full batch.\\n    The legacy option is retained for old checkpoints and controlled ablations.\\n    \\"\\"\\"\\n    if normalization not in (\\"sample_mean\\", \\"batch_weight_mean\\"):\\n        raise ValueError(f\\"Unknown loss normalization: {normalization}\\")\\n    reduction = \\"none\\" if normalization == \\"sample_mean\\" else \\"mean\\"\\n    loss = nn.functional.cross_entropy(logits.float(), labels, weight=weights.float(), reduction=reduction)\\n    return loss.mean()\\n\\n\\nclass PairReaderConfig(PretrainedConfig):\\n    model_type = \\"evidence_pair_reader\\"\\n\\n    def __init__(self, encoder_config=None, projection_size=192, dropout=0.15,\\n                 nli_labels=None, class_weights=None, distillation_weight=0.1, freeze_word_embeddings=True,\\n                 loss_normalization=\\"batch_weight_mean\\", **kwargs):\\n        kwargs.pop(\\"num_labels\\", None)\\n        super().__init__(num_labels=4, **kwargs)\\n        self.encoder_config = encoder_config or {}\\n        self.projection_size = projection_size\\n        self.dropout = dropout\\n        self.nli_labels = nli_labels or [\\"contradiction\\", \\"entailment\\", \\"neutral\\"]\\n        self.class_weights = class_weights or [1.0] * 4\\n        self.distillation_weight = distillation_weight\\n        self.freeze_word_embeddings = freeze_word_embeddings\\n        if loss_normalization not in (\\"sample_mean\\", \\"batch_weight_mean\\"):\\n            raise ValueError(\\"Unknown verdict-loss normalization\\")\\n        self.loss_normalization = loss_normalization\\n        self.id2label = dict(enumerate(CANONICAL_LABELS))\\n        self.label2id = {v: k for k, v in self.id2label.items()}\\n\\n\\nclass EvidencePairReader(PreTrainedModel):\\n    config_class = PairReaderConfig\\n    base_model_prefix = \\"reader\\"\\n    accepts_loss_kwargs = False\\n\\n    def __init__(self, config):\\n        super().__init__(config)\\n        encoder = dict(config.encoder_config)\\n        kind = encoder.pop(\\"model_type\\")\\n        self.reader = AutoModelForSequenceClassification.from_config(AutoConfig.for_model(kind, **encoder))\\n        hidden = int(self.reader.config.hidden_size)\\n        size, q = config.projection_size, len(QUALITY_NAMES)\\n        self.project = nn.Sequential(nn.Linear(hidden + 3 + q, size), nn.GELU(), nn.LayerNorm(size))\\n        self.attention = nn.Linear(size, 1)\\n        self.verdict = nn.Sequential(nn.Dropout(config.dropout), nn.Linear(2 * size + q + 8, size),\\n                                     nn.GELU(), nn.Dropout(config.dropout), nn.Linear(size, 4))\\n        self.register_buffer(\\"verdict_weights\\", torch.tensor(config.class_weights, dtype=torch.float32))\\n        self.post_init()\\n        if config.freeze_word_embeddings:\\n            self.reader.get_input_embeddings().requires_grad_(False)\\n\\n    @classmethod\\n    def from_nli(cls, name, revision=None, *, class_weights=None, label_order=None, distillation_weight=0.1,\\n                 freeze_word_embeddings=True, loss_normalization=\\"sample_mean\\"):\\n        reader = AutoModelForSequenceClassification.from_pretrained(name, revision=revision)\\n        if int(reader.config.num_labels) != 3:\\n            raise ValueError(\\"Reader initialization requires a three-label NLI model\\")\\n        order = nli_label_order(reader.config.id2label, label_order)\\n        config = PairReaderConfig(encoder_config=reader.config.to_dict(), nli_labels=order,\\n                                  class_weights=class_weights, distillation_weight=distillation_weight,\\n                                  freeze_word_embeddings=freeze_word_embeddings, loss_normalization=loss_normalization)\\n        model = cls(config)\\n        model.reader = reader\\n        if freeze_word_embeddings:\\n            model.reader.get_input_embeddings().requires_grad_(False)\\n        return model\\n\\n    def forward(self, input_ids, attention_mask, evidence_mask, quality, teacher_probabilities,\\n                token_type_ids=None, labels=None):\\n        batch, passages, length = input_ids.shape\\n        encoded = {\\"input_ids\\": input_ids.reshape(-1, length), \\"attention_mask\\": attention_mask.reshape(-1, length)}\\n        if token_type_ids is not None:\\n            encoded[\\"token_type_ids\\"] = token_type_ids.reshape(-1, length)\\n        output = self.reader(**encoded, output_hidden_states=True, return_dict=True)\\n        hidden = output.hidden_states[-1][:, 0].reshape(batch, passages, -1)\\n        nli = output.logits.float().reshape(batch, passages, 3)\\n        indices = [self.config.nli_labels.index(k) for k in (\\"entailment\\", \\"contradiction\\", \\"neutral\\")]\\n        nli = nli[..., indices]\\n        probabilities = nli.softmax(-1)\\n        valid = evidence_mask.to(hidden.dtype)\\n        projected = self.project(torch.cat([hidden, probabilities.to(hidden.dtype), quality.to(hidden.dtype)], dim=-1))\\n        attention_logits = self.attention(projected).squeeze(-1).masked_fill(valid == 0, -1e4)\\n        weights = attention_logits.softmax(-1) * valid\\n        weights = weights / weights.sum(-1, keepdim=True).clamp_min(1e-6)\\n        pooled = (projected * weights.unsqueeze(-1)).sum(1)\\n        any_evidence = (valid.sum(1, keepdim=True) > 0).to(hidden.dtype)\\n        maximum = projected.masked_fill(valid.unsqueeze(-1) == 0, -1e4).max(1).values * any_evidence\\n        mean_nli = (probabilities * valid.unsqueeze(-1)).sum(1) / valid.sum(1, keepdim=True).clamp_min(1)\\n        max_nli = probabilities.masked_fill(valid.unsqueeze(-1) == 0, 0).max(1).values\\n        mean_quality = (quality * valid.unsqueeze(-1)).sum(1) / valid.sum(1, keepdim=True).clamp_min(1)\\n        conflict = max_nli[:, :1] * max_nli[:, 1:2]\\n        features = torch.cat([pooled, maximum, mean_nli, max_nli, mean_quality, any_evidence, conflict], dim=-1)\\n        logits = self.verdict(features.to(hidden.dtype))\\n        loss = None\\n        if labels is not None:\\n            loss = weighted_verdict_loss(logits, labels, self.verdict_weights, self.config.loss_normalization)\\n            if self.config.distillation_weight:\\n                teacher = teacher_probabilities.float().clamp_min(1e-7)\\n                kl = nn.functional.kl_div(nli.log_softmax(-1), teacher, reduction=\\"none\\").sum(-1)\\n                loss = loss + self.config.distillation_weight * (kl * valid).sum() / valid.sum().clamp_min(1)\\n        return SequenceClassifierOutput(loss=loss, logits=logits)\\n\\n\\nclass PairDataset(torch.utils.data.Dataset):\\n    def __init__(self, records, tokenizer, *, max_length=384, claim_budget=96):\\n        from src.utils.progress import progress\\n\\n        self.records, self.features = records, []\\n        with progress(total=len(records), description=\\"Encode evidence pairs\\", unit=\\"claim\\", leave=False) as bar:\\n            for record in records:\\n                hypothesis = tokenizer.decode(tokenizer.encode(record[\\"claim\\"], add_special_tokens=False)[:claim_budget], skip_special_tokens=True)\\n                evidence = (record.get(\\"retrieved_evidence\\") or [])[:3]\\n                pairs, mask, quality, teacher = [], [], [], []\\n                for i in range(3):\\n                    item = evidence[i] if i < len(evidence) else {\\"text\\": \\"No evidence available.\\"}\\n                    pairs.append(tokenizer(pair_premise(item), hypothesis, truncation=\\"only_first\\", max_length=max_length))\\n                    mask.append(float(i < len(evidence)))\\n                    quality.append(item.get(\\"quality\\", [0.0] * len(QUALITY_NAMES)))\\n                    p = item.get(\\"nli_probabilities\\", {\\"entailment\\": 0.0, \\"contradiction\\": 0.0, \\"neutral\\": 1.0})\\n                    teacher.append([p[k] for k in (\\"entailment\\", \\"contradiction\\", \\"neutral\\")])\\n                self.features.append({\\"pairs\\": pairs, \\"evidence_mask\\": mask, \\"quality\\": quality,\\n                                      \\"teacher_probabilities\\": teacher, \\"labels\\": CANONICAL_LABELS.index(record[\\"label\\"])})\\n                bar.update(1)\\n\\n    def __len__(self):\\n        return len(self.features)\\n\\n    def __getitem__(self, i):\\n        return self.features[i]\\n\\n\\nclass PairCollator:\\n    def __init__(self, tokenizer):\\n        self.tokenizer = tokenizer\\n\\n    def __call__(self, rows):\\n        padded = self.tokenizer.pad([p for r in rows for p in r[\\"pairs\\"]], return_tensors=\\"pt\\", pad_to_multiple_of=8)\\n        result = {k: v.reshape(len(rows), 3, -1) for k, v in padded.items()}\\n        for key in (\\"evidence_mask\\", \\"quality\\", \\"teacher_probabilities\\"):\\n            result[key] = torch.tensor([r[key] for r in rows], dtype=torch.float32)\\n        result[\\"labels\\"] = torch.tensor([r[\\"labels\\"] for r in rows], dtype=torch.long)\\n        return result\\n", "src/verification/pair_training.py": "\\"\\"\\"Train-only model selection and checkpointed fitting of the pair reader.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport gc\\nimport json\\nfrom pathlib import Path\\n\\nimport numpy as np\\n\\nfrom src.data.averitec import CANONICAL_LABELS\\nfrom src.data.evidence import attach_evidence\\nfrom src.evaluation.classification import evaluate_predictions\\nfrom src.evaluation.splits import selection_split\\nfrom src.utils.checkpoints import atomic_jsonl, bind_run, latest_complete_trainer_checkpoint\\nfrom src.utils.io import read_jsonl, write_json\\nfrom src.utils.runs import sha256_file\\nfrom src.verification.inference import predict_cached\\n\\n\\ndef selection_key(report):\\n    return (-report[\\"macro_f1\\"], report[\\"decision_error_rates\\"][\\"false_reassurance_count\\"],\\n            report[\\"decision_error_rates\\"][\\"false_alarm_count\\"])\\n\\n\\ndef train_reader(args):\\n    import torch\\n    from transformers import AutoTokenizer, Trainer, TrainingArguments, set_seed\\n    from src.verification.pair_reader import EvidencePairReader, PairCollator, PairDataset\\n\\n    if args.device.startswith(\\"cuda\\") and not torch.cuda.is_available():\\n        raise RuntimeError(\\"A Colab GPU runtime is required\\")\\n    preparation = json.loads((args.prepared / \\"preparation.json\\").read_text(encoding=\\"utf-8\\"))\\n    if preparation[\\"limit\\"]:\\n        raise ValueError(\\"Training requires full preparation, not the smoke-test subset\\")\\n    if not (args.prepared / \\"preparation_report.json\\").is_file():\\n        raise ValueError(\\"Evidence preparation has not finished\\")\\n    if args.mode in preparation.get(\\"distillation_incompatible_modes\\", []) and args.distillation_weight:\\n        raise ValueError(\\"Changed passage text has no matching NLI teacher scores; set --distillation-weight 0\\")\\n    paths = {\\"train\\": args.data_root / \\"processed/averitec/train.jsonl\\", \\"dev\\": args.data_root / \\"processed/averitec/dev.jsonl\\",\\n             \\"train_rankings\\": args.prepared / args.mode / \\"train_rankings.jsonl\\", \\"dev_rankings\\": args.prepared / args.mode / \\"dev_rankings.jsonl\\"}\\n    records = {s: attach_evidence(read_jsonl(paths[s]), paths[s + \\"_rankings\\"], exact=True) for s in (\\"train\\", \\"dev\\")}\\n    a, b = selection_split(records[\\"train\\"], .1, args.validation_seed, args.validation_policy)\\n    fit_rows, val_rows = [records[\\"train\\"][int(i)] for i in a], [records[\\"train\\"][int(i)] for i in b]\\n    config = {\\"version\\": 1, \\"mode\\": args.mode, \\"input_hashes\\": {k: sha256_file(v) for k, v in paths.items()},\\n              \\"preparation_sha256\\": sha256_file(args.prepared / \\"preparation.json\\"), \\"learning_rates\\": args.learning_rates,\\n              \\"epochs\\": args.epochs, \\"batch_size\\": args.batch_size, \\"accumulation\\": args.accumulation,\\n              \\"seed\\": args.seed, \\"validation_seed\\": args.validation_seed, \\"validation_policy\\": args.validation_policy,\\n              \\"distillation_weight\\": args.distillation_weight, \\"device\\": args.device,\\n              \\"loss_normalization\\": getattr(args, \\"loss_normalization\\", \\"sample_mean\\"),\\n              \\"freeze_word_embeddings\\": not getattr(args, \\"unfreeze_embeddings\\", False),\\n              \\"implementation_sha256\\": sha256_file(Path(__file__)),\\n              \\"architecture_sha256\\": sha256_file(Path(__file__).with_name(\\"pair_reader.py\\"))}\\n    output = args.output\\n    output.mkdir(parents=True, exist_ok=True)\\n    bind_run(output / \\"reader_run.json\\", config)\\n    tokenizer = AutoTokenizer.from_pretrained(preparation[\\"nli_model\\"], revision=preparation[\\"revision\\"], use_fast=True)\\n    collator = PairCollator(tokenizer)\\n    dataset_kwargs = {\\"max_length\\": preparation[\\"max_length\\"], \\"claim_budget\\": preparation[\\"claim_budget\\"]}\\n    train_data, val_data = PairDataset(fit_rows, tokenizer, **dataset_kwargs), PairDataset(val_rows, tokenizer, **dataset_kwargs)\\n\\n    def weights(rows):\\n        counts = np.array([sum(r[\\"label\\"] == label for r in rows) for label in CANONICAL_LABELS])\\n        if np.any(counts == 0):\\n            raise ValueError(f\\"Training fold lacks a verdict class: {counts.tolist()}\\")\\n        return (len(rows) / (4 * counts)).tolist()\\n\\n    def new_model(rows):\\n        set_seed(args.seed)\\n        return EvidencePairReader.from_nli(preparation[\\"nli_model\\"], preparation[\\"revision\\"],\\n                    class_weights=weights(rows), label_order=preparation.get(\\"nli_label_order\\"),\\n                    distillation_weight=args.distillation_weight, freeze_word_embeddings=not getattr(args, \\"unfreeze_embeddings\\", False),\\n                    loss_normalization=getattr(args, \\"loss_normalization\\", \\"sample_mean\\"))\\n\\n    def training_args(directory, lr, epochs, validation):\\n        return TrainingArguments(output_dir=str(directory), learning_rate=lr, num_train_epochs=epochs,\\n            per_device_train_batch_size=args.batch_size, per_device_eval_batch_size=args.batch_size,\\n            gradient_accumulation_steps=args.accumulation, weight_decay=.01, warmup_ratio=.06,\\n            eval_strategy=\\"epoch\\" if validation else \\"no\\", save_strategy=\\"epoch\\", save_total_limit=2,\\n            load_best_model_at_end=validation, metric_for_best_model=\\"macro_f1\\" if validation else None,\\n            greater_is_better=True, remove_unused_columns=False, label_names=[\\"labels\\"],\\n            fp16=args.device.startswith(\\"cuda\\"), use_cpu=args.device == \\"cpu\\", report_to=\\"none\\",\\n            logging_steps=20, disable_tqdm=False, dataloader_num_workers=0, seed=args.seed)\\n\\n    def predictions(logits, rows):\\n        if isinstance(logits, tuple):\\n            logits = logits[0]\\n        probabilities = torch.as_tensor(logits).float().softmax(-1).numpy()\\n        return [{\\"claim_id\\": r[\\"claim_id\\"], \\"true_label\\": r[\\"label\\"],\\n                 \\"predicted_label\\": CANONICAL_LABELS[int(vector.argmax())],\\n                 \\"probabilities\\": dict(zip(CANONICAL_LABELS, map(float, vector), strict=True))} for r, vector in zip(rows, probabilities, strict=True)]\\n\\n    def metric_fn(evaluation):\\n        return {k: evaluate_predictions(predictions(evaluation.predictions, val_rows))[k] for k in (\\"macro_f1\\", \\"accuracy\\")}\\n\\n    trials = []\\n    for lr in args.learning_rates:\\n        directory = output / \\"trials\\" / f\\"lr_{lr:g}\\"\\n        summary_path = directory / \\"summary.json\\"\\n        if summary_path.is_file():\\n            trials.append(json.loads(summary_path.read_text(encoding=\\"utf-8\\")))\\n            continue\\n        trainer = Trainer(model=new_model(fit_rows), args=training_args(directory, lr, args.epochs, True),\\n                          train_dataset=train_data, eval_dataset=val_data, data_collator=collator,\\n                          processing_class=tokenizer, compute_metrics=metric_fn)\\n        checkpoint = latest_complete_trainer_checkpoint(directory)\\n        trainer.train(resume_from_checkpoint=checkpoint)\\n        val_predictions = predictions(trainer.predict(val_data).predictions, val_rows)\\n        checkpoint_state = json.loads((Path(trainer.state.best_model_checkpoint) / \\"trainer_state.json\\").read_text())\\n        report = {**evaluate_predictions(val_predictions), \\"learning_rate\\": lr, \\"best_epoch\\": checkpoint_state[\\"epoch\\"]}\\n        report[\\"predicted_label_counts\\"] = {label: sum(p[\\"predicted_label\\"] == label for p in val_predictions) for label in CANONICAL_LABELS}\\n        atomic_jsonl(val_predictions, directory / \\"validation_predictions.jsonl\\")\\n        write_json(report, summary_path)\\n        trials.append(report)\\n        del trainer\\n        gc.collect()\\n        if torch.cuda.is_available():\\n            torch.cuda.empty_cache()\\n    chosen = sorted(trials, key=selection_key)[0]\\n    selection = {\\"selected\\": chosen, \\"trials\\": trials, \\"selection_metric\\": \\"train-only holdout macro-F1; safety errors break ties\\",\\n                 \\"validation_policy\\": args.validation_policy, \\"fit_claim_ids\\": [r[\\"claim_id\\"] for r in fit_rows],\\n                 \\"validation_claim_ids\\": [r[\\"claim_id\\"] for r in val_rows]}\\n    write_json(selection, output / \\"model_selection.json\\")\\n    print(\\"Selected train-holdout score:\\", chosen[\\"macro_f1\\"], \\"epoch:\\", chosen[\\"best_epoch\\"], flush=True)\\n    print(\\"Holdout predictions per class:\\", chosen.get(\\"predicted_label_counts\\"), flush=True)\\n    print(\\"Holdout F1 per class:\\", {label: chosen[\\"per_class\\"][label][\\"f1-score\\"] for label in CANONICAL_LABELS}, flush=True)\\n    if args.selection_only:\\n        return\\n    final = output / \\"final_model\\"\\n    if not (output / \\"final_fit_complete.json\\").is_file():\\n        trainer = Trainer(model=new_model(records[\\"train\\"]),\\n                          args=training_args(final, chosen[\\"learning_rate\\"], chosen[\\"best_epoch\\"], False),\\n                          train_dataset=PairDataset(records[\\"train\\"], tokenizer, **dataset_kwargs),\\n                          data_collator=collator, processing_class=tokenizer)\\n        checkpoint = latest_complete_trainer_checkpoint(final)\\n        trainer.train(resume_from_checkpoint=checkpoint)\\n        trainer.save_model(str(final))\\n        tokenizer.save_pretrained(final)\\n        write_json({\\"training_claims\\": len(records[\\"train\\"]), \\"selection\\": chosen}, output / \\"final_fit_complete.json\\")\\n        del trainer\\n        gc.collect()\\n        if torch.cuda.is_available():\\n            torch.cuda.empty_cache()\\n    model = EvidencePairReader.from_pretrained(final).to(args.device).eval()\\n\\n    def predict_batch(rows):\\n        dataset = PairDataset(rows, tokenizer, **dataset_kwargs)\\n        batch = {k: v.to(args.device) for k, v in collator([dataset[i] for i in range(len(dataset))]).items() if k != \\"labels\\"}\\n        with torch.inference_mode(), torch.autocast(device_type=\\"cuda\\", dtype=torch.float16, enabled=args.device.startswith(\\"cuda\\")):\\n            return model(**batch).logits.float().softmax(-1).cpu().tolist()\\n\\n    metrics = predict_cached(records[\\"dev\\"], predict_batch, output / \\"dev\\", batch_size=args.batch_size, description=f\\"Pair reader / {args.mode}\\")\\n    write_json({**metrics, \\"selected_train_holdout\\": chosen, \\"model_path\\": str(final)}, output / \\"final_dev_metrics.json\\")\\n    print(json.dumps(metrics, indent=2))\\n", "src/verification/tfidf_logreg.py": "\\"\\"\\"Transparent TF-IDF + multinomial logistic-regression verifier baseline.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom collections.abc import Iterable\\nfrom dataclasses import dataclass\\nfrom typing import Any, Literal\\n\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.pipeline import Pipeline\\n\\nfrom src.data.averitec import CANONICAL_LABELS\\n\\nEvidenceMode = Literal[\\"claim_only\\", \\"gold\\", \\"gold_all\\", \\"retrieved\\"]\\n\\n\\ndef evidence_texts(record: dict[str, Any], evidence_mode: EvidenceMode, max_evidence: int = 3) -> list[str]:\\n    \\"\\"\\"Return the exact evidence texts exposed to a verifier.\\n\\n    `gold_all` is an oracle upper bound and deliberately bypasses the user-facing\\n    three-item cap. Every deployable/end-to-end mode uses at most three items.\\n    \\"\\"\\"\\n    if max_evidence < 1:\\n        raise ValueError(\\"max_evidence must be at least one\\")\\n    if evidence_mode == \\"claim_only\\":\\n        return []\\n    if evidence_mode in {\\"gold\\", \\"gold_all\\"}:\\n        evidence = record.get(\\"evidence\\") or []\\n    elif evidence_mode == \\"retrieved\\":\\n        evidence = record.get(\\"retrieved_evidence\\") or []\\n    else:\\n        raise ValueError(f\\"Unsupported evidence mode: {evidence_mode}\\")\\n    texts = []\\n    for item in evidence:\\n        value = item.get(\\"text\\") if isinstance(item, dict) else item\\n        text = str(value or \\"\\").strip()\\n        if text:\\n            texts.append(text)\\n    return texts if evidence_mode == \\"gold_all\\" else texts[:max_evidence]\\n\\n\\ndef compose_input(record: dict[str, Any], evidence_mode: EvidenceMode, max_evidence: int = 3) -> str:\\n    \\"\\"\\"Format one canonical record for the linear verifier.\\n\\n    Gold answers are allowed only for the isolated gold-evidence experiment. The\\n    end-to-end BM25 experiment will provide retrieved passages in a later phase.\\n    \\"\\"\\"\\n    claim = str(record[\\"claim\\"]).strip()\\n    passages = evidence_texts(record, evidence_mode, max_evidence)\\n    evidence_text = \\"\\\\n\\".join(f\\"[EVIDENCE {index}] {text}\\" for index, text in enumerate(passages, start=1))\\n    return f\\"[CLAIM] {claim}\\\\n{evidence_text}\\" if evidence_text else f\\"[CLAIM] {claim}\\"\\n\\n\\n@dataclass\\nclass TfidfLogRegVerifier:\\n    \\"\\"\\"A reproducible supervised baseline with explicit label ordering.\\"\\"\\"\\n\\n    evidence_mode: EvidenceMode = \\"gold\\"\\n    max_evidence: int = 3\\n    min_df: int = 1\\n    max_features: int | None = None\\n    class_weight: str | None = \\"balanced\\"\\n    random_state: int = 718\\n    max_iter: int = 2000\\n\\n    def __post_init__(self) -> None:\\n        self.pipeline = Pipeline(\\n            [\\n                (\\n                    \\"tfidf\\",\\n                    TfidfVectorizer(\\n                        ngram_range=(1, 2),\\n                        min_df=self.min_df,\\n                        max_features=self.max_features,\\n                        sublinear_tf=True,\\n                    ),\\n                ),\\n                (\\n                    \\"logreg\\",\\n                    LogisticRegression(\\n                        max_iter=self.max_iter,\\n                        class_weight=self.class_weight,\\n                        random_state=self.random_state,\\n                    ),\\n                ),\\n            ]\\n        )\\n\\n    def fit(self, records: Iterable[dict[str, Any]]) -> \\"TfidfLogRegVerifier\\":\\n        materialized = list(records)\\n        labels = [record.get(\\"label\\") for record in materialized]\\n        unknown = set(labels) - set(CANONICAL_LABELS)\\n        if unknown:\\n            raise ValueError(f\\"Training records contain invalid labels: {sorted(unknown)}\\")\\n        self.pipeline.fit([compose_input(record, self.evidence_mode, self.max_evidence) for record in materialized], labels)\\n        return self\\n\\n    def predict_records(self, records: Iterable[dict[str, Any]]) -> list[dict[str, Any]]:\\n        materialized = list(records)\\n        inputs = [compose_input(record, self.evidence_mode, self.max_evidence) for record in materialized]\\n        predicted = self.pipeline.predict(inputs)\\n        probabilities = self.pipeline.predict_proba(inputs)\\n        class_order = list(self.pipeline.named_steps[\\"logreg\\"].classes_)\\n        output = []\\n        for record, label, row_probabilities in zip(materialized, predicted, probabilities, strict=True):\\n            prediction = {\\n                \\"claim_id\\": record[\\"claim_id\\"],\\n                \\"true_label\\": record.get(\\"label\\"),\\n                \\"predicted_label\\": str(label),\\n                \\"probabilities\\": {\\n                    class_label: float(probability)\\n                    for class_label, probability in zip(class_order, row_probabilities, strict=True)\\n                },\\n                \\"evidence_item_count\\": len(evidence_texts(record, self.evidence_mode, self.max_evidence)),\\n            }\\n            if self.evidence_mode == \\"retrieved\\":\\n                prediction[\\"evidence_passage_ids\\"] = [\\n                    str(item[\\"passage_id\\"])\\n                    for item in (record.get(\\"retrieved_evidence\\") or [])[: self.max_evidence]\\n                    if isinstance(item, dict) and item.get(\\"passage_id\\")\\n                ]\\n            output.append(prediction)\\n        return output\\n"}')
SOURCE_SHA256 = '74ac8ba352dfe4a93f1ea5b59d4c9765322a335b747f8d0dd20a5721a19cf142'
assert hashlib.sha256(json.dumps(SOURCES, sort_keys=True, ensure_ascii=True).encode()).hexdigest() == SOURCE_SHA256
RUNTIME = Path('/content') / ('reader_followup_' + SOURCE_SHA256[:12])
for name, module in list(sys.modules.items()):
    if name == 'src' or name.startswith('src.') or name == 'scripts' or name.startswith('scripts.'):
        location = getattr(module, '__file__', None)
        if location and not Path(location).resolve().is_relative_to(RUNTIME):
            raise RuntimeError('A previous project notebook is still loaded. Restart the Colab session, then run this notebook from the top.')
for name, content in SOURCES.items():
    path = RUNTIME / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
os.chdir(RUNTIME)
sys.path.insert(0, str(RUNTIME))
print('Portable source:', SOURCE_SHA256)


## 2. Dependencies and Drive

Start a fresh GPU session. Only package installation uses a subprocess. All experiments run directly with visible progress bars and tracebacks.

In [ ]:
if 'transformers' in sys.modules and sys.modules['transformers'].__version__ != '4.57.1':
    raise RuntimeError('Restart the Colab session and run this notebook first.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(RUNTIME / 'requirements-evidence-reader-colab.txt')], check=True)
from google.colab import drive
drive.mount('/content/drive')
import torch
from transformers import Trainer
if not torch.cuda.is_available():
    raise RuntimeError('Choose Runtime > Change runtime type > GPU, then reconnect.')
print('GPU:', torch.cuda.get_device_name(0), '| PyTorch:', torch.__version__)


## 3. Paths and budget

Keep settings unchanged when resuming. This writes a new directory and preserves the old results. Each variant uses one fixed learning rate from the previous raw reader's **train holdout**, with at most three epochs; this isolates the new changes without another large sweep. Keep `RUN_QA=True` for all five variants. `REFIT_ALL=False` evaluates only the train-holdout winner and saves time, but leaves some planned dev contrasts unavailable.

In [ ]:
from src.utils.checkpoints import bind_run
from src.utils.io import read_jsonl
from src.utils.runs import sha256_file
from src.verification.pair_training import selection_key
from src.data.evidence import index_claims
from src.data.averitec import CANONICAL_LABELS

DATA_ROOT = Path('/content/drive/MyDrive/10718_averitec/v2_source_documents')
OLD = DATA_ROOT / 'outputs/evidence_aware_reader_v1'
OLD_PREPARED = OLD / 'prepared'
OUT = DATA_ROOT / 'outputs/evidence_reader_followup_v2'
PREPARED = OUT / 'prepared'
QUESTIONS = OUT / 'questions'
RUN_QA = True
REFIT_ALL = True
EPOCHS = 3
TRAIN_BATCH = 2
ACCUMULATION = 8
QA_BATCH = 8
QUESTION_MODEL = 'Qwen/Qwen3-0.6B'
QA_MODEL = 'deepset/roberta-base-squad2'
QROOT = DATA_ROOT / 'outputs/initial_ml_solution_question_guided_v1/retrieval'
QUESTION_CACHES = {s: QROOT / (s + '_qwen_ce32_v1') / (s + '_question_guided.partial.jsonl') for s in ('train', 'dev')}

required = [OLD_PREPARED / 'preparation.json', OLD_PREPARED / 'preparation_report.json']
required += [OLD_PREPARED / mode / (s + '_rankings.jsonl') for mode in ('raw', 'clean_nli') for s in ('train', 'dev')]
required += [DATA_ROOT / 'processed/averitec' / (s + '.jsonl') for s in ('train', 'dev')]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError('Update OLD/DATA_ROOT to your completed reader run:\n' + '\n'.join(missing))
prep = json.loads((OLD_PREPARED / 'preparation.json').read_text())
assert prep['limit'] == 0
assert json.loads((OLD_PREPARED / 'preparation_report.json').read_text())['finalized']
for s, count in [('train', 3067), ('dev', 500)]:
    path = DATA_ROOT / 'processed/averitec' / (s + '.jsonl')
    records = read_jsonl(path)
    assert len(records) == count
    assert sha256_file(path) == prep['input_hashes'][s + '_claims']
    for mode in ('raw', 'clean_nli'):
        index_claims(read_jsonl(OLD_PREPARED / mode / (s + '_rankings.jsonl')), expected={r['claim_id'] for r in records})
previous_selection = OLD / 'pair/raw/model_selection.json'
LR = json.loads(previous_selection.read_text())['selected']['learning_rate'] if previous_selection.is_file() else 3e-5
print('Fixed learning rate:', LR, '| source:', previous_selection if previous_selection.is_file() else 'preset fallback')
OUT.mkdir(parents=True, exist_ok=True)
bind_run(OUT / 'notebook.json', {'source_sha256': SOURCE_SHA256, 'old_preparation_sha256': sha256_file(OLD_PREPARED / 'preparation.json'),
    'lr': LR, 'epochs': EPOCHS, 'train_batch': TRAIN_BATCH, 'accumulation': ACCUMULATION, 'qa_batch': QA_BATCH,
    'question_model': QUESTION_MODEL, 'qa_model': QA_MODEL, 'run_qa': RUN_QA, 'refit_all': REFIT_ALL,
    'question_cache_hashes': {s: sha256_file(p) for s, p in QUESTION_CACHES.items() if p.is_file()}})
print('New outputs:', OUT)

def run(label, script, args):
    old_argv, started = sys.argv[:], time.monotonic()
    print('\n===', label, '===', flush=True)
    try:
        sys.argv = [str(RUNTIME / 'scripts' / script), *map(str, args)]
        runpy.run_path(sys.argv[0], run_name='__main__')
    except BaseException:
        (OUT / 'last_error.txt').write_text(traceback.format_exc(), encoding='utf-8')
        raise
    finally:
        sys.argv = old_argv
        gc.collect()
        torch.cuda.empty_cache()
        print(f'{label}: {(time.monotonic() - started)/60:.1f} minutes this invocation', flush=True)

VARIANTS = {
    'raw_legacy_d01': ('raw', 'batch_weight_mean', .1),
    'raw_sample_d01': ('raw', 'sample_mean', .1),
    'raw_sample_d0': ('raw', 'sample_mean', 0),
}
if RUN_QA:
    VARIANTS.update({'qa_same3_d0': ('qa_same3', 'sample_mean', 0), 'qa_selected3_d0': ('qa_selected3', 'sample_mean', 0)})

def reader_args(name, selection_only):
    mode, loss, distillation = VARIANTS[name]
    prepared = OLD_PREPARED if mode == 'raw' else PREPARED
    args = ['train', '--data-root', DATA_ROOT, '--prepared', prepared, '--mode', mode, '--output', OUT / 'pair' / name,
            '--validation-policy', 'grouped', '--validation-seed', 718, '--seed', 718,
            '--batch-size', TRAIN_BATCH, '--accumulation', ACCUMULATION, '--learning-rates', LR, '--epochs', EPOCHS,
            '--loss-normalization', loss, '--distillation-weight', distillation]
    return args + (['--selection-only'] if selection_only else [])


## 4. Verify the loss issue, then run the three training controls

A single-example batch shows the issue clearly: the legacy reduction divides out that example's class weight. Accumulating gradients across batches does not restore it. This does not prove the fix will improve macro-F1, so we measure the effect with everything else held fixed. Holdout reports include all four F1 scores and prediction counts. No dev inference runs in this cell.

In [ ]:
from src.verification.pair_reader import weighted_verdict_loss
toy_logits = torch.zeros((1, 4))
toy_y = torch.tensor([3])
toy_weights = torch.tensor([.5, .5, 2., 4.])
print('Singleton legacy loss:', weighted_verdict_loss(toy_logits, toy_y, toy_weights, 'batch_weight_mean').item())
print('Singleton sample-weighted loss:', weighted_verdict_loss(toy_logits, toy_y, toy_weights, 'sample_mean').item())
for name in ['raw_legacy_d01', 'raw_sample_d01', 'raw_sample_d0']:
    run('Train-only control / ' + name, 'run_evidence_reader.py', reader_args(name, True))


## 5. Generate or reuse two questions per claim

Reuse earlier generated questions when the original claim file hash, prompt and claim text match. If no compatible cache exists, the small Qwen generator receives only the claim. It never sees gold answers, labels or annotated sources. Malformed generations are recorded as fallbacks. These are hypotheses about what to check, not factual evidence.

In [ ]:
if RUN_QA:
    args = ['questions', '--data-root', DATA_ROOT, '--output', QUESTIONS, '--model', QUESTION_MODEL]
    for split, path in QUESTION_CACHES.items():
        if path.is_file():
            args += ['--' + split + '-question-cache', path]
    run('Claim-only question cache', 'run_answer_followup.py', args)
    print((QUESTIONS / 'questions_report.json').read_text())


## 6. Four-claim QA smoke test

The extractor checks both questions against each candidate and compares the best source span with its no-answer score. Up to six cached passages are read: the union of raw top three and cleaned/NLI top three. Both variants preserve source IDs and exact context for audit. The selection policy uses fixed answerability, question coverage, NLI and source-diversity terms; none are tuned on dev. This is a small-pool reading test, not a search over the entire candidate corpus.

In [ ]:
def prepare_args(destination):
    return ['prepare', '--data-root', DATA_ROOT, '--parent-prepared', OLD_PREPARED, '--questions', QUESTIONS,
            '--output', destination, '--model', QA_MODEL, '--batch-size', QA_BATCH, '--max-length', 384]
if RUN_QA:
    started = time.monotonic()
    run('Eight-claim QA smoke test (four per split)', 'run_answer_followup.py', prepare_args(OUT / 'qa_smoke') + ['--limit', 4])
    elapsed = time.monotonic() - started
    print('Smoke seconds including first model download:', round(elapsed, 1))
    sample = read_jsonl(OUT / 'qa_smoke/qa_same3/train_rankings.jsonl')[0]
    print('Example claim:', sample['claim'])
    if sample['retrieved']:
        print('Packed input:\n', sample['retrieved'][0]['text'])
        print('Source URL:', sample['retrieved'][0].get('url'))


## 7. Prepare full answer contexts and inspect token coverage

The full pass saves each completed claim and resumes automatically. No old NLI model is loaded here. Teacher probabilities from the original passage cannot supervise changed QA text, so the two QA variants require distillation weight zero. Review the training-fit audit CSV before treating automatic spans as decisive evidence.

In [ ]:
if RUN_QA:
    run('Prepare answer contexts / all claims', 'run_answer_followup.py', prepare_args(PREPARED))
    print((PREPARED / 'preparation_report.json').read_text())
    from transformers import AutoTokenizer
    from src.retrieval.evidence_quality import pair_premise
    tokenizer = AutoTokenizer.from_pretrained(prep['nli_model'], revision=prep['revision'], use_fast=True)
    for mode in ('qa_same3', 'qa_selected3'):
        rows = read_jsonl(PREPARED / mode / 'train_rankings.jsonl')
        # Training input audit only; do not tune packing on dev scores.
        lengths = [len(tokenizer.encode(pair_premise(p), add_special_tokens=False)) for r in rows for p in r['retrieved']]
        budget = prep['max_length'] - prep['claim_budget'] - 4
        print(mode, 'passages:', len(lengths), '| above conservative evidence budget:', sum(n > budget for n in lengths), '/', len(lengths))
    print('Training-fit audit:', PREPARED / 'train_fit_answer_audit.csv')


## 8. Train the two answer-focused variants

Both use corrected class weighting, no distillation, the same grouped train holdout and the same model/learning-rate budget as the raw controls. The source extractor stays frozen. Only the NLI-initialized verdict encoder and aggregation head are trained; vocabulary embeddings remain frozen to limit GPU memory.

In [ ]:
if RUN_QA:
    for name in ['qa_same3_d0', 'qa_selected3_d0']:
        run('Train-only answer-focused reader / ' + name, 'run_evidence_reader.py', reader_args(name, True))


## 9. Record the train-only choice, then refit and evaluate

Choose by macro-F1 on the grouped official-train holdout. Safety-error counts break ties. All variants must use identical holdout IDs. Save the choice before running any new dev inference. With `REFIT_ALL=True`, every planned variant is refit on all 3,067 train claims and evaluated on the same 500 dev claims. Per-class and safety results matter more than a high accuracy from predicting only the majority class.

In [ ]:
summaries, holdout_ids = {}, None
for name in VARIANTS:
    path = OUT / 'pair' / name / 'model_selection.json'
    result = json.loads(path.read_text())
    ids = set(result['validation_claim_ids'])
    if holdout_ids is not None:
        assert holdout_ids == ids, 'Holdout changed between experiments'
    holdout_ids = ids
    summaries[name] = result['selected']
    counts = result['selected'].get('predicted_label_counts', {})
    print(name, '| holdout macro-F1:', round(summaries[name]['macro_f1'], 4), '| predictions:', counts)
chosen = sorted(summaries, key=lambda name: (selection_key(summaries[name]), name))[0]
bind_run(OUT / 'chosen_before_dev.json', {'chosen': chosen, 'holdout_claim_ids': sorted(holdout_ids),
    'criterion': 'Grouped official-train holdout macro-F1; FR/FA counts break ties', 'metrics': summaries})
print('Recorded choice:', chosen)
for name in (list(VARIANTS) if REFIT_ALL else [chosen]):
    run('Refit + dev / ' + name, 'run_evidence_reader.py', reader_args(name, False))


## 10. Export the comparisons

Results include per-class F1, predicted rare-class counts, calibration, FR/FA, the original URL-proxy denominator, a text-overlap subset, and planned paired bootstrap comparisons. Previous results are read from saved predictions. Intervals describe uncertainty on these claims; they do not account for retraining randomness or repeated dev inspection.

In [ ]:
args = ['compare', '--data-root', DATA_ROOT, '--prepared', PREPARED if RUN_QA else OLD_PREPARED,
        '--output', OUT, '--reference-root', OLD]
qrels = DATA_ROOT / 'processed/averitec/dev_qrels.jsonl'
if qrels.is_file():
    args += ['--qrels', qrels]
run('Export follow-up comparisons', 'run_evidence_reader.py', args)
from IPython.display import Markdown, display
display(Markdown((OUT / 'evidence_reader_results.md').read_text()))
print('Share these files:')
for path in [OUT / 'evidence_reader_results.json', OUT / 'chosen_before_dev.json',
             PREPARED / 'preparation_report.json', QUESTIONS / 'questions_report.json',
             PREPARED / 'train_fit_answer_audit.csv']:
    if path.is_file():
        print(path)


## What the results will tell us

- Legacy vs corrected weighting, with distillation unchanged: whether the old reduction was limiting rare-class learning.
- Corrected weighting with vs without distillation: whether generic NLI preservation helps or restricts task adaptation.
- Raw vs QA on the same three sources: whether focused reading helps, independently of retrieved source identity. Packing also changes context length, so it does not isolate question wording alone.
- Same-three QA vs QA selected from up to six: whether answerability-based selection adds value within the available cached pool.

An answer span can be confidently extracted from a misleading or irrelevant source. Zero/no-answer scores do not imply a Not Enough Evidence gold label, and one matching URL does not prove decisive passage relevance. If answer-focused inputs still fail, inspect the 80 training-fit audit rows before investing in another larger model: useful next supervision would be human-checked decisive spans and hard negatives that mention the topic without resolving it.

To resume, reconnect Drive and rerun in order with the same settings and this notebook version. Completed question/extraction rows and training summaries are reused. An interrupted epoch repeats work since the latest complete epoch checkpoint. Changed settings or source hashes require a new `OUT` directory. Nothing deletes prior data or checkpoints. This notebook cannot recover useful evidence outside its small candidate union, certify historical publication dates, or establish a real-world user benefit from offline F1 alone.